# 08 - Recomendador híbrido final

Este notebook implementa un primer prototipo seguro del recomendador híbrido final. Sustituye LightFM como ranking principal y combina varias señales trazables:

- perfil de contenido explicable;
- filtrado colaborativo item-item basado en MovieLens;
- ratings y vistas reales de Trakt;
- filtros de calidad;
- diversidad;
- explicaciones breves de cada recomendación.

LightFM queda como experimento independiente en el notebooks/experiments/07_lightfm_hybrid_model.ipynb y no se usa en este ranking final.

## 1. Imports y configuración

In [ ]:
import ast
import re
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("default")

RANDOM_STATE = 42

ROOT = Path("..")
SRC_ROOT = ROOT.resolve()
if not (SRC_ROOT / "src").is_dir():
    SRC_ROOT = Path.cwd().resolve()
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from src.export_utils import clean_dataframe_text_for_csv, export_debug_snapshot, safe_to_csv
from src.paths import ensure_directories

DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
REPORTS_RESULTADOS = ROOT / "reports" / "resultados"
REPORTS_DEBUG = ROOT / "reports" / "debug"
POWERBI_DATASETS = ROOT / "powerbi" / "datasets"
POWERBI_EXPECTED_FILES = [
    POWERBI_DATASETS / "recomendaciones_finales.csv",
    POWERBI_DATASETS / "perfil_usuario_trakt.csv",
    POWERBI_DATASETS / "metricas_recomendador.csv",
    POWERBI_DATASETS / "distribucion_recomendaciones.csv",
]

EXPORT_FULL_RESULTS = True
EXPORT_DEBUG_SNAPSHOT = False
EXPORT_LEGACY_EXPORTS = False
TAGS_SEMANTIC_CLEAN_PATH = DATA_PROCESSED / "tags_semantic_clean.csv"
TAG_SEMANTIC_STATS_PATH = DATA_PROCESSED / "tag_semantic_stats.csv"

REPORTS_RESULTADOS.mkdir(parents=True, exist_ok=True)

## 2. Funciones auxiliares de carga

In [ ]:
def require_file(path, message):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo esperado: {path}. {message}")
    return path


def load_csv_checked(path, message):
    return pd.read_csv(require_file(path, message))

## 3. Carga de datos

In [ ]:
ratings = load_csv_checked(
    DATA_RAW / "ratings.csv",
    "Ejecuta 01_carga_datos.ipynb o verifica la carpeta data/raw.",
)
movies = load_csv_checked(
    DATA_PROCESSED / "movies_clean.csv",
    "Ejecuta 02_limpieza_transformacion.ipynb.",
)
trakt_ratings = load_csv_checked(
    DATA_PROCESSED / "trakt_ratings_mapped.csv",
    "Ejecuta 04_trakt_api_integracion.ipynb.",
)
trakt_watched = load_csv_checked(
    DATA_PROCESSED / "trakt_watched_mapped.csv",
    "Ejecuta 04_trakt_api_integracion.ipynb.",
)

datasets = {
    "ratings": ratings,
    "movies": movies,
    "trakt_ratings": trakt_ratings,
    "trakt_watched": trakt_watched,
}

for name, df in datasets.items():
    print(f"{name}: shape={df.shape}")
    print(f"Columnas: {list(df.columns)}\n")

print(f"Usuarios MovieLens: {ratings['userId'].nunique() if 'userId' in ratings.columns else 'columna userId no encontrada'}")
print(f"Películas MovieLens en ratings: {ratings['movieId'].nunique() if 'movieId' in ratings.columns else 'columna movieId no encontrada'}")
print(f"Ratings MovieLens: {len(ratings):,}")
print(f"Ratings Trakt mapeados: {len(trakt_ratings):,}")
print(f"Vistas Trakt mapeadas: {len(trakt_watched):,}")

## 4. Normalización defensiva de columnas

In [ ]:
def _normalized_name(name):
    return str(name).strip().lower().replace("_", "").replace("-", "").replace(" ", "")


def find_column(df, candidates):
    exact = {str(col): col for col in df.columns}
    for candidate in candidates:
        if candidate in exact:
            return exact[candidate]
    normalized = {_normalized_name(col): col for col in df.columns}
    for candidate in candidates:
        col = normalized.get(_normalized_name(candidate))
        if col is not None:
            return col
    return None


def rename_first_match(df, target, candidates, required=True):
    if target in df.columns:
        return df
    source = find_column(df, candidates)
    if source is None:
        if required:
            raise KeyError(f"No se pudo encontrar una columna equivalente a '{target}'. Candidatas: {candidates}")
        return df
    return df.rename(columns={source: target})


ratings = rename_first_match(ratings, "userId", ["userId", "user_id", "userid"])
ratings = rename_first_match(ratings, "movieId", ["movieId", "movie_id", "movieid"])
ratings = rename_first_match(ratings, "rating", ["rating", "score"])

movies = rename_first_match(movies, "movieId", ["movieId", "movie_id", "movieid"])
movies = rename_first_match(movies, "title", ["title", "movie_title", "name"])
movies = rename_first_match(movies, "genres", ["genres", "genre", "movie_genres"])
movies = rename_first_match(movies, "year", ["year", "release_year"], required=False)
movies = rename_first_match(movies, "rating_mean", ["rating_mean", "mean_rating", "avg_rating", "average_rating"], required=False)
movies = rename_first_match(movies, "rating_count", ["rating_count", "num_ratings", "n_ratings", "rating_n"], required=False)

trakt_ratings = rename_first_match(trakt_ratings, "movieId", ["movieId", "movie_id", "movieid"])
trakt_watched = rename_first_match(trakt_watched, "movieId", ["movieId", "movie_id", "movieid"])

for df in [ratings, movies, trakt_ratings, trakt_watched]:
    df["movieId"] = pd.to_numeric(df["movieId"], errors="coerce")

ratings["userId"] = pd.to_numeric(ratings["userId"], errors="coerce")
ratings["rating"] = pd.to_numeric(ratings["rating"], errors="coerce")
ratings = ratings.dropna(subset=["userId", "movieId", "rating"]).copy()
ratings["userId"] = ratings["userId"].astype(int)
ratings["movieId"] = ratings["movieId"].astype(int)

movies = movies.dropna(subset=["movieId", "title", "genres"]).copy()
movies["movieId"] = movies["movieId"].astype(int)

if "year" not in movies.columns:
    movies["year"] = movies["title"].astype(str).str.extract(r"\((\d{4})\)", expand=False)
movies["year"] = pd.to_numeric(movies["year"], errors="coerce")

rating_stats = ratings.groupby("movieId")["rating"].agg(rating_mean_calc="mean", rating_count_calc="count").reset_index()
movies = movies.merge(rating_stats, on="movieId", how="left")
if "rating_mean" not in movies.columns:
    movies["rating_mean"] = movies["rating_mean_calc"]
else:
    movies["rating_mean"] = pd.to_numeric(movies["rating_mean"], errors="coerce").fillna(movies["rating_mean_calc"])
if "rating_count" not in movies.columns:
    movies["rating_count"] = movies["rating_count_calc"]
else:
    movies["rating_count"] = pd.to_numeric(movies["rating_count"], errors="coerce").fillna(movies["rating_count_calc"])
movies = movies.drop(columns=["rating_mean_calc", "rating_count_calc"])
movies["rating_count"] = movies["rating_count"].fillna(0).astype(int)

personal_rating_col = find_column(
    trakt_ratings,
    ["user_rating_normalized", "rating_normalized", "rating", "user_rating"],
)
if personal_rating_col is None:
    raise KeyError("No se encontró una columna de rating personal en Trakt.")

trakt_ratings["user_rating_5"] = pd.to_numeric(trakt_ratings[personal_rating_col], errors="coerce")
if trakt_ratings["user_rating_5"].dropna().max() > 5:
    trakt_ratings["user_rating_5"] = trakt_ratings["user_rating_5"] / 2.0
trakt_ratings["user_rating_5"] = trakt_ratings["user_rating_5"].clip(0, 5)

trakt_ratings = trakt_ratings.dropna(subset=["movieId", "user_rating_5"]).copy()
trakt_watched = trakt_watched.dropna(subset=["movieId"]).copy()
trakt_ratings["movieId"] = trakt_ratings["movieId"].astype(int)
trakt_watched["movieId"] = trakt_watched["movieId"].astype(int)

print(f"Columna de rating personal detectada: {personal_rating_col}")
print(f"Películas limpias disponibles: {len(movies):,}")

## 5. Diagnóstico de perfil Trakt

In [ ]:
# Diagnóstico de perfil Trakt

movie_profile_cols = ["movieId", "title", "genres", "year", "rating_mean", "rating_count"]
movie_profile_cols = [col for col in movie_profile_cols if col in movies.columns]

trakt_ratings_profile = trakt_ratings.merge(
    movies[movie_profile_cols],
    on="movieId",
    how="left",
    suffixes=("_trakt", "_movie")
)

# Resolver columnas duplicadas tras el merge
if "title" not in trakt_ratings_profile.columns:
    if "title_movie" in trakt_ratings_profile.columns:
        trakt_ratings_profile["title"] = trakt_ratings_profile["title_movie"]
    elif "title_trakt" in trakt_ratings_profile.columns:
        trakt_ratings_profile["title"] = trakt_ratings_profile["title_trakt"]
    else:
        trakt_ratings_profile["title"] = "movieId=" + trakt_ratings_profile["movieId"].astype(str)

if "genres" not in trakt_ratings_profile.columns:
    if "genres_movie" in trakt_ratings_profile.columns:
        trakt_ratings_profile["genres"] = trakt_ratings_profile["genres_movie"]
    elif "genres_trakt" in trakt_ratings_profile.columns:
        trakt_ratings_profile["genres"] = trakt_ratings_profile["genres_trakt"]
    else:
        trakt_ratings_profile["genres"] = ""

# Asegurar user_rating_5
if "user_rating_5" not in trakt_ratings_profile.columns:
    raise KeyError(
        "No existe la columna user_rating_5. Revisa la celda anterior de normalización de ratings de Trakt."
    )

liked_movies = trakt_ratings_profile[trakt_ratings_profile["user_rating_5"] >= 4.0].copy()
neutral_movies = trakt_ratings_profile[
    (trakt_ratings_profile["user_rating_5"] > 2.5)
    & (trakt_ratings_profile["user_rating_5"] < 4.0)
].copy()
disliked_movies = trakt_ratings_profile[trakt_ratings_profile["user_rating_5"] <= 2.5].copy()

print(f"Películas gustadas: {len(liked_movies)}")
print(f"Películas neutras: {len(neutral_movies)}")
print(f"Películas no gustadas: {len(disliked_movies)}")

if len(liked_movies) < 3:
    warnings.warn("Perfil positivo muy pequeño; las recomendaciones pueden ser poco estables.")

display_cols = ["title", "user_rating_5"]
display(liked_movies.sort_values("user_rating_5", ascending=False)[display_cols].head(10))
display(disliked_movies.sort_values("user_rating_5")[display_cols].head(10))

## 6. Preparación de matriz colaborativa item-item

In [ ]:
valid_movie_ids = set(movies["movieId"].dropna().astype(int))
popular_movie_ids = set(movies.loc[movies["rating_count"] >= 50, "movieId"].astype(int))

ratings_cf = ratings[ratings["movieId"].isin(valid_movie_ids & popular_movie_ids)].copy()
user_counts = ratings_cf.groupby("userId").size()
eligible_users = user_counts[user_counts >= 20].index
ratings_cf = ratings_cf[ratings_cf["userId"].isin(eligible_users)].copy()

if ratings_cf.empty:
    warnings.warn("La matriz colaborativa queda vacía tras los filtros de calidad. Revisa ratings.csv y movies_clean.csv.")
    user_item_matrix = sparse.csr_matrix((0, 0), dtype=np.float32)
    movie_id_to_col = {}
    col_to_movie_id = {}
    movie_id_to_title = {}
else:
    ratings_cf["mean_rating_user"] = ratings_cf.groupby("userId")["rating"].transform("mean")
    ratings_cf["rating_centered"] = ratings_cf["rating"] - ratings_cf["mean_rating_user"]

    user_codes, user_ids = pd.factorize(ratings_cf["userId"], sort=True)
    movie_codes, movie_ids = pd.factorize(ratings_cf["movieId"], sort=True)

    user_item_matrix = sparse.csr_matrix(
        (ratings_cf["rating_centered"].astype(np.float32), (user_codes, movie_codes)),
        shape=(len(user_ids), len(movie_ids)),
        dtype=np.float32,
    )

    movie_id_to_col = {int(movie_id): int(col) for col, movie_id in enumerate(movie_ids)}
    col_to_movie_id = {int(col): int(movie_id) for col, movie_id in enumerate(movie_ids)}
    movie_id_to_title = movies.set_index("movieId")["title"].to_dict()

matrix_cells = user_item_matrix.shape[0] * user_item_matrix.shape[1]
density = user_item_matrix.nnz / matrix_cells if matrix_cells else 0
print(f"Dimensiones matriz usuario-película: {user_item_matrix.shape}")
print(f"Interacciones no nulas: {user_item_matrix.nnz:,}")
print(f"Densidad aproximada: {density:.6f}")

## 7. Función de similitud item-item bajo demanda

In [ ]:
SIMILAR_MOVIES_COLUMNS = ["movieId", "similarity", "title", "genres", "year", "rating_mean", "rating_count"]


def get_similar_movies_item_item(
    target_movie_id,
    user_item_matrix,
    movie_id_to_col,
    col_to_movie_id,
    movies_df,
    top_n=50,
    min_similarity=0.05,
):
    target_movie_id = int(target_movie_id)
    if target_movie_id not in movie_id_to_col or user_item_matrix.shape[1] == 0:
        return pd.DataFrame(columns=SIMILAR_MOVIES_COLUMNS)

    target_col = movie_id_to_col[target_movie_id]
    target_vector = user_item_matrix[:, target_col].T
    similarities = cosine_similarity(target_vector, user_item_matrix.T).ravel()

    result = pd.DataFrame(
        {
            "movieId": [col_to_movie_id[col] for col in range(len(similarities))],
            "similarity": similarities,
        }
    )
    result = result[(result["movieId"] != target_movie_id) & (result["similarity"] > min_similarity)]
    result = result.sort_values("similarity", ascending=False).head(top_n)

    metadata_cols = ["movieId", "title", "genres", "year", "rating_mean", "rating_count"]
    result = result.merge(movies_df[metadata_cols], on="movieId", how="left")
    return result[SIMILAR_MOVIES_COLUMNS]

## 8. Sanity check de similitudes

In [ ]:
liked_available = liked_movies[liked_movies["movieId"].isin(movie_id_to_col.keys())].head(3)

if liked_available.empty:
    warnings.warn("Ninguna película gustada está disponible en la matriz colaborativa item-item.")
else:
    for _, liked_row in liked_available.iterrows():
        title = liked_row.get("title", f"movieId={liked_row['movieId']}")
        print(f"\nPelícula base: {title}")
        similar = get_similar_movies_item_item(
            liked_row["movieId"],
            user_item_matrix,
            movie_id_to_col,
            col_to_movie_id,
            movies,
            top_n=10,
            min_similarity=0.05,
        )
        display(similar)

## 9. Score colaborativo personalizado

In [ ]:
COLLAB_COLUMNS = [
    "movieId",
    "collab_raw_score",
    "collab_positive_raw",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "positive_collab_score",
    "negative_collab_score",
    "n_collab_evidence",
    "similar_liked_movies",
    "similar_disliked_movies",
]


def _short_unique_titles(titles, max_titles=3):
    clean_titles = []
    for title in titles:
        if pd.notna(title) and str(title) not in clean_titles:
            clean_titles.append(str(title))
        if len(clean_titles) >= max_titles:
            break
    return ", ".join(clean_titles)


def _robust_minmax(series):
    series = pd.to_numeric(series, errors="coerce").fillna(0)
    if series.empty:
        return series
    low, high = np.nanpercentile(series, [5, 95])
    if np.isclose(low, high):
        return pd.Series(np.where(series > 0, 1.0, 0.0), index=series.index)
    return ((series - low) / (high - low)).clip(0, 1)


def compute_item_item_collab_scores(
    user_ratings_df,
    user_item_matrix,
    movie_id_to_col,
    col_to_movie_id,
    movies_df,
    top_n_per_seed=200,
    min_similarity=0.03,
):
    if user_ratings_df.empty or user_item_matrix.shape[1] == 0:
        return pd.DataFrame(columns=COLLAB_COLUMNS)

    seed_titles = movies_df.set_index("movieId")["title"].to_dict()
    aggregates = {}

    for _, user_row in user_ratings_df.dropna(subset=["movieId", "user_rating_5"]).iterrows():
        seed_movie_id = int(user_row["movieId"])
        if seed_movie_id not in movie_id_to_col:
            continue

        personal_weight = float(user_row["user_rating_5"]) - 3.0
        if np.isclose(personal_weight, 0):
            continue

        similar_movies = get_similar_movies_item_item(
            seed_movie_id,
            user_item_matrix,
            movie_id_to_col,
            col_to_movie_id,
            movies_df,
            top_n=top_n_per_seed,
            min_similarity=min_similarity,
        )
        seed_title = seed_titles.get(seed_movie_id, str(seed_movie_id))

        for _, similar_row in similar_movies.iterrows():
            candidate_id = int(similar_row["movieId"])
            contribution = float(similar_row["similarity"]) * personal_weight
            candidate = aggregates.setdefault(
                candidate_id,
                {
                    "collab_raw_score": 0.0,
                    "positive_collab_score": 0.0,
                    "negative_collab_score": 0.0,
                    "n_collab_evidence": 0,
                    "similar_liked_movies": [],
                    "similar_disliked_movies": [],
                },
            )
            candidate["collab_raw_score"] += contribution
            candidate["n_collab_evidence"] += 1
            if contribution > 0:
                candidate["positive_collab_score"] += contribution
                candidate["similar_liked_movies"].append(seed_title)
            elif contribution < 0:
                candidate["negative_collab_score"] += abs(contribution)
                candidate["similar_disliked_movies"].append(seed_title)

    if not aggregates:
        return pd.DataFrame(columns=COLLAB_COLUMNS)

    rows = []
    for movie_id, values in aggregates.items():
        rows.append(
            {
                "movieId": movie_id,
                "collab_raw_score": values["collab_raw_score"],
                "positive_collab_score": values["positive_collab_score"],
                "negative_collab_score": values["negative_collab_score"],
                "n_collab_evidence": values["n_collab_evidence"],
                "similar_liked_movies": _short_unique_titles(values["similar_liked_movies"]),
                "similar_disliked_movies": _short_unique_titles(values["similar_disliked_movies"]),
            }
        )

    scores = pd.DataFrame(rows)
    scores["collab_positive_raw"] = scores["collab_raw_score"].clip(lower=0)
    scores["item_item_collab_score"] = scores["collab_positive_raw"].rank(pct=True)
    scores.loc[scores["collab_positive_raw"] <= 0, "item_item_collab_score"] = 0
    scores["item_item_negative_collab_score"] = scores["negative_collab_score"].rank(pct=True)
    scores.loc[scores["negative_collab_score"] <= 0, "item_item_negative_collab_score"] = 0
    return scores[COLLAB_COLUMNS]


collab_scores = compute_item_item_collab_scores(
    trakt_ratings,
    user_item_matrix,
    movie_id_to_col,
    col_to_movie_id,
    movies,
)

print(f"Películas con señal colaborativa personalizada: {len(collab_scores):,}")
display(collab_scores.sort_values("item_item_collab_score", ascending=False).head(10))

## 10. Candidatos base y filtros

In [ ]:
rated_movie_ids = set(trakt_ratings["movieId"].dropna().astype(int))
watched_movie_ids = set(trakt_watched["movieId"].dropna().astype(int))
excluded_movie_ids = rated_movie_ids | watched_movie_ids

base_candidates = movies.copy()
base_candidates["genres_text"] = base_candidates["genres"].fillna("").astype(str).str.strip()

candidate_mask = (
    ~base_candidates["movieId"].isin(excluded_movie_ids)
    & base_candidates["year"].notna()
    & base_candidates["genres_text"].ne("")
    & base_candidates["genres_text"].ne("(no genres listed)")
    & (base_candidates["rating_mean"] >= 3.2)
    & (base_candidates["rating_count"] >= 300)
)
candidates = base_candidates[candidate_mask].copy()

if len(candidates) < 100:
    warnings.warn("Quedan menos de 100 candidatos con rating_count >= 300; se relaja rating_count a 100.")
    candidate_mask = (
        ~base_candidates["movieId"].isin(excluded_movie_ids)
        & base_candidates["year"].notna()
        & base_candidates["genres_text"].ne("")
        & base_candidates["genres_text"].ne("(no genres listed)")
        & (base_candidates["rating_mean"] >= 3.2)
        & (base_candidates["rating_count"] >= 100)
    )
    candidates = base_candidates[candidate_mask].copy()

print(f"Candidatos tras filtros: {len(candidates):,}")

## 11. Score de contenido simple y explicable

In [ ]:
def split_genres(genres):
    if pd.isna(genres):
        return []
    genres = str(genres).strip()
    if not genres or genres == "(no genres listed)":
        return []
    return [genre.strip() for genre in genres.split("|") if genre.strip()]


def build_genre_weights(profile_df):
    counts = {}
    for genres in profile_df["genres"].dropna():
        for genre in split_genres(genres):
            counts[genre] = counts.get(genre, 0) + 1
    if not counts:
        return {}
    max_count = max(counts.values())
    return {genre: count / max_count for genre, count in counts.items()}


def genre_affinity(genres, weights):
    genre_list = split_genres(genres)
    if not genre_list or not weights:
        return 0.0
    score = sum(weights.get(genre, 0.0) for genre in genre_list)
    normalizer = sum(weights.values()) if weights else 1.0
    return float(np.clip(score / normalizer, 0, 1))


positive_genre_weights = build_genre_weights(liked_movies)
negative_genre_weights = build_genre_weights(disliked_movies)

tag_columns = [col for col in ["tags", "tags_clean", "tags_features", "tags_features_en", "tag_features"] if col in movies.columns]
if tag_columns:
    print(f"Columnas de tags detectadas para fase futura, no usadas en este prototipo: {tag_columns}")

candidates["genre_profile_score"] = candidates["genres"].apply(lambda value: genre_affinity(value, positive_genre_weights))
candidates["negative_genre_score"] = candidates["genres"].apply(lambda value: genre_affinity(value, negative_genre_weights))
candidates["content_profile_score"] = candidates["genre_profile_score"]
candidates["negative_similarity_score"] = candidates["negative_genre_score"]

print("Pesos positivos de géneros:")
print(positive_genre_weights)
print("Pesos negativos de géneros:")
print(negative_genre_weights)

## Perfil semántico del usuario con tags/genome de MovieLens

In [ ]:
import re


tags_path = DATA_RAW / "tags.csv"
genome_scores_path = DATA_RAW / "genome-scores.csv"
genome_tags_path = DATA_RAW / "genome-tags.csv"

semantic_files = {
    "tags_semantic_clean.csv": TAGS_SEMANTIC_CLEAN_PATH.exists(),
    "tag_semantic_stats.csv": TAG_SEMANTIC_STATS_PATH.exists(),
    "tags.csv": tags_path.exists(),
    "genome-scores.csv": genome_scores_path.exists(),
    "genome-tags.csv": genome_tags_path.exists(),
}
print("Archivos semánticos disponibles:")
for filename, exists in semantic_files.items():
    print(f"- {filename}: {'sí' if exists else 'no'}")

if TAGS_SEMANTIC_CLEAN_PATH.exists() and TAG_SEMANTIC_STATS_PATH.exists():
    semantic_source = "processed_tags"
elif tags_path.exists():
    warnings.warn("No existen tags semánticos procesados. Ejecuta primero notebooks/05_preprocesado_tags_semanticos.ipynb.")
    semantic_source = "raw_tags_fallback"
else:
    semantic_source = "none"
print(f"Fuente semántica usada: {semantic_source}")

SEMANTIC_COLUMNS = [
    "semantic_raw_score",
    "negative_semantic_raw_score",
    "semantic_profile_score",
    "negative_semantic_score",
    "semantic_explanation_terms",
    "negative_semantic_terms",
]

candidates["semantic_raw_score"] = 0.0
candidates["negative_semantic_raw_score"] = 0.0
candidates["semantic_profile_score"] = 0.0
candidates["negative_semantic_score"] = 0.0
candidates["semantic_explanation_terms"] = ""
candidates["negative_semantic_terms"] = ""
top_positive_semantic_terms = []
top_negative_semantic_terms = []
positive_tag_profile = pd.DataFrame(columns=["tag_clean", "positive_weight"])
negative_tag_profile = pd.DataFrame(columns=["tag_clean", "negative_weight"])


def normalize_positive_rank(raw_scores):
    normalized = pd.Series(0.0, index=raw_scores.index, dtype=float)
    positive_mask = raw_scores.fillna(0) > 0
    if positive_mask.any():
        normalized.loc[positive_mask] = raw_scores.loc[positive_mask].rank(pct=True)
    return normalized


def top_terms_from_contrib(contrib_df, term_col="tag", value_col="term_contribution", max_terms=5):
    if contrib_df.empty:
        return pd.Series(dtype=str)
    ordered = contrib_df.sort_values(["movieId", value_col], ascending=[True, False])
    return ordered.groupby("movieId")[term_col].apply(
        lambda values: ", ".join(values.astype(str).drop_duplicates().head(max_terms))
    )


def normalize_tag_text(tag):
    if pd.isna(tag):
        return None
    tag_clean = str(tag).lower().strip()
    tag_clean = tag_clean.strip('"\'`´“”‘’')
    tag_clean = tag_clean.replace("_", " ")
    protected_hyphen_terms = {"thought-provoking", "post-apocalyptic"}
    if tag_clean not in protected_hyphen_terms:
        tag_clean = re.sub(r"\s*-\s*", " ", tag_clean)
    tag_clean = re.sub(r"\s+", " ", tag_clean).strip()
    tag_clean = tag_clean.strip('"\'`´“”‘’')
    if not tag_clean or tag_clean in {"nan", "none", "null"}:
        return None
    return tag_clean


def is_bad_tag(tag_clean):
    if tag_clean is None or not str(tag_clean).strip():
        return True
    tag_clean = str(tag_clean).strip()
    if len(tag_clean) < 3:
        return True
    if tag_clean.isnumeric():
        return True
    if re.fullmatch(r"[\W_]+", tag_clean):
        return True
    if re.search(r"\b(18|19|20)\d{2}\b", tag_clean):
        return True
    if re.fullmatch(r"\d{1,2}[/-]\d{1,2}([/-]\d{2,4})?", tag_clean):
        return True
    if "http" in tag_clean or "www." in tag_clean:
        return True
    if not re.search(r"[a-z]", tag_clean):
        return True

    exact_whitelist = {
        "great soundtrack", "visuals", "visually appealing", "dark comedy", "black comedy",
        "social commentary", "twist ending", "thought-provoking", "mindfuck", "time travel",
        "artificial intelligence", "post-apocalyptic", "coming of age", "atmospheric", "surreal",
        "psychological", "dystopia", "dreamlike", "stylized", "cinematography", "soundtrack",
        "bittersweet", "quirky", "nonlinear",
    }
    administrative_patterns = [
        "imdb", "oscar", "academy award", "criterion", "netflix", "dvd", "blu", "owned",
        "watchlist", "want to see", "seen", "watched", "top 250", "1001 movies", "afi",
        "award nominated", "award winner", "bd ", "bd-",
    ]
    if any(pattern in tag_clean for pattern in administrative_patterns):
        return True
    if tag_clean in exact_whitelist:
        return False

    exact_blacklist = {
        "owned", "own", "seen", "watched", "watchlist", "want to see", "to watch", "dvd",
        "bd-r", "bd r", "bdr", "blu-ray", "blu ray", "bluray", "blue-ray", "blue ray",
        "netflix", "amazon", "hulu", "criterion", "criterion collection", "library", "collection",
        "on dvr", "recorded", "vhs", "tv", "tivo", "imdb", "imdb top 250", "top 250",
        "top 100", "afi", "afi 100", "1001 movies", "1001 movies you must see before you die",
        "must see", "classic", "classics", "cult classic", "oscar", "oscars", "oscar winner",
        "oscar nominated", "academy award", "academy awards", "award winner", "award nominated",
        "best picture", "golden globe", "palme d'or", "good", "great", "best", "bad", "worst",
        "boring", "funny", "very funny", "hilarious", "overrated", "underrated", "favorite",
        "favourite", "favorites", "favourites", "excellent", "amazing", "awesome", "terrible",
        "awful", "mediocre", "masterpiece", "movie", "movies", "film", "films", "cinema",
        "cinematic", "based on", "based on book", "based on a book", "adapted from", "adaptation",
        "remake", "sequel", "franchise", "original", "drama", "comedy", "action", "thriller",
        "romance", "horror", "sci-fi", "science fiction", "crime", "adventure", "animation",
        "children", "fantasy", "mystery", "documentary", "war", "western", "musical", "film-noir",
        "film noir", "noir", "f word", "f-word",
    }
    if tag_clean in exact_blacklist:
        return True
    return False


def build_clean_tags_dataframe(tags_raw):
    tags_clean = tags_raw.copy()
    original_rows = len(tags_clean)
    original_unique_tags = tags_clean["tag"].nunique(dropna=True) if "tag" in tags_clean.columns else 0
    tags_clean["tag_clean"] = tags_clean["tag"].apply(normalize_tag_text)
    tags_clean["bad_tag"] = tags_clean["tag_clean"].apply(is_bad_tag)
    tags_clean = tags_clean[tags_clean["tag_clean"].notna() & ~tags_clean["bad_tag"]].copy()
    keep_cols = [col for col in ["userId", "movieId", "tag", "tag_clean", "timestamp"] if col in tags_clean.columns]
    tags_clean = tags_clean[keep_cols].copy()
    tags_clean["movieId"] = pd.to_numeric(tags_clean["movieId"], errors="coerce")
    tags_clean = tags_clean.dropna(subset=["movieId"])
    tags_clean["movieId"] = tags_clean["movieId"].astype(int)
    if "userId" in tags_clean.columns:
        tags_clean["userId"] = pd.to_numeric(tags_clean["userId"], errors="coerce")
        tags_clean = tags_clean.drop_duplicates(["userId", "movieId", "tag_clean"])
    else:
        tags_clean = tags_clean.drop_duplicates(["movieId", "tag_clean"])
    removed_rows = original_rows - len(tags_clean)
    removed_pct = removed_rows / original_rows * 100 if original_rows else 0
    print(f"Filas originales tags.csv: {original_rows:,}")
    print(f"Tags únicos originales: {original_unique_tags:,}")
    print(f"Filas tras limpieza básica: {len(tags_clean):,}")
    print(f"Tags únicos tras limpieza: {tags_clean['tag_clean'].nunique():,}")
    print(f"Filas eliminadas: {removed_rows:,} ({removed_pct:.2f}%)")
    return tags_clean


def build_tag_statistics(tags_clean):
    n_movies_total = tags_clean["movieId"].nunique()
    stats = tags_clean.groupby("tag_clean").agg(n_uses=("movieId", "size"), n_movies=("movieId", "nunique")).reset_index()
    if "userId" in tags_clean.columns:
        stats_users = tags_clean.groupby("tag_clean")["userId"].nunique().rename("n_users").reset_index()
        user_tag_counts = tags_clean.groupby(["tag_clean", "userId"]).size().rename("user_uses").reset_index()
        top_user = user_tag_counts.groupby("tag_clean")["user_uses"].max().rename("top_user_uses").reset_index()
        stats = stats.merge(stats_users, on="tag_clean", how="left").merge(top_user, on="tag_clean", how="left")
        stats["top_user_share"] = stats["top_user_uses"] / stats["n_uses"]
    else:
        stats["n_users"] = np.nan
        stats["top_user_uses"] = np.nan
        stats["top_user_share"] = 0.0
    stats["pct_movies"] = stats["n_movies"] / n_movies_total if n_movies_total else 0
    stats["idf"] = np.log((1 + n_movies_total) / (1 + stats["n_movies"])) + 1
    low_movies = stats["n_movies"] < 5
    low_users = stats["n_users"] < 5 if "userId" in tags_clean.columns else pd.Series(False, index=stats.index)
    high_pct_movies = stats["pct_movies"] > 0.05
    high_top_user_share = stats["top_user_share"] > 0.60 if "userId" in tags_clean.columns else pd.Series(False, index=stats.index)
    stats["is_reliable_tag"] = ~(low_movies | low_users | high_pct_movies | high_top_user_share)
    print(f"Tags antes del filtro de fiabilidad: {len(stats):,}")
    print(f"Tags fiables: {stats['is_reliable_tag'].sum():,}")
    print(f"Eliminados por n_movies bajo: {low_movies.sum():,}")
    print(f"Eliminados por n_users bajo: {low_users.sum():,}")
    print(f"Eliminados por pct_movies alto: {high_pct_movies.sum():,}")
    print(f"Eliminados por top_user_share alto: {high_top_user_share.sum():,}")
    reliable = stats[stats["is_reliable_tag"]].copy()
    display(reliable.sort_values("n_uses", ascending=False).head(30))
    display(reliable.sort_values("n_movies", ascending=False).head(30))
    display(reliable[reliable["n_movies"] >= 5].sort_values("idf", ascending=False).head(30))
    display(stats[high_top_user_share].sort_values("top_user_share", ascending=False).head(30))
    return stats


def build_semantic_profiles_from_tags(tags_clean, liked_movies, disliked_movies, tag_stats):
    idf_map = tag_stats.set_index("tag_clean")["idf"]
    positive_seed = liked_movies.loc[liked_movies["user_rating_5"] >= 4.0, ["movieId", "user_rating_5"]].dropna().copy()
    negative_seed = disliked_movies.loc[disliked_movies["user_rating_5"] <= 2.5, ["movieId", "user_rating_5"]].dropna().copy()
    positive_seed["movieId"] = positive_seed["movieId"].astype(int)
    negative_seed["movieId"] = negative_seed["movieId"].astype(int)

    positive_profile = pd.DataFrame(columns=["tag_clean", "positive_weight", "positive_n_seed_movies"])
    if not positive_seed.empty:
        positive_tags = tags_clean.merge(positive_seed, on="movieId", how="inner")
        positive_tags["idf"] = positive_tags["tag_clean"].map(idf_map).fillna(1.0)
        positive_tags["personal_weight"] = positive_tags["user_rating_5"] - 3.0
        positive_tags["tag_weight"] = positive_tags["personal_weight"] * positive_tags["idf"]
        positive_profile = positive_tags.groupby("tag_clean").agg(
            positive_weight=("tag_weight", "sum"),
            positive_n_seed_movies=("movieId", "nunique"),
        ).reset_index()
        positive_profile = positive_profile[positive_profile["positive_n_seed_movies"] >= 1]
        positive_profile = positive_profile.sort_values("positive_weight", ascending=False).head(80)

    negative_profile = pd.DataFrame(columns=["tag_clean", "negative_weight", "negative_n_seed_movies"])
    if not negative_seed.empty:
        negative_tags = tags_clean.merge(negative_seed, on="movieId", how="inner")
        negative_tags["idf"] = negative_tags["tag_clean"].map(idf_map).fillna(1.0)
        negative_tags["personal_weight"] = 3.0 - negative_tags["user_rating_5"]
        negative_tags["tag_weight"] = negative_tags["personal_weight"] * negative_tags["idf"]
        negative_profile = negative_tags.groupby("tag_clean").agg(
            negative_weight=("tag_weight", "sum"),
            negative_n_seed_movies=("movieId", "nunique"),
        ).reset_index()
        negative_profile = negative_profile.sort_values("negative_weight", ascending=False).head(80)

    overlap = sorted(set(positive_profile["tag_clean"]) & set(negative_profile["tag_clean"]))
    if overlap:
        positive_profile.loc[positive_profile["tag_clean"].isin(overlap), "positive_weight"] *= 0.5
        negative_profile.loc[negative_profile["tag_clean"].isin(overlap), "negative_weight"] *= 0.5
        positive_profile = positive_profile.sort_values("positive_weight", ascending=False)
        negative_profile = negative_profile.sort_values("negative_weight", ascending=False)

    display(positive_profile.head(30))
    display(negative_profile.head(30))
    print(f"Tags en intersección positivo/negativo: {overlap[:30]}")
    print(f"Número de tags positivos: {len(positive_profile)}")
    print(f"Número de tags negativos: {len(negative_profile)}")
    return positive_profile, negative_profile


def build_semantic_profiles_from_processed_tags(tags_clean, liked_movies, disliked_movies):
    required_columns = {"movieId", "tag_clean", "idf"}
    if not required_columns.issubset(tags_clean.columns):
        missing = sorted(required_columns - set(tags_clean.columns))
        raise ValueError(f"tags_semantic_clean.csv no tiene las columnas esperadas: {missing}")

    profile_tags = tags_clean[["movieId", "tag_clean", "idf"]].copy()
    profile_tags["movieId"] = pd.to_numeric(profile_tags["movieId"], errors="coerce")
    profile_tags["idf"] = pd.to_numeric(profile_tags["idf"], errors="coerce").fillna(1.0)
    profile_tags = profile_tags.dropna(subset=["movieId", "tag_clean"])
    profile_tags["movieId"] = profile_tags["movieId"].astype(int)
    profile_tags = profile_tags.drop_duplicates(["movieId", "tag_clean"])

    positive_seed = liked_movies.loc[liked_movies["user_rating_5"] >= 4.0, ["movieId", "user_rating_5"]].dropna().copy()
    negative_seed = disliked_movies.loc[disliked_movies["user_rating_5"] <= 2.5, ["movieId", "user_rating_5"]].dropna().copy()
    positive_seed["movieId"] = positive_seed["movieId"].astype(int)
    negative_seed["movieId"] = negative_seed["movieId"].astype(int)

    positive_profile = pd.DataFrame(columns=["tag_clean", "positive_weight", "positive_n_seed_movies"])
    if not positive_seed.empty:
        positive_tags = profile_tags.merge(positive_seed, on="movieId", how="inner")
        positive_tags["tag_weight"] = (positive_tags["user_rating_5"] - 3.0) * positive_tags["idf"]
        positive_profile = positive_tags.groupby("tag_clean", as_index=False).agg(
            positive_weight=("tag_weight", "sum"),
            positive_n_seed_movies=("movieId", "nunique"),
        )

    negative_profile = pd.DataFrame(columns=["tag_clean", "negative_weight", "negative_n_seed_movies"])
    if not negative_seed.empty:
        negative_tags = profile_tags.merge(negative_seed, on="movieId", how="inner")
        negative_tags["tag_weight"] = (3.0 - negative_tags["user_rating_5"]) * negative_tags["idf"]
        negative_profile = negative_tags.groupby("tag_clean", as_index=False).agg(
            negative_weight=("tag_weight", "sum"),
            negative_n_seed_movies=("movieId", "nunique"),
        )

    overlap = sorted(set(positive_profile["tag_clean"]) & set(negative_profile["tag_clean"]))
    if overlap:
        ambiguity_penalty = 0.25
        positive_profile.loc[positive_profile["tag_clean"].isin(overlap), "positive_weight"] *= ambiguity_penalty
        negative_profile.loc[negative_profile["tag_clean"].isin(overlap), "negative_weight"] *= ambiguity_penalty

    positive_profile = positive_profile.sort_values("positive_weight", ascending=False).head(80)
    negative_profile = negative_profile.sort_values("negative_weight", ascending=False).head(80)

    display(positive_profile.head(30))
    display(negative_profile.head(30))
    print(f"Tags en intersección positivo/negativo: {overlap[:30]}")
    print(f"Número de tags positivos: {len(positive_profile)}")
    print(f"Número de tags negativos: {len(negative_profile)}")
    return positive_profile, negative_profile


def compute_semantic_scores_from_tags(candidates, tags_clean, positive_tag_profile, negative_tag_profile):
    result = candidates[["movieId"]].copy()
    result["semantic_raw_score"] = 0.0
    result["negative_semantic_raw_score"] = 0.0
    result["semantic_profile_score"] = 0.0
    result["negative_semantic_score"] = 0.0
    result["semantic_explanation_terms"] = ""
    result["negative_semantic_terms"] = ""
    candidate_tags = tags_clean[tags_clean["movieId"].isin(result["movieId"].astype(int))][["movieId", "tag_clean"]].drop_duplicates()
    if candidate_tags.empty:
        return result

    if not positive_tag_profile.empty:
        positive_contrib = candidate_tags.merge(positive_tag_profile[["tag_clean", "positive_weight"]], on="tag_clean", how="inner")
        positive_contrib["contribution"] = positive_contrib["positive_weight"]
        raw_positive = positive_contrib.groupby("movieId")["contribution"].sum()
        result["semantic_raw_score"] = result["movieId"].map(raw_positive).fillna(0.0)
        result["semantic_profile_score"] = normalize_positive_rank(result["semantic_raw_score"])
        result["semantic_explanation_terms"] = result["movieId"].map(
            top_terms_from_contrib(positive_contrib, term_col="tag_clean", value_col="contribution", max_terms=5)
        ).fillna("")

    if not negative_tag_profile.empty:
        negative_contrib = candidate_tags.merge(negative_tag_profile[["tag_clean", "negative_weight"]], on="tag_clean", how="inner")
        negative_contrib["contribution"] = negative_contrib["negative_weight"]
        raw_negative = negative_contrib.groupby("movieId")["contribution"].sum()
        result["negative_semantic_raw_score"] = result["movieId"].map(raw_negative).fillna(0.0)
        result["negative_semantic_score"] = normalize_positive_rank(result["negative_semantic_raw_score"])
        result["negative_semantic_terms"] = result["movieId"].map(
            top_terms_from_contrib(negative_contrib, term_col="tag_clean", value_col="contribution", max_terms=5)
        ).fillna("")
    return result


def compute_semantic_scores_from_processed_tags(candidates, tags_clean, positive_tag_profile, negative_tag_profile):
    return compute_semantic_scores_from_tags(candidates, tags_clean, positive_tag_profile, negative_tag_profile)


positive_seed_ratings = liked_movies.loc[liked_movies["user_rating_5"] >= 4.0, ["movieId", "user_rating_5"]].dropna().copy()
negative_seed_ratings = disliked_movies.loc[disliked_movies["user_rating_5"] <= 2.5, ["movieId", "user_rating_5"]].dropna().copy()
positive_seed_ratings["movieId"] = positive_seed_ratings["movieId"].astype(int)
negative_seed_ratings["movieId"] = negative_seed_ratings["movieId"].astype(int)

if semantic_source == "processed_tags":
    tags_clean = pd.read_csv(TAGS_SEMANTIC_CLEAN_PATH)
    tag_stats = pd.read_csv(TAG_SEMANTIC_STATS_PATH)
    required_clean_cols = {"movieId", "tag_clean", "idf"}
    required_stats_cols = {"tag_clean", "idf", "is_reliable_tag"}
    if not required_clean_cols.issubset(tags_clean.columns) or not required_stats_cols.issubset(tag_stats.columns):
        warnings.warn("Los tags semánticos procesados no tienen las columnas esperadas; se dejan scores semánticos en 0.")
        semantic_source = "none"
    else:
        tags_clean = tags_clean.copy()
        tags_clean["movieId"] = pd.to_numeric(tags_clean["movieId"], errors="coerce")
        tags_clean["idf"] = pd.to_numeric(tags_clean["idf"], errors="coerce").fillna(1.0)
        tags_clean = tags_clean.dropna(subset=["movieId", "tag_clean"])
        tags_clean["movieId"] = tags_clean["movieId"].astype(int)
        tags_clean = tags_clean.drop_duplicates(["movieId", "tag_clean"])
        print(f"Tags limpios cargados: {len(tags_clean):,}")
        print(f"Tags únicos limpios: {tags_clean['tag_clean'].nunique():,}")
        print(f"Películas con tags limpios: {tags_clean['movieId'].nunique():,}")
        positive_tag_profile, negative_tag_profile = build_semantic_profiles_from_processed_tags(tags_clean, liked_movies, disliked_movies)
        semantic_scores = compute_semantic_scores_from_processed_tags(candidates, tags_clean, positive_tag_profile, negative_tag_profile)
        candidates = candidates.drop(columns=[col for col in SEMANTIC_COLUMNS if col in candidates.columns]).merge(
            semantic_scores, on="movieId", how="left"
        )
        for col in ["semantic_raw_score", "negative_semantic_raw_score", "semantic_profile_score", "negative_semantic_score"]:
            candidates[col] = candidates[col].fillna(0.0)
        for col in ["semantic_explanation_terms", "negative_semantic_terms"]:
            candidates[col] = candidates[col].fillna("")
        top_positive_semantic_terms = positive_tag_profile["tag_clean"].dropna().head(20).tolist()
        top_negative_semantic_terms = negative_tag_profile["tag_clean"].dropna().head(20).tolist()

elif semantic_source == "genome":
    genome_scores = pd.read_csv(genome_scores_path)
    genome_tags = pd.read_csv(genome_tags_path)
    required_genome_cols = {"movieId", "tagId", "relevance"}
    required_tag_cols = {"tagId", "tag"}
    if not required_genome_cols.issubset(genome_scores.columns) or not required_tag_cols.issubset(genome_tags.columns):
        warnings.warn("Los archivos genome no tienen las columnas esperadas; se dejan scores semánticos en 0.")
        semantic_source = "none"
    else:
        genome_scores["movieId"] = pd.to_numeric(genome_scores["movieId"], errors="coerce")
        genome_scores["tagId"] = pd.to_numeric(genome_scores["tagId"], errors="coerce")
        genome_scores["relevance"] = pd.to_numeric(genome_scores["relevance"], errors="coerce")
        genome_scores = genome_scores.dropna(subset=["movieId", "tagId", "relevance"]).copy()
        genome_scores["movieId"] = genome_scores["movieId"].astype(int)
        genome_scores["tagId"] = genome_scores["tagId"].astype(int)
        genome_tags["tagId"] = pd.to_numeric(genome_tags["tagId"], errors="coerce")
        genome_tags = genome_tags.dropna(subset=["tagId", "tag"]).copy()
        genome_tags["tagId"] = genome_tags["tagId"].astype(int)

        relevant_movie_ids = set(candidates["movieId"].astype(int)) | set(positive_seed_ratings["movieId"]) | set(negative_seed_ratings["movieId"])
        genome_scores = genome_scores[genome_scores["movieId"].isin(relevant_movie_ids)].copy()
        candidate_genome = genome_scores[genome_scores["movieId"].isin(candidates["movieId"].astype(int))].copy()

        positive_profile = pd.DataFrame(columns=["tagId", "tag_weight"])
        if not positive_seed_ratings.empty:
            positive_profile = genome_scores.merge(positive_seed_ratings, on="movieId", how="inner")
            positive_profile["personal_weight"] = positive_profile["user_rating_5"] - 3.0
            positive_profile["weighted_relevance"] = positive_profile["relevance"] * positive_profile["personal_weight"]
            positive_profile = positive_profile.groupby("tagId", as_index=False)["weighted_relevance"].sum()
            positive_profile = positive_profile.sort_values("weighted_relevance", ascending=False).head(100)
            max_positive = positive_profile["weighted_relevance"].max()
            positive_profile["tag_weight"] = positive_profile["weighted_relevance"] / max_positive if max_positive > 0 else 0

        negative_profile = pd.DataFrame(columns=["tagId", "tag_weight"])
        if not negative_seed_ratings.empty:
            negative_profile = genome_scores.merge(negative_seed_ratings, on="movieId", how="inner")
            negative_profile["personal_weight"] = 3.0 - negative_profile["user_rating_5"]
            negative_profile["weighted_relevance"] = negative_profile["relevance"] * negative_profile["personal_weight"]
            negative_profile = negative_profile.groupby("tagId", as_index=False)["weighted_relevance"].sum()
            negative_profile = negative_profile.sort_values("weighted_relevance", ascending=False).head(100)
            max_negative = negative_profile["weighted_relevance"].max()
            negative_profile["tag_weight"] = negative_profile["weighted_relevance"] / max_negative if max_negative > 0 else 0

        if not positive_profile.empty and not candidate_genome.empty:
            positive_contrib = candidate_genome.merge(positive_profile[["tagId", "tag_weight"]], on="tagId", how="inner")
            positive_contrib["term_contribution"] = positive_contrib["relevance"] * positive_contrib["tag_weight"]
            raw_positive = positive_contrib.groupby("movieId")["term_contribution"].sum()
            candidates["semantic_raw_score"] = candidates["movieId"].map(raw_positive).fillna(0.0)
            candidates["semantic_profile_score"] = normalize_positive_rank(candidates["semantic_raw_score"])
            positive_terms = positive_contrib.merge(genome_tags, on="tagId", how="left")
            candidates["semantic_explanation_terms"] = candidates["movieId"].map(top_terms_from_contrib(positive_terms)).fillna("")
            top_positive_semantic_terms = positive_profile.merge(genome_tags, on="tagId", how="left")["tag"].dropna().head(15).tolist()

        if not negative_profile.empty and not candidate_genome.empty:
            negative_contrib = candidate_genome.merge(negative_profile[["tagId", "tag_weight"]], on="tagId", how="inner")
            negative_contrib["term_contribution"] = negative_contrib["relevance"] * negative_contrib["tag_weight"]
            raw_negative = negative_contrib.groupby("movieId")["term_contribution"].sum()
            candidates["negative_semantic_raw_score"] = candidates["movieId"].map(raw_negative).fillna(0.0)
            candidates["negative_semantic_score"] = normalize_positive_rank(candidates["negative_semantic_raw_score"])
            negative_terms = negative_contrib.merge(genome_tags, on="tagId", how="left")
            candidates["negative_semantic_terms"] = candidates["movieId"].map(top_terms_from_contrib(negative_terms)).fillna("")
            top_negative_semantic_terms = negative_profile.merge(genome_tags, on="tagId", how="left")["tag"].dropna().head(15).tolist()

elif semantic_source == "raw_tags_fallback":
    tags_raw = pd.read_csv(tags_path)
    if not {"movieId", "tag"}.issubset(tags_raw.columns):
        warnings.warn("tags.csv no tiene las columnas esperadas; se dejan scores semánticos en 0.")
        semantic_source = "none"
    else:
        tags_clean = build_clean_tags_dataframe(tags_raw)
        tag_stats = build_tag_statistics(tags_clean)
        reliable_tags = set(tag_stats.loc[tag_stats["is_reliable_tag"], "tag_clean"])
        tags_clean = tags_clean[tags_clean["tag_clean"].isin(reliable_tags)].copy()
        print(f"Filas finales de tags fiables: {len(tags_clean):,}")
        print(f"Tags únicos finales: {tags_clean['tag_clean'].nunique():,}")
        print(f"Películas con al menos un tag fiable: {tags_clean['movieId'].nunique():,}")
        positive_tag_profile, negative_tag_profile = build_semantic_profiles_from_tags(tags_clean, liked_movies, disliked_movies, tag_stats)
        semantic_scores = compute_semantic_scores_from_tags(candidates, tags_clean, positive_tag_profile, negative_tag_profile)
        candidates = candidates.drop(columns=[col for col in SEMANTIC_COLUMNS if col in candidates.columns]).merge(
            semantic_scores, on="movieId", how="left"
        )
        for col in ["semantic_raw_score", "negative_semantic_raw_score", "semantic_profile_score", "negative_semantic_score"]:
            candidates[col] = candidates[col].fillna(0.0)
        for col in ["semantic_explanation_terms", "negative_semantic_terms"]:
            candidates[col] = candidates[col].fillna("")
        top_positive_semantic_terms = positive_tag_profile["tag_clean"].dropna().head(20).tolist()
        top_negative_semantic_terms = negative_tag_profile["tag_clean"].dropna().head(20).tolist()

if semantic_source == "none":
    warnings.warn("No hay genome-scores/genome-tags ni tags.csv; los scores semánticos se dejan en 0.")

for col in ["semantic_raw_score", "negative_semantic_raw_score", "semantic_profile_score", "negative_semantic_score"]:
    candidates[col] = candidates[col].fillna(0.0)
for col in ["semantic_explanation_terms", "negative_semantic_terms"]:
    candidates[col] = candidates[col].fillna("")

FORBIDDEN_SEMANTIC_TERMS = [
    "hans zimmer", "tom hanks", "coen brothers", "christopher nolan", "pixar",
    "great acting", "good acting", "cult film", "inspirational", "touching", "slow paced",
]
found_forbidden_terms = set()
if "tag_clean" in positive_tag_profile.columns:
    found_forbidden_terms.update(set(positive_tag_profile["tag_clean"].dropna()) & set(FORBIDDEN_SEMANTIC_TERMS))
if "tag_clean" in negative_tag_profile.columns:
    found_forbidden_terms.update(set(negative_tag_profile["tag_clean"].dropna()) & set(FORBIDDEN_SEMANTIC_TERMS))
semantic_terms_text = " ".join(candidates["semantic_explanation_terms"].fillna("").astype(str).str.lower())
for term in FORBIDDEN_SEMANTIC_TERMS:
    if term in semantic_terms_text:
        found_forbidden_terms.add(term)
if found_forbidden_terms:
    print("Advertencia: siguen apareciendo tags que deberían haber sido filtrados por el 09.")
    print(sorted(found_forbidden_terms))

positive_semantic_count = (candidates["semantic_profile_score"] > 0).sum()
negative_semantic_count = (candidates["negative_semantic_score"] > 0).sum()
print(f"Fuente semántica usada final: {semantic_source}")
print(f"Candidatos con semantic_profile_score > 0: {positive_semantic_count:,} ({positive_semantic_count / len(candidates) * 100:.2f}%)")
print(f"Candidatos con negative_semantic_score > 0: {negative_semantic_count:,} ({negative_semantic_count / len(candidates) * 100:.2f}%)")
print("Top 20 tags positivos finales:")
print(top_positive_semantic_terms[:20])
print("Top 20 tags negativos finales:")
print(top_negative_semantic_terms[:20])


## 12. Scores de calidad y popularidad

In [ ]:
candidates["rating_score"] = ((candidates["rating_mean"] - 3.0) / 2.0).clip(0, 1)

log_counts = np.log1p(candidates["rating_count"].clip(lower=0))
if np.isclose(log_counts.max(), log_counts.min()):
    candidates["popularity_score"] = 0.0
else:
    candidates["popularity_score"] = ((log_counts - log_counts.min()) / (log_counts.max() - log_counts.min())).clip(0, 1)

display(candidates[["title", "rating_mean", "rating_count", "rating_score", "popularity_score"]].head())

## 13. Fusionar señales

In [ ]:
candidates_scored = candidates.merge(collab_scores, on="movieId", how="left")

numeric_collab_cols = [
    "collab_raw_score",
    "collab_positive_raw",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "positive_collab_score",
    "negative_collab_score",
    "n_collab_evidence",
]
for col in numeric_collab_cols:
    candidates_scored[col] = candidates_scored[col].fillna(0)

for col in ["similar_liked_movies", "similar_disliked_movies"]:
    candidates_scored[col] = candidates_scored[col].fillna("")

print(f"Candidatos con scoring fusionado: {len(candidates_scored):,}")

## Diagnóstico adaptativo del perfil de usuario

In [ ]:
def _clip01(value):
    return float(np.clip(value, 0.0, 1.0))


def rank_normalize_positive(series):
    raw_scores = pd.to_numeric(pd.Series(series), errors="coerce").fillna(0.0)
    normalized = pd.Series(0.0, index=raw_scores.index, dtype=float)
    positive_mask = raw_scores > 0
    if positive_mask.any():
        normalized.loc[positive_mask] = raw_scores.loc[positive_mask].rank(pct=True)
    return normalized.clip(0.0, 1.0)


def _rank_positive(raw_scores):
    return rank_normalize_positive(raw_scores)


def safe_text(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value).strip().lower()


def contains_any(text, terms):
    text = safe_text(text)
    return any(safe_text(term) in text for term in terms if safe_text(term))


def _top_terms_from_weighted_tags(contrib_df, term_col="tag_clean", value_col="contribution", max_terms=5):
    if contrib_df.empty:
        return pd.Series(dtype=str)
    ordered = contrib_df.sort_values(["movieId", value_col], ascending=[True, False])
    return ordered.groupby("movieId")[term_col].apply(
        lambda values: ", ".join(values.astype(str).drop_duplicates().head(max_terms))
    )


def main_genre_from_genres(genres):
    genre_list = split_genres(genres)
    return genre_list[0] if genre_list else "Unknown"


SEMANTIC_BRANCHES = [
    "psychological_thriller", "psychological_drama", "crime_thriller",
    "sci_fi_reflective", "surreal_fantasy", "animation_dark_surreal",
    "animation_family", "documentary_reflective", "comedy_satire",
    "romantic_comedy_drama", "dark_drama", "classic_adventure", "general",
]


def _row_text_terms(row):
    pieces = [
        row.get("core_semantic_explanation_terms", ""),
        row.get("semantic_explanation_terms", ""),
        row.get("core_negative_semantic_terms", ""),
    ]
    return " ".join(safe_text(piece) for piece in pieces if safe_text(piece))


def _branch_warning_for_row(row):
    genres = set(split_genres(row.get("genres", "")))
    branch = row.get("semantic_branch", "")
    all_text = (" ".join(sorted(genres)) + " " + _row_text_terms(row)).lower()
    thriller_genres = {"Thriller", "Mystery", "Horror", "Crime", "Sci-Fi"}
    very_strong = ["mindfuck", "paranoia", "unreliable narrator", "hallucination", "psychological thriller"]
    if {"Comedy", "Romance"}.issubset(genres) and branch == "psychological_thriller":
        return "comedy_romance_as_psychological_thriller"
    if branch == "psychological_thriller" and not (genres & thriller_genres) and not contains_any(all_text, very_strong):
        return "no_thriller_genre_as_psychological_thriller"
    if branch == "animation_family" and float(row.get("semantic_relevance_adjusted_score", 0.0) or 0.0) > 0.85:
        return "animation_family_high_score"
    return ""


def assign_semantic_branch(row):
    genres = set(split_genres(row.get("genres", "")))
    main_genre = row.get("main_genre", "")
    if pd.notna(main_genre) and str(main_genre).strip():
        genres.add(str(main_genre).strip())
    genres_text = " ".join(sorted(genres)).lower()
    tags_text = _row_text_terms(row)
    all_text = f"{genres_text} {tags_text}".strip()
    year = pd.to_numeric(row.get("year"), errors="coerce")

    def has(terms):
        return contains_any(all_text, terms)

    thriller_genres = {"Thriller", "Mystery", "Horror", "Crime", "Sci-Fi"}
    psych_strong = [
        "psychological", "psychological thriller", "paranoia", "mindfuck", "identity",
        "hallucination", "unreliable narrator", "obsession", "mental illness",
        "disturbing", "ambiguous ending", "twist ending", "cerebral",
        "nightmare", "surrealism", "dreamlike",
    ]
    psych_very_strong = [
        "psychological thriller", "mindfuck", "paranoia", "unreliable narrator",
        "hallucination", "identity crisis", "ambiguous ending", "disturbing",
    ]

    if "Documentary" in genres:
        return "documentary_reflective"
    if "Animation" in genres and has([
        "psychological", "psychological thriller", "paranoia", "mindfuck", "identity",
        "surrealism", "dreamlike", "disturbing", "hallucination",
        "ambiguous ending", "dark", "horror", "tragedy", "adult animation",
    ]):
        return "animation_dark_surreal"
    if "Animation" in genres or "Children" in genres:
        return "animation_family"
    if {"Romance", "Comedy"}.issubset(genres):
        return "romantic_comedy_drama"
    if {"Romance", "Drama"}.issubset(genres) and not (genres & thriller_genres):
        return "romantic_comedy_drama"
    if "Sci-Fi" in genres and has([
        "time travel", "artificial intelligence", "identity", "dystopia",
        "existentialism", "philosophical", "mindfuck", "cerebral",
        "ambiguous ending", "paranoia", "memory", "technology",
    ]):
        return "sci_fi_reflective"
    if ("Crime" in genres or "Thriller" in genres) and has([
        "tense", "investigation", "murder", "serial killer", "neo-noir",
        "police", "conspiracy", "revenge", "suspense", "mystery", "crime",
    ]):
        return "crime_thriller"

    romance_comedy_drama = {"Comedy", "Drama", "Romance"}.issubset(genres) or ({"Drama", "Romance"}.issubset(genres) and not (genres & thriller_genres))
    if ((genres & thriller_genres) and has(psych_strong)) or has(psych_very_strong):
        if not romance_comedy_drama or has(psych_very_strong):
            return "psychological_thriller"
    if has(["surrealism", "dreamlike", "magical realism", "weird", "fantasy", "fairy tale", "absurd", "bizarre"]):
        return "surreal_fantasy"
    if "Comedy" in genres and has(["satire", "satirical", "dark comedy", "black comedy", "absurd", "witty", "quirky"]):
        return "comedy_satire"
    if "Drama" in genres and not (genres & thriller_genres) and has([
        "psychology", "psychological", "mental illness", "depression", "grief",
        "family", "character study", "trauma", "loneliness", "identity",
    ]):
        return "psychological_drama"
    if "Drama" in genres and has(["dark", "bleak", "depressing", "tragedy", "tragic", "disturbing", "moral", "existentialism"]):
        return "dark_drama"
    return "general"

def _normalized_entropy(distribution):
    values = np.array([float(value) for value in distribution if float(value) > 0], dtype=float)
    if len(values) <= 1:
        return 0.0, 0.0
    probabilities = values / values.sum()
    entropy = float(-(probabilities * np.log(probabilities)).sum())
    return entropy, float(entropy / np.log(len(probabilities)))



def _top_tags_by_movie_for_branch(liked_movies, tags_clean=None, discriminative_tag_profile=None, max_tags=8):
    if tags_clean is None or tags_clean.empty or "movieId" not in tags_clean.columns or "tag_clean" not in tags_clean.columns:
        return pd.Series(dtype=str)
    liked_ids = liked_movies["movieId"].dropna().astype(int).unique() if "movieId" in liked_movies.columns else []
    movie_tags = tags_clean[tags_clean["movieId"].isin(liked_ids)][["movieId", "tag_clean"]].dropna().drop_duplicates().copy()
    if movie_tags.empty:
        return pd.Series(dtype=str)
    if discriminative_tag_profile is not None and not discriminative_tag_profile.empty and "discriminative_weight" in discriminative_tag_profile.columns:
        movie_tags = movie_tags.merge(discriminative_tag_profile[["tag_clean", "discriminative_weight"]], on="tag_clean", how="left")
        movie_tags["tag_weight"] = movie_tags["discriminative_weight"].fillna(0.0)
    else:
        movie_tags["tag_weight"] = 1.0
    movie_tags = movie_tags.sort_values(["movieId", "tag_weight", "tag_clean"], ascending=[True, False, True])
    return movie_tags.groupby("movieId")["tag_clean"].apply(lambda values: ", ".join(values.astype(str).drop_duplicates().head(max_tags)))


def build_user_branch_profile(liked_movies, tags_clean=None, discriminative_tag_profile=None, assign_branch_func=None):
    assign_branch_func = assign_branch_func or assign_semantic_branch
    liked = liked_movies.copy() if liked_movies is not None and not liked_movies.empty else pd.DataFrame()
    for col in ["movieId", "genres", "year", "user_rating_5"]:
        if col not in liked.columns:
            liked[col] = "" if col == "genres" else np.nan
    if "title" not in liked.columns:
        liked["title"] = ""
    liked["movieId"] = pd.to_numeric(liked["movieId"], errors="coerce")
    liked = liked.dropna(subset=["movieId"]).copy()
    liked["movieId"] = liked["movieId"].astype(int) if not liked.empty else liked.get("movieId", pd.Series(dtype=int))
    liked["user_rating_5"] = pd.to_numeric(liked["user_rating_5"], errors="coerce")
    if not liked.empty and "semantic_branch" not in liked.columns:
        top_tags = _top_tags_by_movie_for_branch(liked, tags_clean, discriminative_tag_profile)
        liked["core_semantic_explanation_terms"] = liked["movieId"].map(top_tags).fillna("")
        liked["semantic_explanation_terms"] = liked["core_semantic_explanation_terms"]
        if "main_genre" not in liked.columns:
            liked["main_genre"] = liked["genres"].apply(main_genre_from_genres)
        liked["semantic_branch"] = liked.apply(assign_branch_func, axis=1)
    if liked.empty:
        profile = pd.DataFrame(columns=["semantic_branch", "branch_weight", "n_seed_movies", "example_liked_movies"])
    else:
        liked["branch_seed_weight"] = (liked["user_rating_5"].fillna(4.0) - 3.0).clip(lower=0.5)
        profile = liked.groupby("semantic_branch", as_index=False).agg(
            branch_weight=("branch_seed_weight", "sum"),
            n_seed_movies=("movieId", "nunique"),
            example_liked_movies=("title", lambda values: " | ".join(values.astype(str).drop_duplicates().head(5))),
        )
    candidate_branches = set(candidates_scored["semantic_branch"].dropna().astype(str)) if "candidates_scored" in globals() and "semantic_branch" in candidates_scored.columns else set()
    all_branches = sorted(set(SEMANTIC_BRANCHES) | set(profile.get("semantic_branch", pd.Series(dtype=str))) | candidate_branches)
    profile = pd.DataFrame({"semantic_branch": all_branches}).merge(profile, on="semantic_branch", how="left")
    profile["branch_weight"] = pd.to_numeric(profile["branch_weight"], errors="coerce").fillna(0.0)
    profile["n_seed_movies"] = pd.to_numeric(profile["n_seed_movies"], errors="coerce").fillna(0).astype(int)
    profile["example_liked_movies"] = profile["example_liked_movies"].fillna("")
    total_weight = profile["branch_weight"].sum()
    profile["branch_share"] = profile["branch_weight"] / total_weight if total_weight > 0 else 0.0
    profile = profile.sort_values(["branch_share", "branch_weight", "semantic_branch"], ascending=[False, False, True]).reset_index(drop=True)
    profile["branch_rank"] = np.where(profile["branch_share"] > 0, np.arange(1, len(profile) + 1), np.nan)
    def classify_branch(row):
        share = row["branch_share"]
        rank = row["branch_rank"]
        if share <= 0:
            return "unseen"
        if share >= 0.18 or (pd.notna(rank) and rank <= 2):
            return "strong"
        if share >= 0.08:
            return "medium"
        if share >= 0.03:
            return "weak"
        return "unseen"
    profile["branch_strength"] = profile.apply(classify_branch, axis=1)
    return profile[["semantic_branch", "branch_weight", "branch_share", "branch_rank", "branch_strength", "n_seed_movies", "example_liked_movies"]]


def add_branch_affinity_to_candidates(candidates, user_branch_profile, branch_col="semantic_branch"):
    scored = candidates.copy()
    drop_cols = ["branch_share", "branch_strength", "n_branch_seed_movies", "branch_affinity_score", "branch_is_profile_supported"]
    scored = scored.drop(columns=[col for col in drop_cols if col in scored.columns])
    profile_cols = ["semantic_branch", "branch_share", "branch_strength", "n_seed_movies"]
    scored = scored.merge(user_branch_profile[profile_cols], left_on=branch_col, right_on="semantic_branch", how="left", suffixes=("", "_profile"))
    if "semantic_branch_profile" in scored.columns:
        scored["semantic_branch"] = scored["semantic_branch"].fillna(scored["semantic_branch_profile"])
        scored = scored.drop(columns=["semantic_branch_profile"])
    scored["branch_share"] = scored["branch_share"].fillna(0.0)
    scored["branch_strength"] = scored["branch_strength"].fillna("unseen")
    scored["n_branch_seed_movies"] = scored["n_seed_movies"].fillna(0).astype(int)
    scored = scored.drop(columns=["n_seed_movies"])
    base_scores = {"strong": 1.00, "medium": 0.75, "weak": 0.45, "unseen": 0.15}
    scored["branch_affinity_score"] = scored["branch_strength"].map(base_scores).fillna(0.15)
    scored["branch_affinity_score"] = np.maximum(scored["branch_affinity_score"], np.sqrt(scored["branch_share"].clip(lower=0.0)))
    scored["branch_affinity_score"] = scored["branch_affinity_score"].clip(0.05, 1.0)
    scored["branch_is_profile_supported"] = scored["branch_strength"].isin(["strong", "medium", "weak"])
    return scored


def build_discriminative_semantic_profile(
    tags_clean,
    liked_movies,
    disliked_movies,
    positive_tag_profile=None,
    negative_tag_profile=None,
    smoothing=0.10,
    min_positive_seed_movies=1,
    beta_negative=0.75,
):
    required_cols = {"movieId", "tag_clean", "idf"}
    if tags_clean is None or tags_clean.empty or not required_cols.issubset(tags_clean.columns):
        return pd.DataFrame(columns=[
            "tag_clean", "semantic_category", "positive_weight", "negative_weight",
            "positive_weight_norm", "negative_weight_norm", "positive_n_seed_movies",
            "negative_n_seed_movies", "tag_ambiguity", "discriminative_lift",
            "discriminative_weight_raw", "discriminative_weight",
            "negative_discriminative_weight", "is_core_positive_tag", "is_core_negative_tag",
        ])

    tag_cols = ["movieId", "tag_clean", "idf"] + (["semantic_category"] if "semantic_category" in tags_clean.columns else [])
    tag_features = tags_clean[tag_cols].copy()
    tag_features["movieId"] = pd.to_numeric(tag_features["movieId"], errors="coerce")
    tag_features["idf"] = pd.to_numeric(tag_features["idf"], errors="coerce").fillna(1.0)
    tag_features = tag_features.dropna(subset=["movieId", "tag_clean"])
    tag_features["movieId"] = tag_features["movieId"].astype(int)
    tag_features = tag_features.drop_duplicates(["movieId", "tag_clean"])

    positive_seed = liked_movies.loc[liked_movies["user_rating_5"] >= 4.0, ["movieId", "user_rating_5"]].dropna().copy()
    negative_seed = disliked_movies.loc[disliked_movies["user_rating_5"] <= 2.5, ["movieId", "user_rating_5"]].dropna().copy()
    positive_seed["movieId"] = positive_seed["movieId"].astype(int)
    negative_seed["movieId"] = negative_seed["movieId"].astype(int)

    positive_agg = pd.DataFrame(columns=["tag_clean", "positive_weight", "positive_n_seed_movies", "semantic_category"])
    if not positive_seed.empty:
        positive_tags = tag_features.merge(positive_seed, on="movieId", how="inner")
        positive_tags["positive_personal_weight"] = positive_tags["user_rating_5"] - 3.0
        positive_tags["contribution_positive"] = positive_tags["positive_personal_weight"] * positive_tags["idf"]
        agg_spec = {
            "positive_weight": ("contribution_positive", "sum"),
            "positive_n_seed_movies": ("movieId", "nunique"),
        }
        if "semantic_category" in positive_tags.columns:
            agg_spec["semantic_category"] = ("semantic_category", "first")
        positive_agg = positive_tags.groupby("tag_clean", as_index=False).agg(**agg_spec)
        if "semantic_category" not in positive_agg.columns:
            positive_agg["semantic_category"] = None

    negative_agg = pd.DataFrame(columns=["tag_clean", "negative_weight", "negative_n_seed_movies"])
    if not negative_seed.empty:
        negative_tags = tag_features.merge(negative_seed, on="movieId", how="inner")
        negative_tags["negative_personal_weight"] = 3.0 - negative_tags["user_rating_5"]
        negative_tags["contribution_negative"] = negative_tags["negative_personal_weight"] * negative_tags["idf"]
        negative_agg = negative_tags.groupby("tag_clean", as_index=False).agg(
            negative_weight=("contribution_negative", "sum"),
            negative_n_seed_movies=("movieId", "nunique"),
        )

    profile = positive_agg.merge(negative_agg, on="tag_clean", how="outer")
    if profile.empty:
        return pd.DataFrame(columns=[
            "tag_clean", "semantic_category", "positive_weight", "negative_weight",
            "positive_weight_norm", "negative_weight_norm", "positive_n_seed_movies",
            "negative_n_seed_movies", "tag_ambiguity", "discriminative_lift",
            "discriminative_weight_raw", "discriminative_weight",
            "negative_discriminative_weight", "is_core_positive_tag", "is_core_negative_tag",
        ])

    for col in ["positive_weight", "negative_weight", "positive_n_seed_movies", "negative_n_seed_movies"]:
        profile[col] = pd.to_numeric(profile[col], errors="coerce").fillna(0.0)
    if "semantic_category" not in profile.columns:
        profile["semantic_category"] = None

    positive_total = profile["positive_weight"].sum()
    negative_total = profile["negative_weight"].sum()
    profile["positive_weight_norm"] = profile["positive_weight"] / positive_total if positive_total > 0 else 0.0
    profile["negative_weight_norm"] = profile["negative_weight"] / negative_total if negative_total > 0 else 0.0
    profile["tag_ambiguity"] = np.minimum(profile["positive_weight_norm"], profile["negative_weight_norm"])
    profile["discriminative_lift"] = np.log(
        (profile["positive_weight_norm"] + smoothing) / (profile["negative_weight_norm"] + smoothing)
    )
    profile["discriminative_weight_raw"] = (
        profile["positive_weight_norm"]
        * profile["discriminative_lift"].clip(lower=0.0)
        * (1 - profile["tag_ambiguity"])
    )

    many_positive_movies = len(positive_seed) >= 10
    profile.loc[many_positive_movies & (profile["positive_n_seed_movies"] == 1), "discriminative_weight_raw"] *= 0.70
    profile.loc[profile["positive_n_seed_movies"] >= 3, "discriminative_weight_raw"] *= 1.10
    profile.loc[profile["tag_ambiguity"] > 0.20, "discriminative_weight_raw"] *= 0.50
    profile.loc[profile["tag_ambiguity"] > 0.35, "discriminative_weight_raw"] *= 0.25
    profile.loc[profile["positive_n_seed_movies"] < min_positive_seed_movies, "discriminative_weight_raw"] = 0.0

    max_positive = profile["discriminative_weight_raw"].max()
    profile["discriminative_weight"] = profile["discriminative_weight_raw"] / max_positive if max_positive > 0 else 0.0

    profile["negative_discriminative_lift"] = np.log(
        (profile["negative_weight_norm"] + smoothing) / (profile["positive_weight_norm"] + smoothing)
    )
    profile["negative_discriminative_weight_raw"] = (
        profile["negative_weight_norm"]
        * profile["negative_discriminative_lift"].clip(lower=0.0)
        * (1 - profile["tag_ambiguity"])
    )
    max_negative = profile["negative_discriminative_weight_raw"].max()
    profile["negative_discriminative_weight"] = profile["negative_discriminative_weight_raw"] / max_negative if max_negative > 0 else 0.0

    profile["is_core_positive_tag"] = profile["discriminative_weight"] > 0
    profile["is_core_negative_tag"] = profile["negative_discriminative_weight"] > 0

    return profile[[
        "tag_clean", "semantic_category", "positive_weight", "negative_weight",
        "positive_weight_norm", "negative_weight_norm", "positive_n_seed_movies",
        "negative_n_seed_movies", "tag_ambiguity", "discriminative_lift",
        "discriminative_weight_raw", "discriminative_weight",
        "negative_discriminative_weight", "is_core_positive_tag", "is_core_negative_tag",
    ]].sort_values("discriminative_weight", ascending=False).reset_index(drop=True)


def compute_semantic_core_scores(candidates, tags_clean, discriminative_tag_profile):
    result = candidates[["movieId"]].copy()
    result["semantic_core_raw_score"] = 0.0
    result["semantic_core_negative_raw_score"] = 0.0
    result["semantic_core_score"] = 0.0
    result["semantic_core_negative_score"] = 0.0
    result["core_semantic_explanation_terms"] = ""
    result["core_negative_semantic_terms"] = ""

    if tags_clean is None or tags_clean.empty or discriminative_tag_profile is None or discriminative_tag_profile.empty:
        return result

    candidate_tags = tags_clean[tags_clean["movieId"].isin(result["movieId"].astype(int))][["movieId", "tag_clean"]].drop_duplicates()
    if candidate_tags.empty:
        return result

    profile_cols = ["tag_clean", "discriminative_weight", "negative_discriminative_weight"]
    weighted_tags = candidate_tags.merge(discriminative_tag_profile[profile_cols], on="tag_clean", how="inner")
    if weighted_tags.empty:
        return result

    positive_contrib = weighted_tags[weighted_tags["discriminative_weight"].fillna(0.0) > 0].copy()
    if not positive_contrib.empty:
        positive_contrib["contribution"] = positive_contrib["discriminative_weight"]
        raw_positive = positive_contrib.groupby("movieId")["contribution"].sum()
        result["semantic_core_raw_score"] = result["movieId"].map(raw_positive).fillna(0.0)
        result["semantic_core_score"] = _rank_positive(result["semantic_core_raw_score"])
        result["core_semantic_explanation_terms"] = result["movieId"].map(
            _top_terms_from_weighted_tags(positive_contrib, value_col="contribution", max_terms=5)
        ).fillna("")

    negative_contrib = weighted_tags[weighted_tags["negative_discriminative_weight"].fillna(0.0) > 0].copy()
    if not negative_contrib.empty:
        negative_contrib["contribution"] = negative_contrib["negative_discriminative_weight"]
        raw_negative = negative_contrib.groupby("movieId")["contribution"].sum()
        result["semantic_core_negative_raw_score"] = result["movieId"].map(raw_negative).fillna(0.0)
        result["semantic_core_negative_score"] = _rank_positive(result["semantic_core_negative_raw_score"])
        result["core_negative_semantic_terms"] = result["movieId"].map(
            _top_terms_from_weighted_tags(negative_contrib, value_col="contribution", max_terms=5)
        ).fillna("")

    return result


def compute_semantic_net_score(df, beta=0.75):
    scored = df.copy()
    if {"semantic_core_raw_score", "semantic_core_negative_raw_score"}.issubset(scored.columns):
        semantic_raw = scored["semantic_core_raw_score"]
        negative_raw = scored["semantic_core_negative_raw_score"]
    else:
        semantic_raw = scored["semantic_raw_score"] if "semantic_raw_score" in scored.columns else scored.get("semantic_profile_score", 0.0)
        negative_raw = scored["negative_semantic_raw_score"] if "negative_semantic_raw_score" in scored.columns else scored.get("negative_semantic_score", 0.0)
    scored["semantic_net_raw_score"] = semantic_raw.fillna(0.0) - beta * negative_raw.fillna(0.0)
    scored["semantic_net_score"] = _rank_positive(scored["semantic_net_raw_score"])
    return scored


def _first_weight_column(df, preferred):
    if df is None or df.empty:
        return None
    for col in preferred:
        if col in df.columns:
            return col
    numeric_cols = [col for col in df.select_dtypes(include=[np.number]).columns if not col.lower().endswith("id")]
    return numeric_cols[0] if numeric_cols else None


def _normalized_tag_weights(df, preferred_weight_cols):
    if df is None or df.empty or "tag_clean" not in df.columns:
        return pd.DataFrame(columns=["tag_clean", "weight_norm"])
    weight_col = _first_weight_column(df, preferred_weight_cols)
    if weight_col is None:
        return pd.DataFrame(columns=["tag_clean", "weight_norm"])
    weights = df[["tag_clean", weight_col]].dropna().copy()
    weights[weight_col] = pd.to_numeric(weights[weight_col], errors="coerce").fillna(0.0).clip(lower=0.0)
    weights = weights.groupby("tag_clean", as_index=False)[weight_col].sum()
    total = weights[weight_col].sum()
    if total <= 0:
        return pd.DataFrame(columns=["tag_clean", "weight_norm"])
    weights["weight_norm"] = weights[weight_col] / total
    return weights[["tag_clean", "weight_norm"]]


def _semantic_ambiguity(positive_tag_profile, negative_tag_profile):
    positive_weights = _normalized_tag_weights(positive_tag_profile, ["positive_weight", "tag_weight", "weight"])
    negative_weights = _normalized_tag_weights(negative_tag_profile, ["negative_weight", "tag_weight", "weight"])
    if positive_weights.empty or negative_weights.empty:
        return 1.0, 0
    overlap = positive_weights.merge(negative_weights, on="tag_clean", how="inner", suffixes=("_pos", "_neg"))
    if overlap.empty:
        return 0.0, 0
    ambiguity = np.minimum(overlap["weight_norm_pos"], overlap["weight_norm_neg"]).sum()
    return _clip01(ambiguity), len(overlap)


def extract_genre_weights_from_profile(df, rating_col="user_rating_5", positive=True):
    if df is None or df.empty or "genres" not in df.columns or rating_col not in df.columns:
        return pd.DataFrame(columns=["genre", "genre_weight_norm"])
    rows = []
    for _, row in df.dropna(subset=[rating_col]).iterrows():
        base_weight = row[rating_col] - 3.0 if positive else 3.0 - row[rating_col]
        if base_weight <= 0:
            continue
        for genre in split_genres(row.get("genres", "")):
            rows.append({"genre": genre, "weight": base_weight})
    if not rows:
        return pd.DataFrame(columns=["genre", "genre_weight_norm"])
    weights = pd.DataFrame(rows).groupby("genre", as_index=False)["weight"].sum()
    total = weights["weight"].sum()
    if total <= 0:
        return pd.DataFrame(columns=["genre", "genre_weight_norm"])
    weights["genre_weight_norm"] = weights["weight"] / total
    return weights[["genre", "genre_weight_norm"]]



def _valid_year_series(df, year_col="year"):
    if df is None or df.empty or year_col not in df.columns:
        return pd.Series(dtype=float)
    years = pd.to_numeric(df[year_col], errors="coerce")
    return years[(years >= 1874) & (years <= 2035)].dropna()



def build_movie_semantic_documents(movies_df, tags_clean, max_tags_per_movie=30):
    docs = movies_df[["movieId", "genres"]].copy()
    docs["movieId"] = pd.to_numeric(docs["movieId"], errors="coerce")
    docs = docs.dropna(subset=["movieId"]).copy()
    docs["movieId"] = docs["movieId"].astype(int)
    docs["genres_text"] = docs["genres"].fillna("").astype(str).str.replace("|", " ", regex=False)

    if tags_clean is not None and not tags_clean.empty and {"movieId", "tag_clean"}.issubset(tags_clean.columns):
        tag_cols = ["movieId", "tag_clean"]
        if "semantic_category" in tags_clean.columns:
            tag_cols.append("semantic_category")
        if "idf" in tags_clean.columns:
            tag_cols.append("idf")
        tags_doc = tags_clean[tag_cols].copy()
        tags_doc["movieId"] = pd.to_numeric(tags_doc["movieId"], errors="coerce")
        tags_doc = tags_doc.dropna(subset=["movieId", "tag_clean"]).copy()
        tags_doc["movieId"] = tags_doc["movieId"].astype(int)
        tags_doc["idf"] = pd.to_numeric(tags_doc["idf"], errors="coerce").fillna(1.0) if "idf" in tags_doc.columns else 1.0
        tags_doc = tags_doc.sort_values(["movieId", "idf", "tag_clean"], ascending=[True, False, True])
        tags_doc = tags_doc.groupby("movieId").head(max_tags_per_movie)
        tags_text = tags_doc.groupby("movieId")["tag_clean"].apply(lambda s: " ".join(s.astype(str))).rename("tags_text").reset_index()
        docs = docs.merge(tags_text, on="movieId", how="left")
        if "semantic_category" in tags_doc.columns:
            categories_text = tags_doc.dropna(subset=["semantic_category"]).groupby("movieId")["semantic_category"].apply(
                lambda s: " ".join(s.astype(str).drop_duplicates())
            ).rename("semantic_categories_text").reset_index()
            docs = docs.merge(categories_text, on="movieId", how="left")
        else:
            docs["semantic_categories_text"] = ""
    else:
        docs["tags_text"] = ""
        docs["semantic_categories_text"] = ""

    for col in ["tags_text", "semantic_categories_text"]:
        if col not in docs.columns:
            docs[col] = ""
        docs[col] = docs[col].fillna("").astype(str)
    docs["semantic_document"] = (
        docs["genres_text"] + " " + docs["tags_text"] + " " + docs["semantic_categories_text"]
    ).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
    return docs[["movieId", "semantic_document"]]


def build_latent_semantic_space(movie_documents, n_components=64, max_features=8000, min_df=2, random_state=42):
    documents = movie_documents["semantic_document"].fillna("").astype(str)
    vectorizer = TfidfVectorizer(
        max_features=max_features,
        min_df=min_df,
        ngram_range=(1, 2),
        sublinear_tf=True,
        stop_words="english",
    )
    tfidf_matrix = vectorizer.fit_transform(documents)
    n_movies, n_features = tfidf_matrix.shape
    movie_ids = movie_documents["movieId"].astype(int).tolist()
    movie_id_to_latent_idx = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}
    latent_idx_to_movie_id = {idx: movie_id for idx, movie_id in enumerate(movie_ids)}
    n_components_safe = min(n_components, n_features - 1, n_movies - 1)
    if n_components_safe >= 2:
        svd_model = TruncatedSVD(n_components=n_components_safe, random_state=random_state)
        latent_vectors = normalize(svd_model.fit_transform(tfidf_matrix))
    else:
        svd_model = None
        latent_vectors = normalize(tfidf_matrix)
    return latent_vectors, vectorizer, svd_model, movie_id_to_latent_idx, latent_idx_to_movie_id


def _latent_seed_indices(profile_df, movie_id_to_latent_idx, positive=True):
    if profile_df is None or profile_df.empty or "movieId" not in profile_df.columns:
        return [], np.array([], dtype=float), []
    work = profile_df.copy()
    work["movieId"] = pd.to_numeric(work["movieId"], errors="coerce")
    if "user_rating_5" in work.columns:
        work["user_rating_5"] = pd.to_numeric(work["user_rating_5"], errors="coerce")
        work = work[work["user_rating_5"] >= 4.0] if positive else work[work["user_rating_5"] <= 2.5]
        weights = (work["user_rating_5"] - 3.0) if positive else (3.0 - work["user_rating_5"])
        work["latent_weight"] = weights.fillna(1.0).clip(lower=0.05)
    else:
        work["latent_weight"] = 1.0
    work = work.dropna(subset=["movieId"]).copy()
    indices, seed_weights, titles = [], [], []
    for _, row in work.iterrows():
        movie_id = int(row["movieId"])
        if movie_id in movie_id_to_latent_idx:
            indices.append(movie_id_to_latent_idx[movie_id])
            seed_weights.append(float(row.get("latent_weight", 1.0)))
            titles.append(str(row.get("title", f"movieId={movie_id}")))
    return indices, np.array(seed_weights, dtype=float), titles


def compute_latent_user_similarity_scores(
    candidates,
    liked_movies,
    disliked_movies,
    latent_vectors,
    movie_id_to_latent_idx,
    top_k_positive=5,
    top_k_negative=5,
):
    result = candidates[["movieId"]].copy()
    result["latent_core_similarity_raw"] = 0.0
    result["latent_core_similarity_score"] = 0.0
    result["negative_latent_similarity_raw"] = 0.0
    result["negative_latent_similarity_score"] = 0.0
    result["nearest_liked_movies_latent"] = ""
    result["nearest_disliked_movies_latent"] = ""
    result["n_positive_latent_neighbors"] = 0
    result["n_negative_latent_neighbors"] = 0

    candidate_ids = pd.to_numeric(result["movieId"], errors="coerce").astype("Int64")
    candidate_positions = [pos for pos, movie_id in enumerate(candidate_ids) if pd.notna(movie_id) and int(movie_id) in movie_id_to_latent_idx]
    candidate_indices = [movie_id_to_latent_idx[int(candidate_ids.iloc[pos])] for pos in candidate_positions]
    if not candidate_indices:
        return result
    candidate_vectors = latent_vectors[candidate_indices]

    def fill_similarity(profile_df, positive=True):
        seed_indices, seed_weights, seed_titles = _latent_seed_indices(profile_df, movie_id_to_latent_idx, positive=positive)
        if not seed_indices:
            return
        sims = cosine_similarity(candidate_vectors, latent_vectors[seed_indices])
        top_k = min(top_k_positive if positive else top_k_negative, sims.shape[1])
        raw_scores, nearest_titles = [], []
        for row_sims in sims:
            order = np.argsort(row_sims)[::-1][:top_k]
            top_sims = row_sims[order]
            top_weights = seed_weights[order]
            denom = top_weights.sum()
            mean_topk = float(np.average(top_sims, weights=top_weights)) if denom > 0 else float(top_sims.mean())
            max_topk = float(top_sims.max()) if len(top_sims) else 0.0
            raw_scores.append(0.70 * mean_topk + 0.30 * max_topk)
            title_order = order[:3]
            nearest_titles.append(" | ".join(seed_titles[i] for i in title_order if i < len(seed_titles)))
        target_raw = "latent_core_similarity_raw" if positive else "negative_latent_similarity_raw"
        target_score = "latent_core_similarity_score" if positive else "negative_latent_similarity_score"
        target_titles = "nearest_liked_movies_latent" if positive else "nearest_disliked_movies_latent"
        target_n = "n_positive_latent_neighbors" if positive else "n_negative_latent_neighbors"
        raw_series = pd.Series(raw_scores, index=result.index[candidate_positions])
        result.loc[raw_series.index, target_raw] = raw_series
        result[target_score] = _rank_positive(result[target_raw])
        result.loc[raw_series.index, target_titles] = nearest_titles
        result.loc[raw_series.index, target_n] = len(seed_indices)

    fill_similarity(liked_movies, positive=True)
    fill_similarity(disliked_movies, positive=False)
    return result


def compute_positive_anchor_evidence(
    liked_movies,
    tags_clean,
    discriminative_tag_profile,
    user_branch_profile=None,
    assign_branch_func=None,
):
    output_cols = [
        "movieId", "title", "year", "genres", "user_rating_5",
        "semantic_branch", "branch_share", "branch_strength",
        "core_tag_weight_sum", "core_tag_count", "core_category_count",
        "mean_idf", "ambiguity_mean", "negative_tag_overlap_sum",
        "centrality_score", "anchor_weight",
    ]
    if liked_movies is None or liked_movies.empty or "movieId" not in liked_movies.columns:
        return pd.DataFrame(columns=output_cols)

    liked = liked_movies.copy()
    liked["movieId"] = pd.to_numeric(liked["movieId"], errors="coerce")
    liked["user_rating_5"] = pd.to_numeric(liked.get("user_rating_5"), errors="coerce")
    liked = liked.dropna(subset=["movieId", "user_rating_5"]).copy()
    if liked.empty:
        return pd.DataFrame(columns=output_cols)
    liked["movieId"] = liked["movieId"].astype(int)

    positive_seed = liked[liked["user_rating_5"] >= 4.0].copy()
    if len(positive_seed) < 8:
        positive_seed = liked[liked["user_rating_5"] >= 3.5].copy()
    if positive_seed.empty:
        return pd.DataFrame(columns=output_cols)

    base_cols = [col for col in ["movieId", "title", "year", "genres", "user_rating_5", "semantic_branch"] if col in positive_seed.columns]
    movie_base = positive_seed[base_cols].drop_duplicates("movieId").copy()
    for col in ["title", "year", "genres", "semantic_branch"]:
        if col not in movie_base.columns:
            movie_base[col] = "" if col in ["title", "genres", "semantic_branch"] else np.nan

    if tags_clean is None or tags_clean.empty or not {"movieId", "tag_clean"}.issubset(tags_clean.columns):
        evidence = movie_base.copy()
        for col in ["core_tag_weight_sum", "core_tag_count", "core_category_count", "mean_idf", "ambiguity_mean", "negative_tag_overlap_sum"]:
            evidence[col] = 0.0
    else:
        tag_cols = ["movieId", "tag_clean"]
        for col in ["semantic_category", "idf"]:
            if col in tags_clean.columns:
                tag_cols.append(col)
        movie_tags = tags_clean[tag_cols].copy()
        movie_tags["movieId"] = pd.to_numeric(movie_tags["movieId"], errors="coerce")
        movie_tags = movie_tags.dropna(subset=["movieId", "tag_clean"]).copy()
        movie_tags["movieId"] = movie_tags["movieId"].astype(int)
        if "semantic_category" not in movie_tags.columns:
            movie_tags["semantic_category"] = np.nan
        if "idf" not in movie_tags.columns:
            movie_tags["idf"] = 1.0
        movie_tags["idf"] = pd.to_numeric(movie_tags["idf"], errors="coerce").fillna(1.0)
        movie_tags = movie_tags.drop_duplicates(["movieId", "tag_clean"])
        movie_tags = movie_tags.merge(movie_base[["movieId"]], on="movieId", how="inner")

        profile_cols = ["tag_clean"]
        if discriminative_tag_profile is not None and not discriminative_tag_profile.empty:
            for col in ["discriminative_weight", "negative_discriminative_weight", "tag_ambiguity", "semantic_category"]:
                if col in discriminative_tag_profile.columns:
                    profile_cols.append(col)
            weighted_tags = movie_tags.merge(discriminative_tag_profile[profile_cols].drop_duplicates("tag_clean"), on="tag_clean", how="left", suffixes=("", "_profile"))
        else:
            weighted_tags = movie_tags.copy()
        if "semantic_category_profile" in weighted_tags.columns:
            weighted_tags["semantic_category"] = weighted_tags["semantic_category"].fillna(weighted_tags["semantic_category_profile"])
            weighted_tags = weighted_tags.drop(columns=["semantic_category_profile"])
        for col in ["discriminative_weight", "negative_discriminative_weight", "tag_ambiguity"]:
            if col not in weighted_tags.columns:
                weighted_tags[col] = 0.0
            weighted_tags[col] = pd.to_numeric(weighted_tags[col], errors="coerce").fillna(0.0)

        if weighted_tags.empty:
            evidence = movie_base.copy()
            for col in ["core_tag_weight_sum", "core_tag_count", "core_category_count", "mean_idf", "ambiguity_mean", "negative_tag_overlap_sum"]:
                evidence[col] = 0.0
        else:
            useful_tags = weighted_tags[weighted_tags["discriminative_weight"] > 0].copy()
            category_counts = useful_tags.dropna(subset=["semantic_category"]).groupby("movieId")["semantic_category"].nunique()
            top_tags = weighted_tags.sort_values(["movieId", "discriminative_weight", "idf", "tag_clean"], ascending=[True, False, False, True]).groupby("movieId")["tag_clean"].apply(
                lambda values: ", ".join(values.astype(str).drop_duplicates().head(8))
            )
            grouped = weighted_tags.groupby("movieId").agg(
                core_tag_weight_sum=("discriminative_weight", "sum"),
                core_tag_weight_mean=("discriminative_weight", "mean"),
                core_tag_count=("discriminative_weight", lambda values: int((pd.to_numeric(values, errors="coerce").fillna(0.0) > 0).sum())),
                mean_idf=("idf", "mean"),
                ambiguity_mean=("tag_ambiguity", "mean"),
                negative_tag_overlap_sum=("negative_discriminative_weight", "sum"),
            ).reset_index()
            grouped["core_category_count"] = grouped["movieId"].map(category_counts).fillna(0).astype(int)
            grouped["top_tags"] = grouped["movieId"].map(top_tags).fillna("")
            evidence = movie_base.merge(grouped, on="movieId", how="left")

    for col in ["core_tag_weight_sum", "core_tag_weight_mean", "core_tag_count", "core_category_count", "mean_idf", "ambiguity_mean", "negative_tag_overlap_sum"]:
        if col not in evidence.columns:
            evidence[col] = 0.0
        fill_value = 1.0 if col == "mean_idf" else 0.0
        evidence[col] = pd.to_numeric(evidence[col], errors="coerce").fillna(fill_value)
    if "top_tags" not in evidence.columns:
        evidence["top_tags"] = ""

    evidence["core_tag_weight_sum_norm"] = rank_normalize_positive(evidence["core_tag_weight_sum"])
    evidence["core_tag_count_norm"] = rank_normalize_positive(evidence["core_tag_count"])
    evidence["core_category_count_norm"] = rank_normalize_positive(evidence["core_category_count"])
    evidence["mean_idf_norm"] = rank_normalize_positive(evidence["mean_idf"])
    evidence["negative_tag_overlap_norm"] = rank_normalize_positive(evidence["negative_tag_overlap_sum"])

    if "semantic_branch" not in evidence.columns or evidence["semantic_branch"].fillna("").eq("").all():
        evidence["semantic_branch"] = "general"
    if assign_branch_func is not None:
        evidence["core_semantic_explanation_terms"] = evidence["top_tags"].fillna("")
        evidence["semantic_explanation_terms"] = evidence["top_tags"].fillna("")
        if "main_genre" not in evidence.columns:
            evidence["main_genre"] = evidence["genres"].apply(main_genre_from_genres)
        missing_branch = evidence["semantic_branch"].isna() | evidence["semantic_branch"].astype(str).str.strip().eq("") | evidence["semantic_branch"].eq("general")
        evidence.loc[missing_branch, "semantic_branch"] = evidence.loc[missing_branch].apply(assign_branch_func, axis=1)
    evidence["semantic_branch"] = evidence["semantic_branch"].fillna("general").astype(str)

    if user_branch_profile is not None and not user_branch_profile.empty and "semantic_branch" in user_branch_profile.columns:
        branch_cols = [col for col in ["semantic_branch", "branch_share", "branch_strength"] if col in user_branch_profile.columns]
        evidence = evidence.drop(columns=[col for col in ["branch_share", "branch_strength"] if col in evidence.columns]).merge(
            user_branch_profile[branch_cols].drop_duplicates("semantic_branch"), on="semantic_branch", how="left"
        )
    if "branch_share" not in evidence.columns:
        evidence["branch_share"] = 0.5
    if "branch_strength" not in evidence.columns:
        evidence["branch_strength"] = "unknown"
    evidence["branch_share"] = pd.to_numeric(evidence["branch_share"], errors="coerce").fillna(0.5).clip(0.0, 1.0)
    evidence["branch_strength"] = evidence["branch_strength"].fillna("unknown").astype(str)

    evidence["rating_weight"] = (pd.to_numeric(evidence["user_rating_5"], errors="coerce").fillna(3.5) - 3.0).clip(lower=0.5)
    evidence["centrality_score"] = (
        0.30 * evidence["core_tag_weight_sum_norm"]
        + 0.20 * evidence["core_category_count_norm"]
        + 0.15 * evidence["mean_idf_norm"]
        + 0.15 * evidence["core_tag_count_norm"]
        + 0.15 * evidence["branch_share"]
        - 0.15 * evidence["negative_tag_overlap_norm"]
        - 0.10 * evidence["ambiguity_mean"].clip(0.0, 1.0)
    ).clip(0.0, 1.0)
    evidence["anchor_weight"] = evidence["rating_weight"] * (0.30 + 0.70 * evidence["centrality_score"])
    sparse_anchor_mask = (evidence["core_tag_count"] <= 1) & (evidence["core_category_count"] <= 1)
    evidence.loc[sparse_anchor_mask, "anchor_weight"] *= 0.60
    evidence.loc[evidence["branch_strength"].eq("unseen"), "anchor_weight"] *= 0.40

    return evidence[output_cols].sort_values("anchor_weight", ascending=False).reset_index(drop=True)


def select_core_positive_anchors(positive_anchor_evidence, top_n_anchors=30, min_anchors=12):
    output_cols = [
        "movieId", "title", "year", "genres", "user_rating_5", "semantic_branch",
        "branch_share", "branch_strength", "centrality_score", "anchor_weight",
        "is_core_anchor", "anchor_rank", "branch_anchor_limit", "selected_reason",
    ]
    if positive_anchor_evidence is None or positive_anchor_evidence.empty:
        return pd.DataFrame(columns=output_cols)
    ordered = positive_anchor_evidence.sort_values("anchor_weight", ascending=False).reset_index(drop=True).copy()

    def branch_limit(row):
        share = float(row.get("branch_share", 0.0) or 0.0)
        strength = str(row.get("branch_strength", "unknown"))
        expected_slots = top_n_anchors * share
        if strength == "strong":
            limit = max(5, int(np.ceil(expected_slots + 2)))
        elif strength == "medium":
            limit = min(7, max(3, int(np.ceil(expected_slots + 1))))
        elif strength == "weak":
            limit = min(3, max(1, int(np.ceil(expected_slots))))
        elif strength == "unseen":
            limit = 1
        else:
            limit = 3
        return int(max(1, min(top_n_anchors, limit)))

    ordered["branch_anchor_limit"] = ordered.apply(branch_limit, axis=1)
    selected_indices = []
    selected_reasons = {}
    branch_counts = {}
    for idx, row in ordered.iterrows():
        branch = row.get("semantic_branch", "general")
        if branch_counts.get(branch, 0) >= int(row.get("branch_anchor_limit", 1)):
            continue
        selected_indices.append(idx)
        selected_reasons[idx] = "core_anchor"
        branch_counts[branch] = branch_counts.get(branch, 0) + 1
        if len(selected_indices) >= top_n_anchors:
            break
    min_target = min(min_anchors, len(ordered), top_n_anchors)
    if len(selected_indices) < min_target:
        for idx, _ in ordered.iterrows():
            if idx in selected_indices:
                continue
            selected_indices.append(idx)
            selected_reasons[idx] = "minimum_fill"
            if len(selected_indices) >= min_target:
                break
    selected = ordered.loc[selected_indices].copy()
    selected["is_core_anchor"] = True
    selected["anchor_rank"] = np.arange(1, len(selected) + 1)
    selected["selected_reason"] = selected.index.map(selected_reasons).fillna("core_anchor")
    return selected[output_cols].sort_values("anchor_rank").reset_index(drop=True)


def _dense_latent_rows(latent_vectors, indices):
    rows = latent_vectors[indices]
    return rows.toarray() if hasattr(rows, "toarray") else np.asarray(rows)


def compute_latent_core_preference_scores(candidates, core_positive_anchors, disliked_movies, latent_vectors, movie_id_to_latent_idx, user_branch_profile=None, top_k_positive=5, top_k_negative=5):
    result = candidates[["movieId"]].copy()
    defaults = {
        "positive_centroid_similarity": 0.0, "topk_anchor_similarity": 0.0,
        "branch_prototype_similarity": 0.0, "negative_latent_preference_raw": 0.0,
        "negative_latent_preference_score": 0.0, "latent_core_preference_raw": 0.0,
        "latent_core_preference_score": 0.0, "nearest_core_anchor_movies": "",
        "nearest_negative_movies_latent": "", "n_core_anchors_used": 0, "n_negative_anchors_used": 0,
    }
    for col, default in defaults.items():
        result[col] = default
    if candidates is None or candidates.empty or core_positive_anchors is None or core_positive_anchors.empty:
        return result

    candidate_ids = pd.to_numeric(result["movieId"], errors="coerce").astype("Int64")
    candidate_positions = [pos for pos, movie_id in enumerate(candidate_ids) if pd.notna(movie_id) and int(movie_id) in movie_id_to_latent_idx]
    if not candidate_positions:
        return result
    candidate_indices = [movie_id_to_latent_idx[int(candidate_ids.iloc[pos])] for pos in candidate_positions]
    candidate_vectors = _dense_latent_rows(latent_vectors, candidate_indices)

    anchors = core_positive_anchors.copy()
    anchors["movieId"] = pd.to_numeric(anchors["movieId"], errors="coerce")
    anchors = anchors.dropna(subset=["movieId"]).copy()
    anchors["movieId"] = anchors["movieId"].astype(int)
    anchors = anchors[anchors["movieId"].isin(movie_id_to_latent_idx)].copy()
    if anchors.empty:
        return result
    anchor_indices = [movie_id_to_latent_idx[int(movie_id)] for movie_id in anchors["movieId"]]
    anchor_vectors = _dense_latent_rows(latent_vectors, anchor_indices)
    anchor_weights = pd.to_numeric(anchors.get("anchor_weight", 1.0), errors="coerce").fillna(1.0).clip(lower=0.05).to_numpy(dtype=float)
    anchor_titles = anchors.get("title", pd.Series([""] * len(anchors))).fillna("").astype(str).tolist()

    positive_centroid = np.average(anchor_vectors, axis=0, weights=anchor_weights)
    positive_centroid = normalize(np.asarray(positive_centroid).reshape(1, -1))[0]
    result.loc[result.index[candidate_positions], "positive_centroid_similarity"] = cosine_similarity(candidate_vectors, positive_centroid.reshape(1, -1)).ravel()
    result.loc[result.index[candidate_positions], "n_core_anchors_used"] = len(anchors)

    anchor_sims = cosine_similarity(candidate_vectors, anchor_vectors)
    topk_scores, nearest_anchor_titles = [], []
    top_k_pos = min(top_k_positive, anchor_sims.shape[1])
    for row_sims in anchor_sims:
        order = np.argsort(row_sims)[::-1][:top_k_pos]
        top_sims = row_sims[order]
        top_weights = anchor_weights[order]
        mean_topk = float(np.average(top_sims, weights=top_weights)) if top_weights.sum() > 0 else float(top_sims.mean())
        max_topk = float(top_sims.max()) if len(top_sims) else 0.0
        topk_scores.append(0.70 * mean_topk + 0.30 * max_topk)
        nearest_anchor_titles.append(" | ".join(anchor_titles[i] for i in order[:3] if i < len(anchor_titles)))
    result.loc[result.index[candidate_positions], "topk_anchor_similarity"] = topk_scores
    result.loc[result.index[candidate_positions], "nearest_core_anchor_movies"] = nearest_anchor_titles

    branch_prototypes = {}
    for branch, group in anchors.groupby("semantic_branch", dropna=False):
        branch = str(branch) if pd.notna(branch) else "general"
        positions = [anchors.index.get_loc(idx) for idx in group.index]
        prototype = np.average(anchor_vectors[positions], axis=0, weights=anchor_weights[positions])
        branch_prototypes[branch] = normalize(np.asarray(prototype).reshape(1, -1))[0]
    branch_strength_lookup = {}
    if user_branch_profile is not None and not user_branch_profile.empty and "semantic_branch" in user_branch_profile.columns:
        branch_strength_lookup = user_branch_profile.set_index("semantic_branch").get("branch_strength", pd.Series(dtype=str)).to_dict()
    strong_medium_branches = [b for b in branch_prototypes if branch_strength_lookup.get(b) in ["strong", "medium"]] or list(branch_prototypes)

    meta_cols = [col for col in ["movieId", "semantic_branch", "semantic_core_score"] if col in candidates.columns]
    candidate_meta = candidates[meta_cols].copy() if meta_cols else candidates[["movieId"]].copy()
    if "semantic_branch" not in candidate_meta.columns:
        candidate_meta["semantic_branch"] = "general"
    candidate_meta["movieId"] = pd.to_numeric(candidate_meta["movieId"], errors="coerce")
    result_meta = result[["movieId"]].copy()
    result_meta["movieId"] = pd.to_numeric(result_meta["movieId"], errors="coerce")
    result_meta = result_meta.merge(candidate_meta.drop_duplicates("movieId"), on="movieId", how="left")
    result_meta["semantic_branch"] = result_meta["semantic_branch"].fillna("general").astype(str)

    branch_scores = []
    for pos, cand_idx in enumerate(candidate_positions):
        candidate_vector = candidate_vectors[pos].reshape(1, -1)
        branch = result_meta.iloc[cand_idx].get("semantic_branch", "general")
        if branch in branch_prototypes:
            branch_scores.append(float(cosine_similarity(candidate_vector, branch_prototypes[branch].reshape(1, -1))[0, 0]))
        elif strong_medium_branches:
            sims = [float(cosine_similarity(candidate_vector, branch_prototypes[b].reshape(1, -1))[0, 0]) for b in strong_medium_branches]
            branch_scores.append(max(sims) * 0.65 if sims else 0.0)
        else:
            branch_scores.append(0.0)
    result.loc[result.index[candidate_positions], "branch_prototype_similarity"] = branch_scores

    negative_indices, negative_weights, negative_titles = [], [], []
    if disliked_movies is not None and not disliked_movies.empty and "movieId" in disliked_movies.columns:
        negatives = disliked_movies.copy()
        negatives["movieId"] = pd.to_numeric(negatives["movieId"], errors="coerce")
        negatives["user_rating_5"] = pd.to_numeric(negatives.get("user_rating_5"), errors="coerce")
        negatives = negatives[(negatives["user_rating_5"] <= 2.5) & negatives["movieId"].notna()].copy()
        for _, row in negatives.iterrows():
            movie_id = int(row["movieId"])
            if movie_id in movie_id_to_latent_idx:
                negative_indices.append(movie_id_to_latent_idx[movie_id])
                negative_weights.append(max(3.0 - float(row.get("user_rating_5", 2.5)), 0.05))
                negative_titles.append(str(row.get("title", f"movieId={movie_id}")))
    if negative_indices:
        negative_vectors = _dense_latent_rows(latent_vectors, negative_indices)
        negative_sims = cosine_similarity(candidate_vectors, negative_vectors)
        negative_weights = np.array(negative_weights, dtype=float)
        top_k_neg = min(top_k_negative, negative_sims.shape[1])
        raw_scores, nearest_titles = [], []
        for row_sims in negative_sims:
            order = np.argsort(row_sims)[::-1][:top_k_neg]
            top_sims = row_sims[order]
            top_weights = negative_weights[order]
            mean_topk = float(np.average(top_sims, weights=top_weights)) if top_weights.sum() > 0 else float(top_sims.mean())
            max_topk = float(top_sims.max()) if len(top_sims) else 0.0
            raw_scores.append(0.70 * mean_topk + 0.30 * max_topk)
            nearest_titles.append(" | ".join(negative_titles[i] for i in order[:3] if i < len(negative_titles)))
        result.loc[result.index[candidate_positions], "negative_latent_preference_raw"] = raw_scores
        result.loc[result.index[candidate_positions], "nearest_negative_movies_latent"] = nearest_titles
        result.loc[result.index[candidate_positions], "n_negative_anchors_used"] = len(negative_indices)
    result["negative_latent_preference_score"] = rank_normalize_positive(result["negative_latent_preference_raw"])

    if user_branch_profile is not None and not user_branch_profile.empty and {"semantic_branch", "branch_share", "branch_strength"}.issubset(user_branch_profile.columns):
        branch_meta = result_meta.merge(user_branch_profile[["semantic_branch", "branch_share", "branch_strength"]].drop_duplicates("semantic_branch"), on="semantic_branch", how="left")
    else:
        branch_meta = result_meta.copy()
        branch_meta["branch_share"] = 0.0
        branch_meta["branch_strength"] = "unseen"
    branch_multiplier = branch_meta["branch_strength"].fillna("unseen").map({"strong":1.00, "medium":0.88, "weak":0.68, "unseen":0.45, "unknown":0.70}).fillna(0.70)
    if "semantic_core_score" in result_meta.columns:
        semantic_support_multiplier = 0.65 + 0.35 * pd.to_numeric(result_meta["semantic_core_score"], errors="coerce").fillna(0.0).clip(0.0, 1.0)
    else:
        semantic_support_multiplier = pd.Series(1.0, index=result.index)

    raw = 0.42 * result["positive_centroid_similarity"] + 0.38 * result["topk_anchor_similarity"] + 0.20 * result["branch_prototype_similarity"] - 0.50 * result["negative_latent_preference_raw"]
    result["latent_core_preference_raw"] = (raw * branch_multiplier.reindex(result.index).fillna(0.70) * semantic_support_multiplier.reindex(result.index).fillna(1.0)).fillna(0.0)
    result["latent_core_preference_score"] = rank_normalize_positive(result["latent_core_preference_raw"])
    return result


def compute_semantic_relevance_scores(df):
    scored = df.copy()

    if "semantic_net_score" in scored.columns:
        semantic_fallback = scored["semantic_net_score"]
    elif "semantic_core_score" in scored.columns:
        semantic_fallback = scored["semantic_core_score"]
    else:
        semantic_fallback = scored.get("semantic_profile_score", 0.0)

    if "latent_core_preference_score" in scored.columns and pd.to_numeric(scored["latent_core_preference_score"], errors="coerce").fillna(0.0).var() > 0:
        latent_core = pd.to_numeric(scored["latent_core_preference_score"], errors="coerce").fillna(0.0)
        if "semantic_net_score" in scored.columns:
            semantic_net = pd.to_numeric(scored["semantic_net_score"], errors="coerce").fillna(0.0)
            scored["semantic_relevance_raw"] = 0.82 * latent_core + 0.18 * semantic_net
        else:
            scored["semantic_relevance_raw"] = latent_core
    else:
        scored["semantic_relevance_raw"] = pd.Series(semantic_fallback, index=scored.index).fillna(0.0)
    scored["semantic_relevance_score"] = rank_normalize_positive(scored["semantic_relevance_raw"])

    if "semantic_core_negative_score" in scored.columns:
        negative_existing = scored["semantic_core_negative_score"]
    else:
        negative_existing = scored.get("negative_semantic_score", 0.0)

    if "negative_latent_preference_score" in scored.columns:
        negative_existing_series = pd.Series(negative_existing, index=scored.index).fillna(0.0)
        scored["negative_semantic_relevance_raw"] = (
            0.65 * pd.to_numeric(scored["negative_latent_preference_score"], errors="coerce").fillna(0.0)
            + 0.35 * negative_existing_series
        )
    else:
        scored["negative_semantic_relevance_raw"] = pd.Series(negative_existing, index=scored.index).fillna(0.0)
    scored["negative_semantic_relevance_score"] = rank_normalize_positive(scored["negative_semantic_relevance_raw"])

    if "branch_strength" in scored.columns:
        branch_multiplier = scored["branch_strength"].fillna("unknown").map({
            "strong": 1.00,
            "medium": 0.82,
            "weak": 0.62,
            "unseen": 0.40,
            "unknown": 0.70,
        }).fillna(0.70)
        if "latent_core_preference_score" in scored.columns:
            latent_core = pd.to_numeric(scored["latent_core_preference_score"], errors="coerce").fillna(0.0)
            p90 = latent_core.quantile(0.90) if len(latent_core) else np.inf
            branch_multiplier = branch_multiplier.mask(latent_core >= p90, np.maximum(branch_multiplier, 0.75))
        scored["branch_support_multiplier"] = branch_multiplier
        scored["semantic_relevance_adjusted_score"] = rank_normalize_positive(scored["semantic_relevance_score"].fillna(0.0) * branch_multiplier)
    else:
        scored["branch_support_multiplier"] = 1.0
        scored["semantic_relevance_adjusted_score"] = scored["semantic_relevance_score"].fillna(0.0).clip(0.0, 1.0)
    return scored

def analyze_temporal_profile(
    user_movies,
    liked_movies=None,
    watched_movies=None,
    year_col="year",
    rating_col="user_rating_5",
):
    liked_years = _valid_year_series(liked_movies, year_col)
    user_years = _valid_year_series(user_movies, year_col)
    watched_years = _valid_year_series(watched_movies, year_col)

    profile_years = liked_years if len(liked_years) >= 5 else user_years
    if len(profile_years) < 5 and len(watched_years) > len(profile_years):
        profile_years = watched_years

    liked_bounds_years = liked_years if not liked_years.empty else profile_years

    if profile_years.empty:
        return {
            "n_temporal_movies": 0,
            "min_year": np.nan,
            "max_year": np.nan,
            "q10_year": np.nan,
            "q25_year": np.nan,
            "median_year": np.nan,
            "q75_year": np.nan,
            "q90_year": np.nan,
            "mean_year": np.nan,
            "std_year": np.nan,
            "earliest_liked_year": liked_bounds_years.min() if not liked_bounds_years.empty else np.nan,
            "latest_liked_year": liked_bounds_years.max() if not liked_bounds_years.empty else np.nan,
            "pct_before_1960": 0.0,
            "pct_before_1970": 0.0,
            "pct_before_1980": 0.0,
            "pct_before_1990": 0.0,
            "pct_after_2000": 0.0,
            "pct_after_2010": 0.0,
            "temporal_profile_strength": 0.0,
            "classic_tolerance": 0.0,
            "temporal_strictness": 0.0,
        }

    n_temporal_movies = len(profile_years)
    q10_year = float(profile_years.quantile(0.10))
    q90_year = float(profile_years.quantile(0.90))
    pct_before_1960 = float((profile_years < 1960).mean())
    pct_before_1970 = float((profile_years < 1970).mean())
    pct_before_1980 = float((profile_years < 1980).mean())
    pct_before_1990 = float((profile_years < 1990).mean())
    pct_after_2000 = float((profile_years >= 2000).mean())
    pct_after_2010 = float((profile_years >= 2010).mean())

    if n_temporal_movies < 5:
        temporal_profile_strength = min(0.4, n_temporal_movies / 10)
    else:
        temporal_profile_strength = _clip01(
            min(1.0, n_temporal_movies / 30) * 0.60
            + min(1.0, max(1.0, q90_year - q10_year) / 40) * 0.10
            + 0.30
        )

    classic_tolerance = _clip01(
        0.50 * pct_before_1980
        + 0.30 * pct_before_1970
        + 0.20 * pct_before_1960
    )
    temporal_strictness = _clip01(temporal_profile_strength * (1 - classic_tolerance))

    return {
        "n_temporal_movies": int(n_temporal_movies),
        "min_year": float(profile_years.min()),
        "max_year": float(profile_years.max()),
        "q10_year": q10_year,
        "q25_year": float(profile_years.quantile(0.25)),
        "median_year": float(profile_years.median()),
        "q75_year": float(profile_years.quantile(0.75)),
        "q90_year": q90_year,
        "mean_year": float(profile_years.mean()),
        "std_year": float(profile_years.std(ddof=0)) if n_temporal_movies > 1 else 0.0,
        "earliest_liked_year": float(liked_bounds_years.min()) if not liked_bounds_years.empty else np.nan,
        "latest_liked_year": float(liked_bounds_years.max()) if not liked_bounds_years.empty else np.nan,
        "pct_before_1960": pct_before_1960,
        "pct_before_1970": pct_before_1970,
        "pct_before_1980": pct_before_1980,
        "pct_before_1990": pct_before_1990,
        "pct_after_2000": pct_after_2000,
        "pct_after_2010": pct_after_2010,
        "temporal_profile_strength": temporal_profile_strength,
        "classic_tolerance": classic_tolerance,
        "temporal_strictness": temporal_strictness,
    }


def compute_year_affinity_scores(candidates, temporal_metrics, year_col="year"):
    scored = candidates.copy()
    temporal_profile_strength = float(temporal_metrics.get("temporal_profile_strength", 0.0) or 0.0)
    temporal_strictness = float(temporal_metrics.get("temporal_strictness", 0.0) or 0.0)
    classic_tolerance = float(temporal_metrics.get("classic_tolerance", 0.0) or 0.0)

    scored["year_affinity_score"] = 1.0
    scored["temporal_mismatch_penalty"] = 0.0
    scored["temporal_distance_from_profile"] = 0.0
    scored["is_temporal_outlier"] = False

    years = pd.to_numeric(scored.get(year_col), errors="coerce")
    invalid_year_mask = years.isna()
    if invalid_year_mask.any():
        scored.loc[invalid_year_mask, "year_affinity_score"] = 0.5
        scored.loc[invalid_year_mask, "temporal_mismatch_penalty"] = 0.5 * temporal_strictness
        scored.loc[invalid_year_mask, "temporal_distance_from_profile"] = np.nan
        scored.loc[invalid_year_mask, "is_temporal_outlier"] = True

    if temporal_profile_strength < 0.25:
        scored.loc[~invalid_year_mask, "year_affinity_score"] = 1.0
        scored.loc[~invalid_year_mask, "temporal_mismatch_penalty"] = 0.0
        scored.loc[~invalid_year_mask, "temporal_distance_from_profile"] = 0.0
        scored.loc[~invalid_year_mask, "is_temporal_outlier"] = False
        return scored

    q10_year = temporal_metrics.get("q10_year")
    q90_year = temporal_metrics.get("q90_year")
    if pd.isna(q10_year) or pd.isna(q90_year):
        return scored

    preferred_low = float(q10_year) - 5
    preferred_high = float(q90_year) + 5
    valid_year_mask = ~invalid_year_mask
    before_mask = valid_year_mask & (years < preferred_low)
    after_mask = valid_year_mask & (years > preferred_high)
    inside_mask = valid_year_mask & ~(before_mask | after_mask)

    scored.loc[inside_mask, "year_affinity_score"] = 1.0
    scored.loc[inside_mask, "temporal_distance_from_profile"] = 0.0

    if after_mask.any():
        distance = years.loc[after_mask] - preferred_high
        scored.loc[after_mask, "year_affinity_score"] = np.maximum(np.exp(-distance / 35), 0.65)
        scored.loc[after_mask, "temporal_distance_from_profile"] = distance

    if before_mask.any():
        distance = preferred_low - years.loc[before_mask]
        decay = 15 + 35 * classic_tolerance
        floor = 0.35 if classic_tolerance >= 0.30 else 0.05
        scored.loc[before_mask, "year_affinity_score"] = np.maximum(np.exp(-distance / decay), floor)
        scored.loc[before_mask, "temporal_distance_from_profile"] = distance

    scored["year_affinity_score"] = scored["year_affinity_score"].clip(0.0, 1.0)
    scored["temporal_mismatch_penalty"] = (1 - scored["year_affinity_score"]) * temporal_strictness
    scored.loc[invalid_year_mask, "temporal_mismatch_penalty"] = 0.5 * temporal_strictness
    scored["is_temporal_outlier"] = (
        (years < preferred_low - 20)
        & (classic_tolerance < 0.20)
        & (temporal_profile_strength >= 0.50)
    ).fillna(False)
    scored.loc[invalid_year_mask, "is_temporal_outlier"] = True
    return scored


def apply_temporal_outlier_allowance(candidates, min_candidates_after_filter=100):
    scored = candidates.copy()
    if "is_temporal_outlier" not in scored.columns:
        scored["allow_temporal_outlier"] = True
        return scored

    collab_p80 = scored["item_item_collab_score"].quantile(0.80) if "item_item_collab_score" in scored.columns else np.inf
    semantic_reference = "semantic_net_score" if "semantic_net_score" in scored.columns else "semantic_profile_score"
    semantic_p90 = scored[semantic_reference].quantile(0.90) if semantic_reference in scored.columns else np.inf
    rating_p95 = scored["rating_score"].quantile(0.95) if "rating_score" in scored.columns else np.inf

    scored["allow_temporal_outlier"] = (
        ~scored["is_temporal_outlier"].fillna(False)
        | (scored.get("item_item_collab_score", 0) >= collab_p80)
        | (scored.get(semantic_reference, 0) >= semantic_p90)
        | (scored.get("rating_score", 0) >= rating_p95)
    )

    filtered = scored[scored["allow_temporal_outlier"]].copy()
    if len(filtered) >= min_candidates_after_filter:
        return filtered

    warnings.warn(
        "No se aplica el filtro suave de outliers temporales porque dejar?a menos de "
        f"{min_candidates_after_filter} candidatos."
    )
    scored["allow_temporal_outlier"] = True
    return scored

def analyze_user_profile_for_adaptive_weights(
    liked_movies,
    neutral_movies,
    disliked_movies,
    positive_tag_profile,
    negative_tag_profile,
    trakt_ratings_profile,
    movie_id_to_col=None,
    candidates_scored=None,
    discriminative_tag_profile=None,
):
    n_liked = len(liked_movies)
    n_neutral = len(neutral_movies)
    n_disliked = len(disliked_movies)
    n_total_ratings = len(trakt_ratings_profile)
    profile_size_score = _clip01(n_total_ratings / 80)
    liked_size_score = _clip01(n_liked / 30)
    disliked_size_score = _clip01(n_disliked / 15)

    n_positive_semantic_tags = len(positive_tag_profile) if positive_tag_profile is not None else 0
    n_negative_semantic_tags = len(negative_tag_profile) if negative_tag_profile is not None else 0
    semantic_ambiguity, n_shared_semantic_tags = _semantic_ambiguity(positive_tag_profile, negative_tag_profile)

    if positive_tag_profile is not None and "semantic_category" in positive_tag_profile.columns:
        category_strength = _clip01(positive_tag_profile["semantic_category"].dropna().nunique() / 4)
    else:
        category_strength = 0.5
    semantic_profile_strength = _clip01(
        0.45 * min(1.0, n_positive_semantic_tags / 40)
        + 0.35 * liked_size_score
        + 0.20 * category_strength
    )
    semantic_reliability = _clip01(semantic_profile_strength * (1 - semantic_ambiguity))

    n_core_positive_tags = 0
    n_core_negative_tags = 0
    mean_core_tag_ambiguity = semantic_ambiguity
    core_semantic_strength = semantic_profile_strength
    if discriminative_tag_profile is not None and not discriminative_tag_profile.empty:
        core_positive = discriminative_tag_profile[discriminative_tag_profile["is_core_positive_tag"]].copy()
        core_negative = discriminative_tag_profile[discriminative_tag_profile["is_core_negative_tag"]].copy()
        n_core_positive_tags = len(core_positive)
        n_core_negative_tags = len(core_negative)
        mean_core_tag_ambiguity = core_positive["tag_ambiguity"].mean() if not core_positive.empty else 1.0
        if not core_positive.empty and "semantic_category" in core_positive.columns:
            n_core_categories = core_positive["semantic_category"].dropna().nunique()
        else:
            n_core_categories = 0
        core_semantic_strength = _clip01(
            min(1.0, n_core_positive_tags / 30) * 0.45
            + min(1.0, n_core_categories / 4) * 0.25
            + liked_size_score * 0.30
        )
        semantic_reliability = _clip01(core_semantic_strength * (1 - mean_core_tag_ambiguity))

    negative_profile_strength = _clip01(0.50 * disliked_size_score + 0.50 * min(1.0, max(n_negative_semantic_tags, n_core_negative_tags) / 30))
    negative_semantic_reliability = _clip01(negative_profile_strength * (1 - mean_core_tag_ambiguity))

    positive_genres = extract_genre_weights_from_profile(liked_movies, positive=True)
    negative_genres = extract_genre_weights_from_profile(disliked_movies, positive=False)
    n_positive_genres = len(positive_genres)
    n_negative_genres = len(negative_genres)
    if positive_genres.empty or negative_genres.empty:
        genre_ambiguity = 1.0
    else:
        genre_overlap = positive_genres.merge(negative_genres, on="genre", how="inner", suffixes=("_pos", "_neg"))
        genre_ambiguity = _clip01(np.minimum(genre_overlap["genre_weight_norm_pos"], genre_overlap["genre_weight_norm_neg"]).sum()) if not genre_overlap.empty else 0.0
    genre_profile_strength = _clip01(0.50 * min(1.0, n_positive_genres / 6) + 0.50 * liked_size_score)
    genre_reliability = _clip01(genre_profile_strength * (1 - genre_ambiguity))

    if movie_id_to_col is not None:
        rated_ids = set(trakt_ratings_profile["movieId"].dropna().astype(int)) if "movieId" in trakt_ratings_profile.columns else set()
        collab_coverage = len(rated_ids & set(movie_id_to_col.keys())) / len(rated_ids) if rated_ids else 0.0
    else:
        collab_coverage = 0.5
    if candidates_scored is not None and "n_collab_evidence" in candidates_scored.columns and len(candidates_scored):
        collab_candidate_evidence = (candidates_scored["n_collab_evidence"].fillna(0) > 0).mean()
        collab_reliability = _clip01(0.70 * collab_coverage + 0.30 * collab_candidate_evidence)
    else:
        collab_candidate_evidence = np.nan
        collab_reliability = _clip01(collab_coverage)

    profile_weakness = _clip01(1 - profile_size_score)
    return {
        "n_liked": n_liked,
        "n_neutral": n_neutral,
        "n_disliked": n_disliked,
        "n_total_ratings": n_total_ratings,
        "profile_size_score": profile_size_score,
        "liked_size_score": liked_size_score,
        "disliked_size_score": disliked_size_score,
        "n_positive_semantic_tags": n_positive_semantic_tags,
        "n_negative_semantic_tags": n_negative_semantic_tags,
        "n_shared_semantic_tags": n_shared_semantic_tags,
        "semantic_ambiguity": semantic_ambiguity,
        "semantic_profile_strength": semantic_profile_strength,
        "semantic_reliability": semantic_reliability,
        "negative_profile_strength": negative_profile_strength,
        "negative_semantic_reliability": negative_semantic_reliability,
        "n_core_positive_tags": n_core_positive_tags,
        "n_core_negative_tags": n_core_negative_tags,
        "mean_core_tag_ambiguity": _clip01(mean_core_tag_ambiguity),
        "core_semantic_strength": core_semantic_strength,
        "n_positive_genres": n_positive_genres,
        "n_negative_genres": n_negative_genres,
        "genre_ambiguity": genre_ambiguity,
        "genre_profile_strength": genre_profile_strength,
        "genre_reliability": genre_reliability,
        "collab_coverage": _clip01(collab_coverage),
        "collab_candidate_evidence": collab_candidate_evidence,
        "collab_reliability": collab_reliability,
        "profile_weakness": profile_weakness,
    }


def _bounded_normalize(raw_weights, target_total, min_bounds=None, max_bounds=None):
    min_bounds = min_bounds or {}
    max_bounds = max_bounds or {}
    keys = list(raw_weights)
    weights = {key: max(float(raw_weights[key]), 0.0) for key in keys}
    raw_total = sum(weights.values())
    if raw_total <= 0:
        return {key: target_total / len(keys) for key in keys}
    weights = {key: value / raw_total * target_total for key, value in weights.items()}
    fixed = {}
    free = set(keys)
    for _ in range(10):
        changed = False
        for key in list(free):
            lower = min_bounds.get(key, 0.0)
            upper = max_bounds.get(key, float("inf"))
            if weights[key] < lower:
                fixed[key] = lower
                free.remove(key)
                changed = True
            elif weights[key] > upper:
                fixed[key] = upper
                free.remove(key)
                changed = True
        remaining = target_total - sum(fixed.values())
        if remaining <= 0 or not free:
            break
        free_raw_total = sum(raw_weights[key] for key in free)
        if free_raw_total <= 0:
            for key in free:
                weights[key] = remaining / len(free)
        else:
            for key in free:
                weights[key] = raw_weights[key] / free_raw_total * remaining
        if not changed:
            break
    weights.update(fixed)
    return weights


def derive_adaptive_weights(profile_metrics):
    genre_reliability = profile_metrics.get("genre_reliability", 0.0)
    semantic_reliability = profile_metrics.get("semantic_reliability", 0.0)
    collab_reliability = profile_metrics.get("collab_reliability", 0.0)
    negative_semantic_reliability = profile_metrics.get("negative_semantic_reliability", 0.0)
    negative_profile_strength = profile_metrics.get("negative_profile_strength", 0.0)
    profile_weakness = profile_metrics.get("profile_weakness", 1.0)
    temporal_profile_strength = profile_metrics.get("temporal_profile_strength", 0.0)
    temporal_strictness = profile_metrics.get("temporal_strictness", 0.0)

    raw_positive = {
        "genre": 0.06 + 0.12 * genre_reliability,
        "semantic": 0.10 + 0.35 * semantic_reliability,
        "collab": 0.10 + 0.22 * collab_reliability,
        "rating": 0.14 + 0.14 * profile_weakness,
        "popularity": 0.02 + 0.08 * profile_weakness,
        "year_affinity": 0.04 + 0.08 * temporal_profile_strength,
    }
    raw_negative = {
        "negative_genre": 0.10 + 0.15 * genre_reliability * negative_profile_strength,
        "negative_semantic": 0.15 + 0.35 * negative_semantic_reliability,
        "negative_collab": 0.08 + 0.20 * collab_reliability * negative_profile_strength,
        "temporal_mismatch": 0.08 + 0.22 * temporal_strictness,
    }

    positive_min = {"rating": 0.12}
    if collab_reliability >= 0.70:
        positive_min["collab"] = 0.18
    positive_max = {"semantic": 0.30, "popularity": 0.06, "collab": 0.25, "rating": 0.25, "year_affinity": 0.10}
    negative_max = {"negative_semantic": 0.35, "temporal_mismatch": 0.22}

    weights = _bounded_normalize(raw_positive, 0.75, min_bounds=positive_min, max_bounds=positive_max)
    weights.update(_bounded_normalize(raw_negative, 0.60, max_bounds=negative_max))
    return weights


discriminative_tag_profile = build_discriminative_semantic_profile(
    tags_clean=tags_clean,
    liked_movies=liked_movies,
    disliked_movies=disliked_movies,
    positive_tag_profile=positive_tag_profile,
    negative_tag_profile=negative_tag_profile,
)

print("Top 30 tags por discriminative_weight")
display(discriminative_tag_profile.sort_values("discriminative_weight", ascending=False).head(30))
print("Top 30 tags por negative_discriminative_weight")
display(discriminative_tag_profile.sort_values("negative_discriminative_weight", ascending=False).head(30))
print("Media de tag_ambiguity:", discriminative_tag_profile["tag_ambiguity"].mean() if not discriminative_tag_profile.empty else np.nan)
print("Número de core positive tags:", int(discriminative_tag_profile["is_core_positive_tag"].sum()) if not discriminative_tag_profile.empty else 0)
print("Número de core negative tags:", int(discriminative_tag_profile["is_core_negative_tag"].sum()) if not discriminative_tag_profile.empty else 0)
if not discriminative_tag_profile.empty and "semantic_category" in discriminative_tag_profile.columns:
    display(discriminative_tag_profile.loc[discriminative_tag_profile["is_core_positive_tag"], "semantic_category"].value_counts(dropna=False))

if EXPORT_LEGACY_EXPORTS:
    discriminative_tag_profile.to_csv(REPORTS_RESULTADOS / "discriminative_tag_profile.csv", index=False)

semantic_core_scores = compute_semantic_core_scores(
    candidates=candidates_scored,
    tags_clean=tags_clean,
    discriminative_tag_profile=discriminative_tag_profile,
)
candidates_scored = candidates_scored.drop(columns=[col for col in semantic_core_scores.columns if col != "movieId" and col in candidates_scored.columns]).merge(
    semantic_core_scores,
    on="movieId",
    how="left",
)
for col in ["semantic_core_raw_score", "semantic_core_negative_raw_score", "semantic_core_score", "semantic_core_negative_score"]:
    candidates_scored[col] = candidates_scored[col].fillna(0.0)
for col in ["core_semantic_explanation_terms", "core_negative_semantic_terms"]:
    candidates_scored[col] = candidates_scored[col].fillna("")

if "main_genre" not in candidates_scored.columns:
    candidates_scored["main_genre"] = candidates_scored["genres"].apply(main_genre_from_genres)
candidates_scored["semantic_branch"] = candidates_scored.apply(assign_semantic_branch, axis=1)

user_branch_profile = build_user_branch_profile(
    liked_movies=liked_movies,
    tags_clean=tags_clean,
    discriminative_tag_profile=discriminative_tag_profile,
    assign_branch_func=assign_semantic_branch,
)
candidates_scored = add_branch_affinity_to_candidates(
    candidates_scored,
    user_branch_profile,
    branch_col="semantic_branch",
)


temporal_metrics = analyze_temporal_profile(
    user_movies=trakt_ratings_profile,
    liked_movies=liked_movies,
    watched_movies=trakt_watched.merge(movies[["movieId", "year"]], on="movieId", how="left") if "year" not in trakt_watched.columns else trakt_watched,
)
temporal_metrics_df = pd.DataFrame(temporal_metrics.items(), columns=["metric", "value"])
display(temporal_metrics_df)
if EXPORT_LEGACY_EXPORTS:
    temporal_metrics_df.to_csv(REPORTS_RESULTADOS / "temporal_profile_metrics.csv", index=False)

candidates_scored = compute_year_affinity_scores(candidates_scored, temporal_metrics, year_col="year")

profile_metrics = analyze_user_profile_for_adaptive_weights(
    liked_movies=liked_movies,
    neutral_movies=neutral_movies,
    disliked_movies=disliked_movies,
    positive_tag_profile=positive_tag_profile,
    negative_tag_profile=negative_tag_profile,
    trakt_ratings_profile=trakt_ratings,
    movie_id_to_col=movie_id_to_col,
    candidates_scored=candidates_scored,
    discriminative_tag_profile=discriminative_tag_profile,
)
profile_metrics.update(temporal_metrics)
profile_metrics_df = pd.DataFrame(profile_metrics.items(), columns=["metric", "value"])
display(profile_metrics_df.sort_values("metric"))
if EXPORT_LEGACY_EXPORTS:
    profile_metrics_df.to_csv(REPORTS_RESULTADOS / "adaptive_profile_metrics.csv", index=False)

semantic_negative_beta = float(np.clip(0.50 + 0.75 * profile_metrics["negative_profile_strength"], 0.50, 1.25))
candidates_scored = compute_semantic_net_score(candidates_scored, beta=semantic_negative_beta)

positive_anchor_evidence = pd.DataFrame()
core_positive_anchors = pd.DataFrame()
try:
    positive_anchor_evidence = compute_positive_anchor_evidence(
        liked_movies=liked_movies,
        tags_clean=tags_clean,
        discriminative_tag_profile=discriminative_tag_profile,
        user_branch_profile=user_branch_profile if "user_branch_profile" in globals() else None,
        assign_branch_func=assign_semantic_branch if "assign_semantic_branch" in globals() else None,
    )
    if EXPORT_LEGACY_EXPORTS:
        positive_anchor_evidence.to_csv(REPORTS_RESULTADOS / "positive_anchor_evidence.csv", index=False)
    print("Top 25 positive_anchor_evidence por anchor_weight:")
    display(positive_anchor_evidence[[
        "title", "year", "user_rating_5", "semantic_branch", "branch_strength",
        "centrality_score", "anchor_weight"
    ]].head(25))

    core_positive_anchors = select_core_positive_anchors(positive_anchor_evidence, top_n_anchors=30, min_anchors=12)
    if EXPORT_LEGACY_EXPORTS:
        core_positive_anchors.to_csv(REPORTS_RESULTADOS / "core_positive_anchors_selected.csv", index=False)
    print("Distribucion de ramas de core_positive_anchors_selected:")
    display(core_positive_anchors["semantic_branch"].value_counts(dropna=False))
    display(core_positive_anchors[[
        "title", "year", "user_rating_5", "semantic_branch", "branch_strength",
        "centrality_score", "anchor_weight", "selected_reason"
    ]].head(30))
except Exception as exc:
    warnings.warn(f"No se pudo calcular la evidencia de anclas positivas; se usara fallback semantico. Detalle: {exc}")
    positive_anchor_evidence = pd.DataFrame()
    core_positive_anchors = pd.DataFrame()

latent_semantic_diagnostics = {
    "n_movie_documents": 0,
    "tfidf_n_features": 0,
    "latent_n_components": 0,
    "n_liked_movies_latent": 0,
    "n_disliked_movies_latent": 0,
}
try:
    movie_semantic_documents = build_movie_semantic_documents(movies, tags_clean, max_tags_per_movie=30)
    latent_vectors, latent_vectorizer, latent_svd_model, movie_id_to_latent_idx, latent_idx_to_movie_id = build_latent_semantic_space(
        movie_semantic_documents,
        n_components=64,
        max_features=8000,
        min_df=2,
        random_state=RANDOM_STATE,
    )
    latent_similarity_scores = compute_latent_user_similarity_scores(
        candidates=candidates_scored,
        liked_movies=liked_movies,
        disliked_movies=disliked_movies,
        latent_vectors=latent_vectors,
        movie_id_to_latent_idx=movie_id_to_latent_idx,
        top_k_positive=5,
        top_k_negative=5,
    )
    candidates_scored = candidates_scored.drop(columns=[col for col in latent_similarity_scores.columns if col != "movieId" and col in candidates_scored.columns]).merge(
        latent_similarity_scores,
        on="movieId",
        how="left",
    )

    latent_core_preference_scores = compute_latent_core_preference_scores(
        candidates=candidates_scored,
        core_positive_anchors=core_positive_anchors,
        disliked_movies=disliked_movies,
        latent_vectors=latent_vectors,
        movie_id_to_latent_idx=movie_id_to_latent_idx,
        user_branch_profile=user_branch_profile if "user_branch_profile" in globals() else None,
        top_k_positive=5,
        top_k_negative=5,
    )
    candidates_scored = candidates_scored.drop(columns=[col for col in latent_core_preference_scores.columns if col != "movieId" and col in candidates_scored.columns]).merge(
        latent_core_preference_scores,
        on="movieId",
        how="left",
    )
    latent_semantic_diagnostics.update({
        "n_movie_documents": len(movie_semantic_documents),
        "tfidf_n_features": len(getattr(latent_vectorizer, "vocabulary_", {})),
        "latent_n_components": int(latent_vectors.shape[1]) if len(latent_vectors.shape) > 1 else 0,
        "n_liked_movies_latent": int(candidates_scored["n_positive_latent_neighbors"].max()) if "n_positive_latent_neighbors" in candidates_scored.columns else 0,
        "n_disliked_movies_latent": int(candidates_scored["n_negative_latent_neighbors"].max()) if "n_negative_latent_neighbors" in candidates_scored.columns else 0,
        "n_positive_anchor_evidence": int(len(positive_anchor_evidence)),
        "n_core_anchors_selected": int(len(core_positive_anchors)),
        "n_core_anchors_used": int(candidates_scored["n_core_anchors_used"].max()) if "n_core_anchors_used" in candidates_scored.columns else 0,
        "n_negative_anchors_used": int(candidates_scored["n_negative_anchors_used"].max()) if "n_negative_anchors_used" in candidates_scored.columns else 0,
    })
except Exception as exc:
    warnings.warn(f"No se pudo calcular la similitud semantica latente; se usan scores latentes en 0. Detalle: {exc}")
    for col, default in {
        "latent_core_similarity_raw": 0.0,
        "latent_core_similarity_score": 0.0,
        "negative_latent_similarity_raw": 0.0,
        "negative_latent_similarity_score": 0.0,
        "nearest_liked_movies_latent": "",
        "nearest_disliked_movies_latent": "",
        "n_positive_latent_neighbors": 0,
        "n_negative_latent_neighbors": 0,
        "positive_centroid_similarity": 0.0,
        "topk_anchor_similarity": 0.0,
        "branch_prototype_similarity": 0.0,
        "negative_latent_preference_raw": 0.0,
        "negative_latent_preference_score": 0.0,
        "latent_core_preference_raw": 0.0,
        "latent_core_preference_score": 0.0,
        "nearest_core_anchor_movies": "",
        "nearest_negative_movies_latent": "",
        "n_core_anchors_used": 0,
        "n_negative_anchors_used": 0,
    }.items():
        candidates_scored[col] = default

candidates_scored = compute_semantic_relevance_scores(candidates_scored)
print("Peliculas con documentos semanticos:", latent_semantic_diagnostics["n_movie_documents"])
print("Dimensiones TF-IDF:", latent_semantic_diagnostics["tfidf_n_features"])
print("Dimensiones latentes:", latent_semantic_diagnostics["latent_n_components"])
print("Peliculas positivas usadas para similitud latente:", latent_semantic_diagnostics["n_liked_movies_latent"])
print("Peliculas negativas usadas para similitud latente:", latent_semantic_diagnostics["n_disliked_movies_latent"])
if "latent_core_similarity_score" in candidates_scored.columns:
    display(candidates_scored.sort_values("latent_core_similarity_score", ascending=False)[[
        "title", "year", "latent_core_similarity_score", "semantic_net_score", "semantic_relevance_score", "nearest_liked_movies_latent"
    ]].head(20))
if "latent_core_preference_score" in candidates_scored.columns:
    display(candidates_scored.sort_values("latent_core_preference_score", ascending=False)[[
        "title", "year", "genres", "semantic_branch", "branch_strength",
        "latent_core_preference_score", "semantic_net_score", "semantic_relevance_score",
        "nearest_core_anchor_movies"
    ]].head(30))
temporal_outlier_candidate_count = int(candidates_scored["is_temporal_outlier"].sum()) if "is_temporal_outlier" in candidates_scored.columns else 0
candidates_scored = apply_temporal_outlier_allowance(candidates_scored)
temporal_outlier_allowed_count = int((candidates_scored["is_temporal_outlier"] & candidates_scored["allow_temporal_outlier"]).sum()) if {"is_temporal_outlier", "allow_temporal_outlier"}.issubset(candidates_scored.columns) else 0
temporal_outlier_removed_count = max(0, temporal_outlier_candidate_count - temporal_outlier_allowed_count)
candidates_scored["semantic_negative_beta"] = semantic_negative_beta

adaptive_weights = derive_adaptive_weights(profile_metrics)
adaptive_weights_df = pd.DataFrame(adaptive_weights.items(), columns=["component", "weight"])
display(adaptive_weights_df)
if EXPORT_LEGACY_EXPORTS:
    adaptive_weights_df.to_csv(REPORTS_RESULTADOS / "adaptive_weights.csv", index=False)

print(f"semantic_negative_beta: {semantic_negative_beta:.3f}")
print("Pesos adaptativos calculados:")
print(adaptive_weights)

## Análisis y configuración de pesos del score híbrido

In [ ]:
WEIGHT_CONFIGS = {
    "personalizado_semantico": {
        "genre": 0.18,
        "semantic": 0.35,
        "collab": 0.12,
        "rating": 0.15,
        "popularity": 0.03,
        "negative_genre": 0.20,
        "negative_semantic": 0.30,
        "negative_collab": 0.15,
    },
    "equilibrado_semantico": {
        "genre": 0.20,
        "semantic": 0.30,
        "collab": 0.15,
        "rating": 0.17,
        "popularity": 0.04,
        "negative_genre": 0.20,
        "negative_semantic": 0.25,
        "negative_collab": 0.15,
    },
    "descubrimiento_semantico": {
        "genre": 0.18,
        "semantic": 0.40,
        "collab": 0.08,
        "rating": 0.14,
        "popularity": 0.00,
        "negative_genre": 0.20,
        "negative_semantic": 0.35,
        "negative_collab": 0.20,
    },
}

ACTIVE_WEIGHT_CONFIG = "adaptive"


def compute_hybrid_score(df, weights):
    scored = df.copy()
    scored["contrib_genre"] = weights["genre"] * scored["genre_profile_score"]
    if "year_affinity_score" not in scored.columns:
        scored["year_affinity_score"] = 1.0
    if "temporal_mismatch_penalty" not in scored.columns:
        scored["temporal_mismatch_penalty"] = 0.0
    if "semantic_relevance_adjusted_score" in scored.columns:
        semantic_signal = scored["semantic_relevance_adjusted_score"]
    elif "semantic_relevance_score" in scored.columns:
        semantic_signal = scored["semantic_relevance_score"]
    elif "semantic_net_score" in scored.columns:
        semantic_signal = scored["semantic_net_score"]
    elif "semantic_core_score" in scored.columns:
        semantic_signal = scored["semantic_core_score"]
    else:
        semantic_signal = scored["semantic_profile_score"]
    if "negative_semantic_relevance_score" in scored.columns:
        negative_semantic_signal = scored["negative_semantic_relevance_score"]
    elif "semantic_core_negative_score" in scored.columns:
        negative_semantic_signal = scored["semantic_core_negative_score"]
    else:
        negative_semantic_signal = scored["negative_semantic_score"]
    scored["contrib_semantic"] = weights["semantic"] * semantic_signal
    scored["contrib_collab"] = weights["collab"] * scored["item_item_collab_score"]
    scored["contrib_rating"] = weights["rating"] * scored["rating_score"]
    scored["contrib_popularity"] = weights["popularity"] * scored["popularity_score"]
    scored["contrib_year_affinity"] = weights.get("year_affinity", 0.0) * scored["year_affinity_score"]
    scored["penalty_negative_genre"] = weights["negative_genre"] * scored["negative_genre_score"]
    scored["penalty_negative_semantic"] = weights["negative_semantic"] * negative_semantic_signal
    scored["penalty_negative_collab"] = weights["negative_collab"] * scored["item_item_negative_collab_score"]
    scored["penalty_temporal_mismatch"] = weights.get("temporal_mismatch", 0.0) * scored["temporal_mismatch_penalty"]

    scored["hybrid_score"] = (
        scored["contrib_genre"]
        + scored["contrib_semantic"]
        + scored["contrib_collab"]
        + scored["contrib_rating"]
        + scored["contrib_popularity"]
        + scored["contrib_year_affinity"]
        - scored["penalty_negative_genre"]
        - scored["penalty_negative_semantic"]
        - scored["penalty_negative_collab"]
        - scored["penalty_temporal_mismatch"]
    )

    contribution_cols = {
        "semantic": "contrib_semantic",
        "genre": "contrib_genre",
        "collab": "contrib_collab",
        "rating": "contrib_rating",
        "popularity": "contrib_popularity",
        "year_affinity": "contrib_year_affinity",
    }
    scored["dominant_signal"] = scored[list(contribution_cols.values())].idxmax(axis=1)
    scored["dominant_signal"] = scored["dominant_signal"].replace({value: key for key, value in contribution_cols.items()})
    return scored


candidates_scored_base = candidates_scored.copy()
print(f"Configuración activa de pesos: {ACTIVE_WEIGHT_CONFIG}")

## 14. Hybrid score inicial

In [ ]:
weights = adaptive_weights if ACTIVE_WEIGHT_CONFIG == "adaptive" else WEIGHT_CONFIGS[ACTIVE_WEIGHT_CONFIG]

pre_latent_hybrid_input = candidates_scored.drop(columns=["semantic_relevance_score", "semantic_relevance_adjusted_score", "negative_semantic_relevance_score"], errors="ignore")
pre_latent_hybrid_scored = compute_hybrid_score(pre_latent_hybrid_input, weights).sort_values("hybrid_score", ascending=False).reset_index(drop=True)
pre_latent_top20_titles = set(pre_latent_hybrid_scored.head(20)["title"].astype(str))

candidates_scored = compute_hybrid_score(candidates_scored, weights)
post_latent_hybrid_top20 = candidates_scored.sort_values("hybrid_score", ascending=False).head(20).copy()
post_latent_top20_titles = set(post_latent_hybrid_top20["title"].astype(str))
latent_ranking_overlap_top20 = len(pre_latent_top20_titles & post_latent_top20_titles) / 20 if pre_latent_top20_titles else np.nan
n_titles_changed_after_latent = 20 - len(pre_latent_top20_titles & post_latent_top20_titles) if pre_latent_top20_titles else np.nan
overlap_top20_before_after_semantic_adjustment = latent_ranking_overlap_top20
semantic_adjustment_changed_count = n_titles_changed_after_latent
print(f"Overlap top 20 antes/despues de semantic_relevance_adjusted_score: {overlap_top20_before_after_semantic_adjustment:.3f}")
print(f"Titulos cambiados tras aplicar semantic_relevance_adjusted_score: {semantic_adjustment_changed_count}")
if pd.notna(latent_ranking_overlap_top20) and latent_ranking_overlap_top20 > 0.85:
    warnings.warn("El nuevo semantic_relevance_adjusted_score apenas modifica el ranking base.")
if {"latent_core_preference_score", "semantic_relevance_score", "semantic_net_score"}.issubset(candidates_scored.columns):
    relevance_delta = (candidates_scored["semantic_relevance_score"].fillna(0.0) - candidates_scored["semantic_net_score"].fillna(0.0)).abs().mean()
    if relevance_delta < 1e-6:
        warnings.warn("semantic_relevance_score no cambia respecto a semantic_net_score pese a existir latent_core_preference_score.")

semantic_diagnostic_cols = [
    col for col in [
        "semantic_net_score", "latent_core_similarity_score", "latent_core_preference_score",
        "semantic_relevance_score", "semantic_relevance_adjusted_score",
        "negative_semantic_relevance_score", "hybrid_score"
    ] if col in candidates_scored.columns
]
print("Columnas semanticas disponibles:", semantic_diagnostic_cols)
if len(semantic_diagnostic_cols) >= 2:
    display(candidates_scored[semantic_diagnostic_cols].corr(numeric_only=True))

def _top20_titles_by_score(df, score_col):
    if score_col not in df.columns:
        return set()
    return set(df.sort_values(score_col, ascending=False).head(20)["title"].astype(str))

semantic_net_top20_titles = _top20_titles_by_score(candidates_scored, "semantic_net_score")
latent_preference_top20_titles = _top20_titles_by_score(candidates_scored, "latent_core_preference_score")
semantic_relevance_top20_titles = _top20_titles_by_score(candidates_scored, "semantic_relevance_score")
hybrid_pre_rerank_top20_titles = _top20_titles_by_score(candidates_scored, "hybrid_score")
overlap_top20_semantic_net_vs_latent_preference = (
    len(semantic_net_top20_titles & latent_preference_top20_titles) / 20
    if semantic_net_top20_titles and latent_preference_top20_titles else np.nan
)
overlap_top20_latent_preference_vs_hybrid = (
    len(latent_preference_top20_titles & hybrid_pre_rerank_top20_titles) / 20
    if latent_preference_top20_titles and hybrid_pre_rerank_top20_titles else np.nan
)
print("Overlap semantic_net vs latent_core_preference:", overlap_top20_semantic_net_vs_latent_preference)
print("Overlap latent_core_preference vs hybrid pre-rerank:", overlap_top20_latent_preference_vs_hybrid)

for score_col in ["semantic_net_score", "latent_core_preference_score", "semantic_relevance_score", "hybrid_score"]:
    if score_col in candidates_scored.columns:
        print(f"Top 20 por {score_col} antes del re-ranking:")
        display(candidates_scored.sort_values(score_col, ascending=False)[["title", "year", "genres", score_col]].head(20))

if "latent_core_preference_score" in candidates_scored.columns and pd.notna(overlap_top20_latent_preference_vs_hybrid) and overlap_top20_latent_preference_vs_hybrid > 0.85:
    warnings.warn("latent_core_preference_score existe pero apenas cambia el ranking base.")

print("Top 30 candidatos por latent_core_preference_score antes del re-ranking:")
if "latent_core_preference_score" in candidates_scored.columns:
    display(candidates_scored.sort_values("latent_core_preference_score", ascending=False)[[
        "title", "year", "genres", "semantic_branch", "branch_strength",
        "latent_core_preference_score", "semantic_net_score", "semantic_relevance_score",
        "nearest_core_anchor_movies"
    ]].head(30))
print("Top 20 por hybrid_score tras semantic_relevance_score y antes del re-ranking:")
display(post_latent_hybrid_top20[["title", "year", "genres", "hybrid_score", "semantic_relevance_score", "dominant_signal"]])

candidates_scored = candidates_scored.sort_values("hybrid_score", ascending=False).reset_index(drop=True)
candidates_scored["hybrid_rank_raw"] = np.arange(1, len(candidates_scored) + 1)

AUDIT_COLUMNS = [
    "title",
    "genres",
    "genre_profile_score",
    "semantic_profile_score",
    "semantic_core_score",
    "semantic_core_negative_score",
    "semantic_net_score",
    "negative_semantic_relevance_score",
    "semantic_relevance_score",
    "negative_semantic_relevance_raw",
    "semantic_relevance_raw",
    "branch_support_multiplier",
    "negative_latent_similarity_score",
    "latent_core_similarity_score",
    "latent_core_preference_score",
    "nearest_core_anchor_movies",
    "semantic_branch",
    "branch_strength",
    "branch_share",
    "n_negative_anchors_used",
    "n_core_anchors_used",
    "nearest_negative_movies_latent",
    "latent_core_preference_raw",
    "negative_latent_preference_score",
    "branch_prototype_similarity",
    "topk_anchor_similarity",
    "positive_centroid_similarity",
    "negative_genre_score",
    "negative_semantic_score",
    "semantic_negative_beta",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "rating_score",
    "popularity_score",
    "allow_temporal_outlier",
    "is_temporal_outlier",
    "temporal_distance_from_profile",
    "temporal_mismatch_penalty",
    "year_affinity_score",
    "contrib_genre",
    "contrib_semantic",
    "contrib_collab",
    "contrib_rating",
    "contrib_popularity",
    "contrib_year_affinity",
    "penalty_negative_genre",
    "penalty_negative_semantic",
    "penalty_negative_collab",
    "penalty_temporal_mismatch",
    "dominant_signal",
    "semantic_explanation_terms",
    "core_semantic_explanation_terms",
    "core_negative_semantic_terms",
    "hybrid_score",
]

display(candidates_scored[AUDIT_COLUMNS].head(20))
display(candidates_scored.head(50)["dominant_signal"].value_counts())

## 15. Diversidad

In [ ]:
def analyze_user_diversity_profile(
    liked_movies,
    tags_clean=None,
    discriminative_tag_profile=None,
    temporal_metrics=None,
    user_branch_profile=None,
):
    genre_rows = []
    if liked_movies is not None and not liked_movies.empty:
        for _, row in liked_movies.dropna(subset=["user_rating_5"]).iterrows():
            movie_weight = max(float(row.get("user_rating_5", 0)) - 3.0, 0.0)
            if movie_weight <= 0:
                continue
            for genre in split_genres(row.get("genres", "")):
                genre_rows.append({"genre": genre, "weight": movie_weight})

    if genre_rows:
        genre_profile = pd.DataFrame(genre_rows).groupby("genre", as_index=False)["weight"].sum()
        genre_total = genre_profile["weight"].sum()
        genre_profile["share"] = genre_profile["weight"] / genre_total if genre_total > 0 else 0.0
        genre_profile = genre_profile.sort_values("share", ascending=False)
        n_positive_genres = int(len(genre_profile))
        top_genre = str(genre_profile.iloc[0]["genre"])
        top_genre_share = float(genre_profile.iloc[0]["share"])
        genre_entropy, genre_diversity_score = _normalized_entropy(genre_profile["share"])
    else:
        n_positive_genres = 0
        top_genre = ""
        top_genre_share = 1.0
        genre_entropy = 0.0
        genre_diversity_score = 0.0

    semantic_category_diversity_score = 0.5
    semantic_category_entropy = 0.0
    n_positive_semantic_categories = 0
    top_semantic_category = ""
    top_semantic_category_share = 0.5
    if discriminative_tag_profile is not None and not discriminative_tag_profile.empty and "semantic_category" in discriminative_tag_profile.columns:
        core_tags = discriminative_tag_profile[
            (discriminative_tag_profile["discriminative_weight"].fillna(0) > 0)
            & discriminative_tag_profile["semantic_category"].notna()
        ].copy()
        if not core_tags.empty:
            category_profile = core_tags.groupby("semantic_category", as_index=False)["discriminative_weight"].sum()
            category_total = category_profile["discriminative_weight"].sum()
            category_profile["share"] = category_profile["discriminative_weight"] / category_total if category_total > 0 else 0.0
            category_profile = category_profile.sort_values("share", ascending=False)
            n_positive_semantic_categories = int(len(category_profile))
            top_semantic_category = str(category_profile.iloc[0]["semantic_category"])
            top_semantic_category_share = float(category_profile.iloc[0]["share"])
            semantic_category_entropy, semantic_category_diversity_score = _normalized_entropy(category_profile["share"])

    branch_positive = user_branch_profile[user_branch_profile["branch_share"].fillna(0) > 0].copy() if user_branch_profile is not None and not user_branch_profile.empty else pd.DataFrame()
    if branch_positive.empty:
        n_user_branches_supported = 0
        top_branch = ""
        top_branch_share = 1.0
        branch_entropy = 0.0
        branch_diversity_score = 0.0
    else:
        branch_positive = branch_positive.sort_values("branch_share", ascending=False)
        n_user_branches_supported = int(len(branch_positive))
        top_branch = str(branch_positive.iloc[0]["semantic_branch"])
        top_branch_share = float(branch_positive.iloc[0]["branch_share"])
        branch_entropy, branch_diversity_score = _normalized_entropy(branch_positive["branch_share"])
    branch_profile_concentration = top_branch_share

    temporal_metrics = temporal_metrics or {}
    q10_year = temporal_metrics.get("q10_year", np.nan)
    q90_year = temporal_metrics.get("q90_year", np.nan)
    classic_tolerance = float(temporal_metrics.get("classic_tolerance", 0.0) or 0.0)
    if pd.isna(q10_year) or pd.isna(q90_year):
        temporal_span = 0.0
        temporal_diversity_score = 0.0
    else:
        temporal_span = max(0.0, float(q90_year) - float(q10_year))
        if temporal_span < 10:
            temporal_diversity_score = 0.0
        elif temporal_span <= 20:
            temporal_diversity_score = 0.3
        elif temporal_span <= 35:
            temporal_diversity_score = 0.6
        else:
            temporal_diversity_score = 1.0
        temporal_diversity_score = _clip01(temporal_diversity_score + 0.25 * classic_tolerance)

    profile_concentration_score = _clip01(
        0.40 * top_branch_share
        + 0.30 * top_genre_share
        + 0.20 * top_semantic_category_share
        + 0.10 * (1 - temporal_diversity_score)
    )
    overall_user_diversity_score = _clip01(
        0.30 * genre_diversity_score
        + 0.35 * branch_diversity_score
        + 0.20 * semantic_category_diversity_score
        + 0.15 * temporal_diversity_score
    )

    if overall_user_diversity_score < 0.35 or profile_concentration_score > 0.70:
        diversity_level = "concentrated"
    elif overall_user_diversity_score > 0.65 and profile_concentration_score <= 0.70:
        diversity_level = "diverse"
    else:
        diversity_level = "medium"

    return {
        "n_positive_genres": n_positive_genres,
        "top_genre": top_genre,
        "top_genre_share": top_genre_share,
        "genre_entropy": genre_entropy,
        "genre_diversity_score": genre_diversity_score,
        "n_positive_semantic_categories": n_positive_semantic_categories,
        "top_semantic_category": top_semantic_category,
        "top_semantic_category_share": top_semantic_category_share,
        "semantic_category_entropy": semantic_category_entropy,
        "semantic_category_diversity_score": semantic_category_diversity_score,
        "n_user_branches_supported": n_user_branches_supported,
        "top_branch": top_branch,
        "top_branch_share": top_branch_share,
        "branch_entropy": branch_entropy,
        "branch_diversity_score": branch_diversity_score,
        "branch_profile_concentration": branch_profile_concentration,
        "temporal_span": temporal_span,
        "temporal_diversity_score": temporal_diversity_score,
        "profile_concentration_score": profile_concentration_score,
        "overall_user_diversity_score": overall_user_diversity_score,
        "diversity_level": diversity_level,
    }


def derive_adaptive_rerank_config(diversity_metrics):
    level = diversity_metrics.get("diversity_level", "medium")
    configs = {
        "concentrated": {
            "max_per_genre": 8,
            "max_per_branch": 8,
            "max_per_signal": 18,
            "diversity_penalty_strength": 0.03,
            "branch_penalty_strength": 0.03,
            "genre_penalty_strength": 0.03,
            "signal_penalty_strength": 0.00,
            "exploration_slots": 1,
            "score_floor_ratio": 0.90,
        },
        "medium": {
            "max_per_genre": 6,
            "max_per_branch": 6,
            "max_per_signal": 17,
            "diversity_penalty_strength": 0.06,
            "branch_penalty_strength": 0.06,
            "genre_penalty_strength": 0.05,
            "signal_penalty_strength": 0.01,
            "exploration_slots": 2,
            "score_floor_ratio": 0.87,
        },
        "diverse": {
            "max_per_genre": 4,
            "max_per_branch": 4,
            "max_per_signal": 16,
            "diversity_penalty_strength": 0.10,
            "branch_penalty_strength": 0.10,
            "genre_penalty_strength": 0.08,
            "signal_penalty_strength": 0.02,
            "exploration_slots": 3,
            "score_floor_ratio": 0.84,
        },
    }
    config = dict(configs.get(level, configs["medium"]))
    config["diversity_level"] = level
    return config


def select_diverse_recommendations(df, top_n=20, max_per_main_genre=3, max_drama_total=5):
    work = df.sort_values("hybrid_score", ascending=False).copy()
    if "main_genre" not in work.columns:
        work["main_genre"] = work["genres"].apply(main_genre_from_genres)
    return work.head(top_n).reset_index(drop=True)


def _branch_slot_limits(user_branch_profile, top_n, diversity_level):
    limits = {}
    for _, row in user_branch_profile.iterrows():
        branch = row["semantic_branch"]
        share = float(row.get("branch_share", 0.0) or 0.0)
        strength = row.get("branch_strength", "unseen")
        expected_slots = top_n * share
        if strength == "strong":
            limit = max(4, int(np.ceil(expected_slots + 2)))
        elif strength == "medium":
            limit = min(3, max(2, int(np.ceil(expected_slots))))
        elif strength == "weak":
            limit = min(2, max(1, int(np.ceil(expected_slots))))
        else:
            limit = 1
        limits[branch] = int(min(top_n, limit))
    return limits


def branch_aware_adaptive_rerank(
    candidates,
    adaptive_rerank_config,
    user_branch_profile,
    top_n=20,
    score_col="hybrid_score",
    genre_col="main_genre",
    branch_col="semantic_branch",
    signal_col="dominant_signal",
    candidate_pool_size=300,
):
    enriched = candidates.copy()
    if "branch_affinity_score" not in enriched.columns or "branch_strength" not in enriched.columns or "branch_share" not in enriched.columns:
        enriched = add_branch_affinity_to_candidates(enriched, user_branch_profile, branch_col=branch_col)
    work = enriched.sort_values(score_col, ascending=False).head(candidate_pool_size).copy()
    if genre_col not in work.columns:
        work[genre_col] = work["genres"].apply(main_genre_from_genres)
    if signal_col not in work.columns:
        work[signal_col] = "unknown"

    work["pre_rerank_rank"] = np.arange(1, len(work) + 1)
    pre_top = work.head(top_n).copy()
    pre_min = float(pre_top[score_col].min()) if not pre_top.empty else float(work[score_col].max())
    pre_avg = float(pre_top[score_col].mean()) if not pre_top.empty else float(work[score_col].mean())
    pool_q75 = float(work[score_col].quantile(0.75)) if len(work) else pre_min
    score_floor_ratio = float(adaptive_rerank_config.get("score_floor_ratio", 0.87))
    score_floor = max(pre_min * score_floor_ratio, pool_q75)

    collab_p80 = work["item_item_collab_score"].quantile(0.80) if "item_item_collab_score" in work.columns else np.inf
    rating_p90 = work["rating_score"].quantile(0.90) if "rating_score" in work.columns else np.inf
    semantic_p90 = work["semantic_relevance_adjusted_score"].quantile(0.90) if "semantic_relevance_adjusted_score" in work.columns else (work["semantic_net_score"].quantile(0.90) if "semantic_net_score" in work.columns else np.inf)

    diversity_level = adaptive_rerank_config.get("diversity_level", "medium")
    branch_limits = _branch_slot_limits(user_branch_profile, top_n, diversity_level)
    max_per_genre = int(adaptive_rerank_config.get("max_per_genre", 6))
    max_per_signal = int(adaptive_rerank_config.get("max_per_signal", 17))
    exploration_budget = int(adaptive_rerank_config.get("exploration_slots", 2))
    genre_penalty_strength = float(adaptive_rerank_config.get("genre_penalty_strength", 0.05))
    branch_penalty_strength = float(adaptive_rerank_config.get("branch_penalty_strength", 0.06))
    signal_penalty_strength = 0.0

    selected_indices = []
    selected_set = set()
    selection_scores = {}
    reasons = {}
    genre_counts = {}
    branch_counts = {}
    signal_counts = {}
    exploration_count = 0

    def is_unseen(row):
        return row.get("branch_strength", "unseen") == "unseen"

    def is_weak(row):
        return row.get("branch_strength", "unseen") == "weak"

    def is_exploration(row):
        return is_unseen(row) or float(row.get("branch_share", 0.0) or 0.0) < 0.03

    def has_strong_unseen_signal(row):
        return (
            float(row.get("item_item_collab_score", 0.0) or 0.0) >= collab_p80
            or float(row.get("rating_score", 0.0) or 0.0) >= rating_p90
            or float(row.get("semantic_net_score", 0.0) or 0.0) >= semantic_p90
        )

    def unseen_is_allowed(row):
        if not is_unseen(row):
            return True
        was_original_top = bool(row["pre_rerank_rank"] <= top_n)
        if branch_counts.get(row.get(branch_col, "general"), 0) >= 1:
            return False
        if not (float(row[score_col]) >= pre_min or was_original_top):
            return False
        if not was_original_top and not has_strong_unseen_signal(row):
            return False
        return True

    def slot_limit_for(row, relaxed=False):
        strength = row.get("branch_strength", "unseen")
        base_limit = branch_limits.get(row.get(branch_col, "general"), 1)
        if strength == "unseen":
            return 1
        if strength == "medium":
            return min(3, base_limit)
        if strength == "weak":
            return min(2, base_limit)
        return base_limit + (1 if relaxed else 0)

    def selection_score_for(row, relaxed=False):
        branch_limit = slot_limit_for(row, relaxed=relaxed)
        branch_soft_start = max(1, int(np.floor(branch_limit / 2)))
        genre_count = genre_counts.get(row.get(genre_col, "Unknown"), 0)
        branch_count = branch_counts.get(row.get(branch_col, "general"), 0)
        signal_count = signal_counts.get(row.get(signal_col, "unknown"), 0)
        strength = row.get("branch_strength", "unseen")
        unsupported_penalty = 0.0 if strength in ["strong", "medium"] else (0.03 if strength == "weak" else 0.08)
        if relaxed and strength == "weak":
            unsupported_penalty *= 0.5
        return (
            float(row[score_col])
            + 0.06 * float(row.get("branch_affinity_score", 0.15) or 0.15)
            - genre_penalty_strength * max(0, genre_count - 1)
            - branch_penalty_strength * max(0, branch_count - branch_soft_start)
            - unsupported_penalty
        )

    def add_candidate(idx, row, reason, adjusted_score):
        nonlocal exploration_count
        selected_indices.append(idx)
        selected_set.add(idx)
        selection_scores[idx] = adjusted_score
        if reason.startswith("fallback_fill"):
            reasons[idx] = reason
        elif is_unseen(row):
            reasons[idx] = "selected_exploration_branch"
        elif is_weak(row):
            reasons[idx] = "selected_relaxed_weak_profile_branch" if "relaxed" in reason else "selected_weak_profile_branch"
        elif row["pre_rerank_rank"] <= top_n:
            reasons[idx] = "kept_from_original_top20"
        else:
            reasons[idx] = reason
        genre = row.get(genre_col, "Unknown")
        branch = row.get(branch_col, "general")
        signal = row.get(signal_col, "unknown")
        genre_counts[genre] = genre_counts.get(genre, 0) + 1
        branch_counts[branch] = branch_counts.get(branch, 0) + 1
        signal_counts[signal] = signal_counts.get(signal, 0) + 1
        if is_exploration(row):
            exploration_count += 1

    def reason_for(row, relaxed=False):
        branch = row.get(branch_col, "general")
        if is_unseen(row):
            return "selected_exploration_branch"
        if is_weak(row):
            return "selected_relaxed_weak_profile_branch" if relaxed else "selected_weak_profile_branch"
        if row.get("branch_strength", "unseen") == "medium" and branch_counts.get(branch, 0) >= slot_limit_for(row, relaxed=False):
            return "kept_high_score_medium_branch"
        return "selected_relaxed_profile_branch" if relaxed else "selected_profile_branch"

    def candidate_allowed(row, relaxed=False):
        branch = row.get(branch_col, "general")
        genre = row.get(genre_col, "Unknown")
        signal = row.get(signal_col, "unknown")
        if is_unseen(row):
            if not unseen_is_allowed(row):
                return False
        current_branch_count = branch_counts.get(branch, 0)
        base_branch_limit = slot_limit_for(row, relaxed=False)
        if current_branch_count >= slot_limit_for(row, relaxed=relaxed):
            if row.get("branch_strength", "unseen") == "medium":
                high_score_medium = (
                    current_branch_count < 4
                    and row["pre_rerank_rank"] <= top_n
                    and float(row[score_col]) >= score_floor
                    and branch != "animation_family"
                )
                if not high_score_medium:
                    return False
            else:
                return False
        if branch == "animation_family" and row.get("branch_strength", "unseen") == "medium" and current_branch_count >= 3:
            return False
        if is_weak(row) and branch_counts.get(branch, 0) >= 2:
            if not relaxed:
                return False
            if not (row["pre_rerank_rank"] <= top_n or float(row[score_col]) >= pre_min):
                return False
        if genre_counts.get(genre, 0) >= max_per_genre + (1 if relaxed else 0):
            return False
        if is_exploration(row) and exploration_count >= exploration_budget + (0 if is_unseen(row) else (1 if relaxed else 0)):
            return False
        return True

    def run_pass(current_score_floor, relaxed=False):
        while len(selected_indices) < top_n:
            best_idx = None
            best_score = -np.inf
            best_row = None
            for idx, row in work.iterrows():
                if idx in selected_set or float(row[score_col]) < current_score_floor:
                    continue
                if not candidate_allowed(row, relaxed=relaxed):
                    continue
                adjusted_score = selection_score_for(row, relaxed=relaxed)
                if adjusted_score > best_score:
                    best_idx = idx
                    best_score = adjusted_score
                    best_row = row
            if best_idx is None:
                break
            add_candidate(best_idx, best_row, reason_for(best_row, relaxed=relaxed), best_score)

    run_pass(score_floor, relaxed=False)
    if len(selected_indices) < top_n:
        run_pass(pre_min * 0.80, relaxed=True)
    if len(selected_indices) < top_n:
        for idx, row in work.iterrows():
            if idx in selected_set:
                continue
            branch = row.get(branch_col, "general")
            if is_unseen(row) and branch_counts.get(branch, 0) >= 1:
                fallback_reason = "fallback_fill_unseen_branch"
            elif branch == "animation_family" and row.get("branch_strength", "unseen") == "medium" and branch_counts.get(branch, 0) >= 3:
                fallback_reason = "fallback_fill_animation_family_over_cap"
            else:
                fallback_reason = "fallback_fill"
            add_candidate(idx, row, fallback_reason, float(row[score_col]))
            if len(selected_indices) >= top_n:
                break

    selected = work.loc[selected_indices].copy()
    selected["selection_source_index"] = selected.index
    selected_index_series = pd.Series(selected.index, index=selected.index)
    selected["final_rank"] = np.arange(1, len(selected) + 1)
    selected["was_in_pre_rerank_top20"] = selected["pre_rerank_rank"] <= top_n
    selected["entered_by_rerank"] = ~selected["was_in_pre_rerank_top20"]
    selected["selected_by_rerank"] = True
    selected["rerank_reason"] = selected_index_series.map(reasons).fillna("fallback_fill")
    selected["rerank_selection_score"] = selected_index_series.map(selection_scores).fillna(selected[score_col])
    selected["branch_slot_limit"] = selected.apply(lambda row: branch_limits.get(row.get(branch_col, "general"), 1), axis=1)
    selected["rerank_score_floor"] = score_floor
    selected["rerank_score_floor_ratio"] = score_floor_ratio
    rerank_summary_payload = {
        "score_floor": score_floor,
        "score_floor_ratio": score_floor_ratio,
        "pre_rerank_top_n_min_score": pre_min,
        "pre_rerank_top_n_avg_score": pre_avg,
        "candidate_pool_q75_score": pool_q75,
        "exploration_budget": exploration_budget,
        "collab_p80": collab_p80,
        "rating_p90": rating_p90,
        "semantic_p90": semantic_p90,
    }
    return selected.reset_index(drop=True)


def compare_weight_configs(base_df, weight_configs, top_n=20):
    rows = []
    for config_name, config_weights in weight_configs.items():
        scored = compute_hybrid_score(base_df, config_weights)
        scored = scored.sort_values("hybrid_score", ascending=False).reset_index(drop=True)
        if "main_genre" not in scored.columns:
            scored["main_genre"] = scored["genres"].apply(main_genre_from_genres)
        selected = scored.head(top_n).copy()
        rows.append(
            {
                "config": config_name,
                "avg_genre_profile_score": selected["genre_profile_score"].mean(),
                "avg_semantic_profile_score": selected["semantic_profile_score"].mean(),
                "avg_negative_genre_score": selected["negative_genre_score"].mean(),
                "avg_negative_semantic_score": selected["negative_semantic_score"].mean(),
                "avg_item_item_collab_score": selected["item_item_collab_score"].mean(),
                "avg_item_item_negative_collab_score": selected["item_item_negative_collab_score"].mean(),
                "avg_rating_mean": selected["rating_mean"].mean(),
                "avg_rating_count": selected["rating_count"].mean(),
                "avg_popularity_score": selected["popularity_score"].mean(),
                "avg_year_affinity_score": selected["year_affinity_score"].mean() if "year_affinity_score" in selected.columns else np.nan,
                "avg_temporal_mismatch_penalty": selected["temporal_mismatch_penalty"].mean() if "temporal_mismatch_penalty" in selected.columns else np.nan,
                "n_temporal_outliers": selected["is_temporal_outlier"].sum() if "is_temporal_outlier" in selected.columns else 0,
                "n_distinct_main_genres": selected["main_genre"].nunique() if "main_genre" in selected.columns else selected["main_genre"].nunique(),
                "dominant_signal_counts": selected["dominant_signal"].value_counts().to_dict(),
                "top_titles": "; ".join(selected["title"].head(5).astype(str)),
            }
        )
    return pd.DataFrame(rows)


def robust_normalize_score(series, lower_q=0.01, upper_q=0.99, default=0.0):
    values = pd.to_numeric(pd.Series(series), errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(default).astype(float)
    normalized = pd.Series(0.0, index=values.index, dtype=float)
    valid = values.dropna()
    if valid.empty or valid.nunique() <= 1:
        return normalized
    lower = valid.quantile(lower_q)
    upper = valid.quantile(upper_q)
    if not np.isfinite(lower) or not np.isfinite(upper) or abs(upper - lower) < 1e-9:
        lower = valid.min()
        upper = valid.max()
    if abs(upper - lower) < 1e-9:
        return normalized
    normalized = ((values - lower) / (upper - lower)).clip(0.0, 1.0)
    return normalized.fillna(0.0)


def make_display_score(series, floor=0.50, ceiling=0.985):
    values = pd.to_numeric(pd.Series(series), errors="coerce").replace([np.inf, -np.inf], np.nan)
    valid = values.dropna().astype(float)
    if valid.empty or valid.nunique() <= 1:
        return pd.Series((floor + ceiling) / 2.0, index=values.index, dtype=float)
    fill_value = float(valid.median())
    values = values.fillna(fill_value).astype(float)
    lower = valid.quantile(0.80)
    upper = valid.quantile(0.995)
    if not np.isfinite(lower) or not np.isfinite(upper) or abs(upper - lower) < 1e-9:
        lower = valid.quantile(0.50)
        upper = valid.max()
    if abs(upper - lower) < 1e-9:
        pct = values.rank(method="average", pct=True)
        return (floor + (ceiling - floor) * pct).clip(upper=ceiling - 1e-6)
    scaled = ((values - lower) / (upper - lower)).clip(0.0, 1.0)
    pct = values.rank(method="average", pct=True)
    score = floor + (ceiling - floor) * (0.78 * scaled + 0.22 * pct)
    return score.clip(lower=floor, upper=ceiling - 1e-6).fillna((floor + ceiling) / 2.0)


def _series_01(df, column, default=0.0, normalize_if_needed=False):
    if column not in df.columns:
        return pd.Series(default, index=df.index, dtype=float)
    values = pd.to_numeric(df[column], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(default).astype(float)
    if normalize_if_needed or values.min() < 0 or values.max() > 1:
        return robust_normalize_score(values, default=default)
    return values.clip(0.0, 1.0)


def _weighted_average_components(components, index):
    available = [(weight, series) for weight, series in components if series is not None]
    if not available:
        return pd.Series(0.0, index=index, dtype=float)
    total = sum(weight for weight, _ in available)
    if total <= 0:
        return pd.Series(0.0, index=index, dtype=float)
    result = pd.Series(0.0, index=index, dtype=float)
    for weight, series in available:
        result = result + (weight / total) * series.reindex(index).fillna(0.0)
    return result.clip(0.0, 1.0)


def _strength_to_score(value):
    return {
        "strong": 1.0,
        "medium": 0.70,
        "weak": 0.40,
        "unseen": 0.15,
    }.get(str(value).strip().lower(), 0.30)


def parse_term_list(value):
    if value is None:
        return []
    if isinstance(value, (list, tuple, set)):
        raw_items = list(value)
    else:
        text = str(value).strip()
        if not text or text.lower() in {"nan", "none"}:
            return []
        raw_items = None
        if text.startswith("[") and text.endswith("]"):
            try:
                parsed = ast.literal_eval(text)
                if isinstance(parsed, (list, tuple, set)):
                    raw_items = list(parsed)
            except (SyntaxError, ValueError):
                raw_items = None
        if raw_items is None:
            raw_items = re.split(r"[|,;]", text)
    cleaned = []
    for item in raw_items:
        term = str(item).strip().strip("'\"").lower()
        term = re.sub(r"\s+", " ", term)
        if term and term not in {"nan", "none"} and term not in cleaned:
            cleaned.append(term)
    return cleaned


def parse_anchor_titles(value):
    if value is None:
        return []
    if isinstance(value, (list, tuple, set)):
        raw_items = list(value)
    else:
        text = str(value).strip()
        if not text or text.lower() in {"nan", "none"}:
            return []
        raw_items = re.split(r"\s*[|;]\s*", text)
    cleaned = []
    for item in raw_items:
        title = re.sub(r"\s+", " ", str(item).strip().strip("'\"")).strip()
        if title and title.lower() not in {"nan", "none"} and title not in cleaned:
            cleaned.append(title)
    return cleaned


def build_candidate_term_lists(df, tags_clean=None):
    movie_terms = {}
    if tags_clean is not None and not tags_clean.empty and {"movieId", "tag_clean"}.issubset(tags_clean.columns) and "movieId" in df.columns:
        ids = set(pd.to_numeric(df["movieId"], errors="coerce").dropna().astype(int))
        tag_rows = tags_clean[tags_clean["movieId"].isin(ids)][["movieId", "tag_clean"]].dropna().drop_duplicates().copy()
        if not tag_rows.empty:
            movie_terms = tag_rows.groupby("movieId")["tag_clean"].apply(
                lambda values: list(dict.fromkeys(str(v).lower().strip() for v in values if str(v).strip()))
            ).to_dict()
    term_columns = [col for col in ["tags_features_en", "tags_clean", "tags", "tag_features", "semantic_explanation_terms", "core_semantic_explanation_terms"] if col in df.columns]
    result = []
    for _, row in df.iterrows():
        terms = []
        movie_id = pd.to_numeric(row.get("movieId", np.nan), errors="coerce")
        if pd.notna(movie_id):
            terms.extend(movie_terms.get(int(movie_id), []))
        for col in term_columns:
            terms.extend(parse_term_list(row.get(col, "")))
        if not terms:
            terms.extend(parse_term_list(row.get("genres", "")))
        result.append(list(dict.fromkeys(term for term in terms if term)))
    return pd.Series(result, index=df.index)


def build_tag_preference_table(candidates, candidate_terms, tags_clean=None, liked_movies=None, disliked_movies=None):
    rows = []
    for terms in candidate_terms:
        rows.extend(set(terms))
    catalog = pd.Series(rows, dtype="object").value_counts().rename("catalog_count").reset_index().rename(columns={"index": "term"})
    if catalog.empty:
        return pd.DataFrame(columns=["term", "tag_catalog_frequency", "tag_positive_frequency", "tag_negative_frequency", "tag_personal_lift", "tag_negative_lift", "tag_net_preference", "tag_specificity_score"])
    if "term" not in catalog.columns:
        catalog = catalog.rename(columns={catalog.columns[0]: "term"})
    n_catalog_movies = max(len(candidate_terms), 1)
    catalog["tag_catalog_frequency"] = catalog["catalog_count"] / n_catalog_movies

    def profile_frequency(profile_movies):
        if profile_movies is None or profile_movies.empty or tags_clean is None or tags_clean.empty or not {"movieId", "tag_clean"}.issubset(tags_clean.columns):
            return pd.DataFrame(columns=["term", "profile_count", "profile_weight"])
        seeds = profile_movies[["movieId", "user_rating_5"]].dropna().copy() if {"movieId", "user_rating_5"}.issubset(profile_movies.columns) else pd.DataFrame()
        if seeds.empty:
            return pd.DataFrame(columns=["term", "profile_count", "profile_weight"])
        seeds["movieId"] = pd.to_numeric(seeds["movieId"], errors="coerce")
        seeds = seeds.dropna(subset=["movieId"])
        seeds["movieId"] = seeds["movieId"].astype(int)
        tag_rows = tags_clean.merge(seeds, on="movieId", how="inner")
        if tag_rows.empty:
            return pd.DataFrame(columns=["term", "profile_count", "profile_weight"])
        tag_rows["profile_weight"] = pd.to_numeric(tag_rows["user_rating_5"], errors="coerce").fillna(3.0)
        return tag_rows.groupby("tag_clean").agg(profile_count=("movieId", "nunique"), profile_weight=("profile_weight", "mean")).reset_index().rename(columns={"tag_clean": "term"})

    positive = profile_frequency(liked_movies)
    negative = profile_frequency(disliked_movies)
    table = catalog.merge(positive, on="term", how="left").rename(columns={"profile_count": "positive_count", "profile_weight": "positive_weight"})
    table = table.merge(negative, on="term", how="left").rename(columns={"profile_count": "negative_count", "profile_weight": "negative_weight"})
    for col in ["positive_count", "negative_count", "positive_weight", "negative_weight"]:
        table[col] = pd.to_numeric(table[col], errors="coerce").fillna(0.0)
    n_positive = max(int(liked_movies["movieId"].nunique()) if liked_movies is not None and not liked_movies.empty and "movieId" in liked_movies.columns else 0, 1)
    n_negative = max(int(disliked_movies["movieId"].nunique()) if disliked_movies is not None and not disliked_movies.empty and "movieId" in disliked_movies.columns else 0, 1)
    table["tag_positive_frequency"] = table["positive_count"] / n_positive
    table["tag_negative_frequency"] = table["negative_count"] / n_negative
    smooth = 1.0 / max(n_catalog_movies, 1)
    table["tag_personal_lift"] = (table["tag_positive_frequency"] + smooth) / (table["tag_catalog_frequency"] + smooth)
    table["tag_negative_lift"] = (table["tag_negative_frequency"] + smooth) / (table["tag_catalog_frequency"] + smooth)
    table["tag_net_preference"] = table["tag_personal_lift"] - table["tag_negative_lift"]
    table["tag_specificity_score"] = -np.log(table["tag_catalog_frequency"].clip(lower=smooth))
    table["tag_specificity_score"] = robust_normalize_score(table["tag_specificity_score"])
    table["tag_net_preference_norm"] = robust_normalize_score(table["tag_net_preference"])
    return table


def _movie_term_metric(terms, lookup, column, default=0.0):
    values = []
    for term in terms:
        row = lookup.get(term)
        if row is None:
            continue
        values.append(float(row.get(column, default) or default))
    if not values:
        return default
    return float(np.mean(values))


def _top_personal_terms(terms, lookup, max_terms=4):
    scored_terms = []
    for term in terms:
        row = lookup.get(term)
        if row is None:
            continue
        net = float(row.get("tag_net_preference", 0.0) or 0.0)
        specificity = float(row.get("tag_specificity_score", 0.0) or 0.0)
        personal_lift = float(row.get("tag_personal_lift", 0.0) or 0.0)
        negative_lift = float(row.get("tag_negative_lift", 0.0) or 0.0)
        if net <= 0 or personal_lift <= negative_lift or specificity < 0.35:
            continue
        scored_terms.append((net + 0.35 * specificity, term))
    scored_terms = sorted(scored_terms, reverse=True)
    return [term for _, term in scored_terms[:max_terms]]


def _safe_quantile(values, q, default=0.0):
    series = pd.to_numeric(pd.Series(values), errors="coerce").dropna()
    return float(series.quantile(q)) if not series.empty else float(default)


def _weighted_decade_distribution(profile_movies, positive=True):
    if profile_movies is None or profile_movies.empty or not {"year", "user_rating_5"}.issubset(profile_movies.columns):
        return {}, 0, np.nan
    years = pd.to_numeric(profile_movies["year"], errors="coerce")
    ratings = pd.to_numeric(profile_movies["user_rating_5"], errors="coerce")
    mask = years.notna() & ratings.notna()
    if positive:
        mask &= ratings >= 4.0
        weights = (ratings.loc[mask] - 3.0).clip(lower=0.25)
    else:
        mask &= ratings <= 2.5
        weights = (3.0 - ratings.loc[mask]).clip(lower=0.25)
    if not mask.any() or weights.sum() <= 0:
        return {}, 0, np.nan
    decades = ((years.loc[mask] // 10) * 10).astype(int)
    weighted = weights.groupby(decades).sum().astype(float)
    share = (weighted / weighted.sum()).to_dict()
    entropy = -sum(float(p) * np.log(float(p)) for p in share.values() if p > 0)
    max_entropy = np.log(max(len(share), 1)) if share else np.nan
    normalized_entropy = float(entropy / max_entropy) if pd.notna(max_entropy) and max_entropy > 0 else 0.0
    return share, int(mask.sum()), normalized_entropy


def infer_recommendation_branch(row):
    genres = set(split_genres(row.get("genres", "")))
    semantic_branch = str(row.get("semantic_branch", "")).strip().lower()
    personal_terms = set(parse_term_list(row.get("personal_explanation_terms", "")))
    rating = float(row.get("rating_score", 0.0) or 0.0)
    popularity = float(row.get("popularity_score", 0.0) or 0.0)
    margin = float(row.get("preference_margin_score", 0.0) or 0.0)
    temporal_distance = float(row.get("temporal_distance_from_profile", 0.0) or 0.0)
    classic_distance_threshold = float(row.get("classic_distance_threshold", np.inf) or np.inf)
    consolidated = rating >= float(row.get("classic_rating_threshold", 0.65) or 0.65) or popularity >= float(row.get("classic_popularity_threshold", 0.65) or 0.65)

    if temporal_distance >= classic_distance_threshold and consolidated and margin < 0.80:
        return "classic_quality_match"
    if "Animation" in genres or "Children" in genres:
        return "animation_family"
    if "Sci-Fi" in genres:
        return "sci_fi_reflective"
    if "Crime" in genres and ("Thriller" in genres or "Mystery" in genres or semantic_branch == "crime_thriller"):
        return "crime_thriller"
    if "Mystery" in genres and "Thriller" in genres:
        return "psychological_thriller"
    if "Thriller" in genres and ({"psychology", "paranoia", "mystery", "twist"} & personal_terms or "psychological" in semantic_branch):
        return "psychological_thriller"
    if "Comedy" in genres and ({"satire", "surrealism", "absurd", "dark comedy"} & personal_terms or "surreal" in semantic_branch):
        return "satire_surreal_comedy"
    if "Drama" in genres and "War" in genres and consolidated and temporal_distance >= classic_distance_threshold:
        return "classic_quality_match"
    if "Drama" in genres and not ({"Crime", "Sci-Fi", "Horror", "Fantasy", "Thriller", "Mystery"} & genres):
        return "emotional_character_drama"
    if "Action" in genres or "Adventure" in genres:
        return "action_quality_match"
    if semantic_branch in {
        "crime_thriller", "psychological_thriller", "sci_fi_reflective",
        "animation_family", "satire_surreal_comedy",
    }:
        return semantic_branch
    return "general_quality_match"



def _recommendation_bucket(row):
    score = float(row.get("final_recommendation_score", row.get("human_like_v3_adjusted_score", row.get("human_like_rank_score", 0.0))) or 0.0)
    anchor = float(row.get("anchor_match_score", 0.0) or 0.0)
    specificity = float(row.get("anchor_specificity_score", 0.0) or 0.0)
    risk = float(row.get("false_positive_risk", 0.0) or 0.0)
    risk_medium = float(row.get("risk_medium_threshold", 0.40) or 0.40)
    risk_high = float(row.get("risk_high_threshold", max(risk_medium, 0.60)) or max(risk_medium, 0.60))
    margin = float(row.get("preference_margin_score", 0.0) or 0.0)
    rating = float(row.get("rating_score", 0.0) or 0.0)
    popularity = float(row.get("popularity_score", 0.0) or 0.0)
    base = float(row.get("base_model_evidence_score", 0.0) or 0.0)
    temporal = float(row.get("user_temporal_affinity_score", 0.5) or 0.5)
    temporal_penalty = float(row.get("temporal_mismatch_penalty", 0.0) or 0.0)
    temporal_medium = float(row.get("temporal_mismatch_medium_threshold", 0.30) or 0.30)
    jump_penalty = float(row.get("rerank_jump_penalty", 0.0) or 0.0)
    anchor_confidence = str(row.get("anchor_confidence", "bajo"))
    num_anchors = int(row.get("num_coherent_anchors", 0) or 0)
    weak_anchor = float(row.get("weak_anchor_penalty", 1.0) or 1.0)
    explanation_penalty = float(row.get("explanation_quality_penalty", 0.0) or 0.0)
    temporal_distance = float(row.get("temporal_distance_from_profile", 0.0) or 0.0)
    classic_distance = float(row.get("classic_distance_threshold", np.inf) or np.inf)
    rating_threshold = float(row.get("classic_rating_threshold", 0.65) or 0.65)
    popularity_threshold = float(row.get("classic_popularity_threshold", 0.65) or 0.65)
    consolidated = rating >= rating_threshold or popularity >= popularity_threshold
    temporally_distant = temporal_distance >= classic_distance or temporal_penalty >= temporal_medium

    strong_personal_evidence = anchor_confidence == "alto" and num_anchors >= 2 and margin >= 0.70 and risk < risk_medium and jump_penalty < 0.25 and weak_anchor <= 0.30
    if risk >= risk_high and not strong_personal_evidence:
        return "riesgo_controlado"
    if risk >= risk_medium or margin < 0.45 or temporal_penalty >= temporal_medium or jump_penalty >= 0.35 or weak_anchor >= 0.70:
        return "clasico_pendiente" if temporally_distant and consolidated and temporal >= 0.35 else "riesgo_controlado"
    if temporally_distant and consolidated and anchor_confidence != "alto":
        return "clasico_pendiente"
    if anchor_confidence == "alto" and num_anchors >= 2 and anchor >= 0.85 and specificity >= 0.45 and risk < risk_medium and jump_penalty < 0.30 and explanation_penalty <= 0.40:
        return "muy_parecida_a_favoritas"
    if score >= 0.72 and risk < risk_medium and margin >= 0.60 and base >= 0.50 and temporal_penalty < temporal_medium and jump_penalty < 0.25 and popularity < 0.90:
        return "apuesta_segura"
    if score >= 0.50 and margin >= 0.50 and base >= 0.40 and temporal >= 0.35 and jump_penalty < 0.40:
        return "descubrimiento_compatible"
    return "riesgo_controlado"


def build_explanation_display_from_row(row):
    bucket = str(row.get("recommendation_bucket", ""))
    branch = str(row.get("recommendation_branch", "general_quality_match"))
    signal = str(row.get("dominant_signal", ""))
    anchors = str(row.get("anchor_movies_matched", "")).strip()
    margin = float(row.get("preference_margin_score", 0.0) or 0.0)
    risk = float(row.get("false_positive_risk", 0.0) or 0.0)
    temporal = float(row.get("temporal_mismatch_penalty", 0.0) or 0.0)
    rating = float(row.get("rating_score", 0.0) or 0.0)
    parts = []
    if bucket == "clasico_pendiente":
        parts.append("Clasico compatible: entra por calidad o popularidad consolidada y afinidad razonable con tu perfil.")
    elif bucket == "riesgo_controlado":
        parts.append("Recomendacion buena pero menos segura: se controla por riesgo, margen o salto de ranking.")
    elif bucket == "muy_parecida_a_favoritas" and anchors:
        parts.append("Recomendacion basada en anclas coherentes de tus peliculas mejor valoradas.")
    elif bucket == "apuesta_segura":
        parts.append("Apuesta segura por buen score final, bajo riesgo relativo y margen positivo frente al perfil negativo.")
    else:
        parts.append("Descubrimiento compatible con tu perfil por equilibrio entre afinidad, evidencia base y riesgo controlado.")
    parts.append(f"Rama: {branch}.")
    if anchors and bucket == "muy_parecida_a_favoritas":
        parts.append("Anclas: " + anchors + ".")
    if margin >= 0.60:
        parts.append("Buen margen positivo-negativo.")
    if temporal <= 0.20:
        parts.append("Encaje temporal cercano al perfil.")
    if rating >= 0.65 and bucket == "clasico_pendiente":
        parts.append("Destaca por calidad media alta.")
    if signal:
        parts.append(f"Senal principal: {signal}.")
    if risk >= float(row.get("risk_medium_threshold", 0.40) or 0.40):
        parts.append("Riesgo revisado de forma explicita.")
    return " ".join(parts).strip()



def _weighted_year_mean(profile_movies):
    if profile_movies is None or profile_movies.empty or not {"year", "user_rating_5"}.issubset(profile_movies.columns):
        return np.nan
    years = pd.to_numeric(profile_movies["year"], errors="coerce")
    ratings = pd.to_numeric(profile_movies["user_rating_5"], errors="coerce")
    mask = years.notna() & ratings.notna()
    if not mask.any():
        return np.nan
    weights = (ratings.loc[mask] - 3.0).clip(lower=0.25)
    return float(np.average(years.loc[mask], weights=weights))


def _temporal_affinity_from_profile(candidate_years, profile_movies, positive=True, bandwidth=12.0):
    years = pd.to_numeric(pd.Series(candidate_years), errors="coerce")
    if profile_movies is None or profile_movies.empty or not {"year", "user_rating_5"}.issubset(profile_movies.columns):
        return pd.Series(0.5, index=years.index, dtype=float)
    seed_years = pd.to_numeric(profile_movies["year"], errors="coerce")
    seed_ratings = pd.to_numeric(profile_movies["user_rating_5"], errors="coerce")
    mask = seed_years.notna() & seed_ratings.notna()
    if positive:
        mask &= seed_ratings >= 4.0
        weights = (seed_ratings.loc[mask] - 3.0).clip(lower=0.25)
    else:
        mask &= seed_ratings <= 2.5
        weights = (3.0 - seed_ratings.loc[mask]).clip(lower=0.25)
    if not mask.any() or weights.sum() <= 0:
        return pd.Series(0.0 if not positive else 0.5, index=years.index, dtype=float)
    seed_years = seed_years.loc[mask].astype(float).to_numpy()
    weights = weights.astype(float).to_numpy()
    result = []
    for year in years:
        if pd.isna(year):
            result.append(0.5 if positive else 0.0)
            continue
        sims = np.exp(-np.abs(float(year) - seed_years) / bandwidth)
        result.append(float(np.average(sims, weights=weights)))
    return pd.Series(result, index=years.index, dtype=float).clip(0.0, 1.0)


def ensure_temporal_features(scored, liked_movies=None, disliked_movies=None):
    result = scored.copy()
    years = pd.to_numeric(result.get("year", pd.Series(np.nan, index=result.index)), errors="coerce")
    result["decade"] = ((years // 10) * 10).astype("Int64")
    positive_decade_share, n_positive_temporal, positive_temporal_entropy = _weighted_decade_distribution(liked_movies, positive=True)
    negative_decade_share, n_negative_temporal, _ = _weighted_decade_distribution(disliked_movies, positive=False)
    candidate_decade_share = result["decade"].value_counts(normalize=True, dropna=True).to_dict()
    fallback_share = 1.0 / max(len(candidate_decade_share), 1)
    exploration_margin = float(0.06 + min(0.18, 1.0 / np.sqrt(max(n_positive_temporal, 1) + 1.0)) * 0.18 + max(0.0, positive_temporal_entropy) * 0.08)
    if not positive_decade_share:
        positive_decade_share = {int(k): float(v) for k, v in candidate_decade_share.items() if pd.notna(k)}

    def expected_share(decade_value):
        if pd.isna(decade_value):
            return fallback_share
        return float(positive_decade_share.get(int(decade_value), 0.25 * candidate_decade_share.get(decade_value, fallback_share)))

    result["expected_decade_share"] = result["decade"].apply(expected_share).astype(float).clip(0.0, 1.0)
    result["negative_decade_share"] = result["decade"].apply(lambda value: float(negative_decade_share.get(int(value), 0.0)) if pd.notna(value) else 0.0).astype(float).clip(0.0, 1.0)
    result["temporal_exploration_margin"] = exploration_margin
    result["actual_top_decade_share"] = 0.0
    result["temporal_overrepresentation_penalty"] = 0.0
    result["temporal_portfolio_penalty"] = 0.0

    computed_positive_affinity = _temporal_affinity_from_profile(years, liked_movies, positive=True)
    computed_negative_affinity = _temporal_affinity_from_profile(years, disliked_movies, positive=False)

    if "year_affinity_score" in result.columns and pd.to_numeric(result["year_affinity_score"], errors="coerce").notna().any():
        result["user_temporal_affinity_score"] = _series_01(result, "year_affinity_score")
    else:
        result["year_affinity_score"] = computed_positive_affinity
        result["user_temporal_affinity_score"] = computed_positive_affinity

    temporal_margin_raw = result["user_temporal_affinity_score"] - computed_negative_affinity
    result["temporal_preference_margin_score"] = robust_normalize_score(temporal_margin_raw, lower_q=0.01, upper_q=0.99)

    positive_year_mean = _weighted_year_mean(liked_movies)
    positive_years = pd.to_numeric(liked_movies["year"], errors="coerce").dropna() if liked_movies is not None and not liked_movies.empty and "year" in liked_movies.columns else pd.Series(dtype=float)
    year_scale = float(max(positive_years.std(ddof=0), 8.0)) if not positive_years.empty else 12.0
    computed_distance = (years - positive_year_mean).abs() / year_scale if pd.notna(positive_year_mean) else pd.Series(0.0, index=result.index)
    if "temporal_distance_from_profile" not in result.columns or pd.to_numeric(result["temporal_distance_from_profile"], errors="coerce").notna().sum() == 0:
        result["temporal_distance_from_profile"] = computed_distance.fillna(0.0)

    computed_mismatch = (
        0.50 * (1.0 - result["user_temporal_affinity_score"])
        + 0.30 * computed_negative_affinity
        + 0.20 * robust_normalize_score(result["temporal_distance_from_profile"], lower_q=0.01, upper_q=0.99)
    ).clip(0.0, 1.0)
    if "temporal_mismatch_penalty" not in result.columns or pd.to_numeric(result["temporal_mismatch_penalty"], errors="coerce").notna().sum() == 0:
        result["temporal_mismatch_penalty"] = computed_mismatch
    else:
        result["temporal_mismatch_penalty"] = pd.to_numeric(result["temporal_mismatch_penalty"], errors="coerce").fillna(computed_mismatch).clip(0.0, 1.0)

    result["temporal_confidence"] = np.select(
        [
            result["user_temporal_affinity_score"] >= 0.65,
            result["user_temporal_affinity_score"] >= 0.40,
        ],
        ["alto", "medio"],
        default="bajo",
    )
    if "is_temporal_outlier" not in result.columns:
        threshold = result["temporal_distance_from_profile"].quantile(0.90) if result["temporal_distance_from_profile"].notna().any() else np.inf
        result["is_temporal_outlier"] = result["temporal_distance_from_profile"] >= threshold
    result["classic_distance_threshold"] = _safe_quantile(result["temporal_distance_from_profile"], 0.80, default=1.0)
    result["temporal_mismatch_medium_threshold"] = _safe_quantile(result["temporal_mismatch_penalty"], 0.70, default=0.30)
    result["classic_rating_threshold"] = _safe_quantile(result.get("rating_score", pd.Series(0.65, index=result.index)), 0.65, default=0.65)
    result["classic_popularity_threshold"] = _safe_quantile(result.get("popularity_score", pd.Series(0.65, index=result.index)), 0.65, default=0.65)
    result.attrs["positive_decade_share"] = positive_decade_share
    result.attrs["positive_temporal_n"] = n_positive_temporal
    result.attrs["positive_temporal_entropy"] = positive_temporal_entropy
    return result


def add_human_like_recommendation_layer(df, tags_clean=None, liked_movies=None, disliked_movies=None):
    scored = df.copy()
    if "main_genre" not in scored.columns and "genres" in scored.columns:
        scored["main_genre"] = scored["genres"].apply(main_genre_from_genres)
    scored = ensure_temporal_features(scored, liked_movies=liked_movies, disliked_movies=disliked_movies)

    n_candidates = max(len(scored), 1)
    scored["previous_hybrid_rank"] = scored["hybrid_score"].rank(method="min", ascending=False).astype(int) if "hybrid_score" in scored.columns else n_candidates
    scored["previous_semantic_rank"] = scored["semantic_relevance_adjusted_score"].rank(method="min", ascending=False).astype(int) if "semantic_relevance_adjusted_score" in scored.columns else n_candidates
    scored["previous_hybrid_percentile"] = 1.0 - ((scored["previous_hybrid_rank"] - 1) / max(n_candidates - 1, 1))
    scored["previous_semantic_percentile"] = 1.0 - ((scored["previous_semantic_rank"] - 1) / max(n_candidates - 1, 1))

    semantic_adjusted = _series_01(scored, "semantic_relevance_adjusted_score")
    semantic_relevance = _series_01(scored, "semantic_relevance_score")
    latent_preference = _series_01(scored, "latent_core_preference_score")
    semantic_profile = _series_01(scored, "semantic_profile_score")
    content_profile = _series_01(scored, "content_profile_score")
    collab = _series_01(scored, "item_item_collab_score")
    rating = _series_01(scored, "rating_score")
    popularity = _series_01(scored, "popularity_score")
    hybrid = _series_01(scored, "hybrid_score", normalize_if_needed=True)
    temporal_affinity = _series_01(scored, "user_temporal_affinity_score")
    temporal_mismatch = _series_01(scored, "temporal_mismatch_penalty")
    temporal_margin = _series_01(scored, "temporal_preference_margin_score")

    negative_semantic = _series_01(scored, "negative_semantic_relevance_score")
    negative_genre = _series_01(scored, "negative_genre_score")
    negative_similarity = _series_01(scored, "negative_similarity_score")
    negative_collab = _series_01(scored, "item_item_negative_collab_score")
    negative_semantic_profile = _series_01(scored, "negative_semantic_score")
    negative_latent = _series_01(scored, "negative_latent_preference_score")
    if "negative_latent_preference_score" not in scored.columns:
        negative_latent = _series_01(scored, "negative_latent_similarity_score")

    candidate_terms = build_candidate_term_lists(scored, tags_clean=tags_clean)
    tag_preference_table = build_tag_preference_table(scored, candidate_terms, tags_clean=tags_clean, liked_movies=liked_movies, disliked_movies=disliked_movies)
    tag_lookup = tag_preference_table.set_index("term").to_dict("index") if not tag_preference_table.empty else {}
    scored["personal_explanation_terms"] = candidate_terms.apply(lambda terms: " | ".join(_top_personal_terms(terms, tag_lookup, max_terms=4)))
    scored["anchor_specificity_score"] = candidate_terms.apply(lambda terms: _movie_term_metric(terms, tag_lookup, "tag_specificity_score", default=0.0))
    scored["tag_net_preference_score"] = candidate_terms.apply(lambda terms: _movie_term_metric(terms, tag_lookup, "tag_net_preference_norm", default=0.0))
    generic_raw = candidate_terms.apply(
        lambda terms: np.mean([
            1.0 - float(tag_lookup.get(term, {}).get("tag_specificity_score", 0.0) or 0.0)
            + max(0.0, 0.55 - float(tag_lookup.get(term, {}).get("tag_net_preference_norm", 0.0) or 0.0))
            for term in terms if term in tag_lookup
        ]) if any(term in tag_lookup for term in terms) else 0.50
    )
    scored["generic_semantic_penalty"] = robust_normalize_score(generic_raw).clip(0.0, 1.0)
    scored["anchor_specificity_score"] = robust_normalize_score(0.65 * scored["anchor_specificity_score"] + 0.35 * scored["tag_net_preference_score"]).clip(0.0, 1.0)
    scored["low_specificity_penalty"] = (1.0 - scored["anchor_specificity_score"]).clip(0.0, 1.0)

    positive_anchor_meta = {}
    if liked_movies is not None and not liked_movies.empty and {"title", "user_rating_5"}.issubset(liked_movies.columns):
        liked_meta = liked_movies.copy()
        liked_meta["user_rating_5"] = pd.to_numeric(liked_meta["user_rating_5"], errors="coerce")
        liked_meta = liked_meta[liked_meta["user_rating_5"] >= 4.0].copy()
        for _, anchor_row in liked_meta.iterrows():
            title = str(anchor_row.get("title", "")).strip()
            if title:
                positive_anchor_meta[title] = {
                    "rating": float(anchor_row.get("user_rating_5", 4.0) or 4.0),
                    "genres": set(split_genres(anchor_row.get("genres", ""))),
                    "year": pd.to_numeric(anchor_row.get("year", np.nan), errors="coerce"),
                }
    negative_title_set = set()
    if disliked_movies is not None and not disliked_movies.empty and {"title", "user_rating_5"}.issubset(disliked_movies.columns):
        negative_title_set = set(disliked_movies.loc[pd.to_numeric(disliked_movies["user_rating_5"], errors="coerce") <= 2.5, "title"].dropna().astype(str))

    def matched_anchors(row):
        candidate_genres = set(split_genres(row.get("genres", "")))
        candidate_year = pd.to_numeric(row.get("year", np.nan), errors="coerce")
        rows = []
        for col in ["nearest_core_anchor_movies", "similar_liked_movies", "nearest_liked_movies_latent"]:
            for title in parse_anchor_titles(row.get(col, "")):
                if title in negative_title_set or not title:
                    continue
                meta = positive_anchor_meta.get(title, {})
                if positive_anchor_meta and title not in positive_anchor_meta and col != "nearest_core_anchor_movies":
                    continue
                anchor_genres = meta.get("genres", set())
                genre_overlap = len(candidate_genres & anchor_genres) / max(len(candidate_genres | anchor_genres), 1) if anchor_genres else 0.0
                anchor_year = meta.get("year", np.nan)
                temporal_compat = np.exp(-abs(float(candidate_year) - float(anchor_year)) / 18.0) if pd.notna(candidate_year) and pd.notna(anchor_year) else 0.5
                compatibility = 0.55 * genre_overlap + 0.25 * temporal_compat + 0.20 * float(row.get("anchor_specificity_score", 0.0) or 0.0)
                if compatibility < 0.28:
                    continue
                if compatibility < 0.35 and (float(row.get("preference_margin_score", 0.0) or 0.0) < 0.75 or float(row.get("anchor_specificity_score", 0.0) or 0.0) < 0.55):
                    continue
                rows.append((compatibility, float(meta.get("rating", 4.0)), title))
        rows = sorted(rows, key=lambda item: (item[0], item[1]), reverse=True)
        anchors = []
        for compatibility, rating_value, title in rows:
            if title not in [anchor_title for anchor_title, _ in anchors]:
                anchors.append((title, compatibility))
            if len(anchors) >= 3:
                break
        return anchors

    base_anchor_signal = _weighted_average_components([
        (0.30, _series_01(scored, "topk_anchor_similarity", normalize_if_needed=True) if "topk_anchor_similarity" in scored.columns else None),
        (0.22, latent_preference if "latent_core_preference_score" in scored.columns else None),
        (0.14, _series_01(scored, "positive_centroid_similarity", normalize_if_needed=True) if "positive_centroid_similarity" in scored.columns else None),
        (0.12, _series_01(scored, "latent_core_similarity_score", normalize_if_needed=True) if "latent_core_similarity_score" in scored.columns else None),
        (0.12, collab),
        (0.10, temporal_affinity),
    ], scored.index)
    scored["anchor_match_score"] = base_anchor_signal.clip(0.0, 1.0)

    anchor_rows = scored.apply(matched_anchors, axis=1)
    anchor_lists = anchor_rows.apply(lambda values: [title for title, _ in values])
    anchor_count = anchor_lists.apply(len)
    scored["num_coherent_anchors"] = anchor_count.astype(int)
    scored["anchor_coherence_score"] = anchor_rows.apply(lambda values: float(np.mean([score for _, score in values])) if values else 0.0).clip(0.0, 1.0)
    scored["weak_anchor_penalty"] = (1.0 - scored["anchor_coherence_score"]).where(anchor_count > 0, 1.0).clip(0.0, 1.0)
    anchor_presence = (anchor_count / 3.0).clip(0.0, 1.0)
    scored["anchor_match_score"] = (0.72 * scored["anchor_match_score"] + 0.18 * anchor_presence + 0.10 * scored["anchor_coherence_score"]).clip(0.0, 1.0)
    scored["anchor_movies_matched"] = anchor_lists.apply(lambda values: " | ".join(values))

    scored["positive_affinity_score"] = _weighted_average_components([
        (0.23, semantic_adjusted),
        (0.12, semantic_relevance if "semantic_relevance_score" in scored.columns else None),
        (0.14, latent_preference if "latent_core_preference_score" in scored.columns else None),
        (0.10, semantic_profile if "semantic_profile_score" in scored.columns else None),
        (0.09, content_profile if "content_profile_score" in scored.columns else None),
        (0.15, collab),
        (0.08, scored["anchor_match_score"]),
        (0.05, rating),
        (0.04, temporal_affinity),
    ], scored.index)
    scored["negative_affinity_score"] = _weighted_average_components([
        (0.30, negative_semantic),
        (0.18, negative_genre),
        (0.16, negative_similarity if "negative_similarity_score" in scored.columns else None),
        (0.14, negative_collab if "item_item_negative_collab_score" in scored.columns else None),
        (0.14, negative_semantic_profile if "negative_semantic_score" in scored.columns else None),
        (0.08, negative_latent),
    ], scored.index)
    scored["preference_margin_raw"] = scored["positive_affinity_score"] - scored["negative_affinity_score"]
    scored["preference_margin_score"] = robust_normalize_score(scored["preference_margin_raw"], lower_q=0.01, upper_q=0.99)

    scored["popularity_without_anchor_penalty"] = (popularity * (1.0 - scored["anchor_match_score"]) * (1.0 - scored["anchor_specificity_score"])).clip(0.0, 1.0)
    scored["false_positive_risk"] = (
        0.30 * scored["negative_affinity_score"]
        + 0.25 * (1.0 - scored["preference_margin_score"])
        + 0.13 * (1.0 - scored["anchor_match_score"])
        + 0.10 * scored["generic_semantic_penalty"]
        + 0.08 * negative_collab
        + 0.05 * scored["popularity_without_anchor_penalty"]
        + 0.05 * scored["low_specificity_penalty"]
        + 0.04 * temporal_mismatch
    ).clip(0.0, 1.0)
    scored["risk_medium_threshold"] = _safe_quantile(scored["false_positive_risk"], 0.70, default=0.40)
    scored["risk_high_threshold"] = _safe_quantile(scored["false_positive_risk"], 0.85, default=0.60)
    scored["anchor_confidence"] = np.select(
        [
            (scored["anchor_match_score"] >= 0.85)
            & (scored["preference_margin_score"] >= 0.70)
            & (scored["false_positive_risk"] < scored["risk_medium_threshold"])
            & (scored["anchor_specificity_score"] >= 0.45)
            & (scored["num_coherent_anchors"] >= 2)
            & (scored["anchor_coherence_score"] >= 0.45)
            & ((temporal_mismatch <= scored["temporal_mismatch_medium_threshold"]) | (scored["anchor_match_score"] >= 0.93)),
            (scored["anchor_match_score"] >= 0.65)
            & (scored["preference_margin_score"] >= 0.55)
            & (scored["false_positive_risk"] <= scored["risk_high_threshold"])
            & (scored["num_coherent_anchors"] >= 1)
            & (scored["anchor_coherence_score"] >= 0.35),
        ],
        ["alto", "medio"],
        default="bajo",
    )

    def explanation_term_quality(terms):
        good_terms = _top_personal_terms(terms, tag_lookup, max_terms=4)
        scores = []
        for term in good_terms:
            row = tag_lookup.get(term, {})
            net = float(row.get("tag_net_preference_norm", 0.0) or 0.0)
            specificity = float(row.get("tag_specificity_score", 0.0) or 0.0)
            personal_lift = float(row.get("tag_personal_lift", 0.0) or 0.0)
            negative_lift = float(row.get("tag_negative_lift", 0.0) or 0.0)
            lift_margin = 1.0 if personal_lift > negative_lift else 0.0
            scores.append(0.45 * net + 0.35 * specificity + 0.20 * lift_margin)
        return float(np.mean(scores)) if scores else 0.0

    scored["explanation_term_score"] = candidate_terms.apply(explanation_term_quality).clip(0.0, 1.0)
    scored["explanation_good_term_count"] = scored["personal_explanation_terms"].apply(lambda value: len(parse_term_list(value)))
    scored["explanation_quality_score"] = (
        0.45 * scored["explanation_term_score"]
        + 0.30 * scored["anchor_coherence_score"]
        + 0.15 * scored["preference_margin_score"]
        + 0.10 * (1.0 - scored["generic_semantic_penalty"])
    ).clip(0.0, 1.0)
    scored.loc[(scored["explanation_good_term_count"] < 2) & (scored["num_coherent_anchors"] == 0), "explanation_quality_score"] *= 0.70
    scored["explanation_quality_penalty"] = (1.0 - scored["explanation_quality_score"]).clip(0.0, 1.0)

    scored["base_model_evidence_score"] = _weighted_average_components([
        (0.30, hybrid),
        (0.24, semantic_adjusted),
        (0.18, collab),
        (0.13, rating),
        (0.05, popularity),
        (0.10, temporal_affinity),
    ], scored.index)
    big_jump = (scored["previous_hybrid_rank"] > 300) & (scored["previous_semantic_rank"] > 300)
    huge_jump = scored["previous_hybrid_rank"] > 800
    jump_raw = robust_normalize_score(np.log1p(scored[["previous_hybrid_rank", "previous_semantic_rank"]].max(axis=1)), lower_q=0.05, upper_q=0.99)
    strong_jump_justification = (
        (scored["anchor_confidence"] == "alto")
        & (scored["preference_margin_score"] >= 0.70)
        & (scored["false_positive_risk"] <= 0.35)
        & (temporal_mismatch <= 0.55)
        & (scored["base_model_evidence_score"] >= 0.45)
    )
    scored["rerank_jump_penalty"] = (jump_raw * (1.0 - scored["base_model_evidence_score"]) * big_jump.astype(float)).clip(0.0, 1.0)
    scored.loc[big_jump & (scored["anchor_confidence"] == "bajo"), "rerank_jump_penalty"] = (scored.loc[big_jump & (scored["anchor_confidence"] == "bajo"), "rerank_jump_penalty"] + 0.15).clip(upper=1.0)
    scored.loc[strong_jump_justification, "rerank_jump_penalty"] *= 0.35

    scored["contrib_anchor"] = 0.19 * scored["anchor_match_score"]
    scored["contrib_margin"] = 0.16 * scored["preference_margin_score"]
    scored["contrib_semantic"] = 0.13 * semantic_adjusted
    scored["contrib_collab"] = 0.11 * collab
    scored["contrib_quality"] = 0.09 * rating
    scored["contrib_popularity"] = 0.05 * popularity
    scored["contrib_specificity"] = 0.07 * scored["anchor_specificity_score"]
    scored["contrib_temporal"] = 0.08 * temporal_affinity
    scored["contrib_base_model"] = 0.05 * scored["base_model_evidence_score"]
    scored["contrib_hybrid_base"] = 0.04 * hybrid
    scored["penalty_risk"] = 0.24 * scored["false_positive_risk"]
    scored["penalty_temporal"] = 0.08 * temporal_mismatch
    scored["penalty_rerank_jump"] = 0.08 * scored["rerank_jump_penalty"]
    scored["human_like_v3_raw_score"] = (
        scored["contrib_anchor"]
        + scored["contrib_margin"]
        + scored["contrib_semantic"]
        + scored["contrib_collab"]
        + scored["contrib_quality"]
        + scored["contrib_popularity"]
        + scored["contrib_specificity"]
        + scored["contrib_temporal"]
        + scored["contrib_base_model"]
        + scored["contrib_hybrid_base"]
        - scored["penalty_risk"]
        - scored["penalty_temporal"]
        - scored["penalty_rerank_jump"]
    )
    scored["human_like_raw_score"] = scored["human_like_v3_raw_score"]
    raw = scored["human_like_v3_raw_score"]
    raw_norm = robust_normalize_score(raw, lower_q=0.01, upper_q=0.99)
    scored["human_like_rank_percentile"] = raw.rank(method="average", pct=True)
    scored["human_like_rank_score"] = (0.72 * raw_norm + 0.28 * scored["human_like_rank_percentile"]).clip(0.0, 1.0)
    scored["human_like_score"] = robust_normalize_score(scored["human_like_v3_raw_score"], lower_q=0.005, upper_q=0.995)
    scored["human_like_v3_adjusted_score"] = (
        scored["human_like_v3_raw_score"]
        - 0.08 * scored["rerank_jump_penalty"]
        - 0.06 * scored["temporal_mismatch_penalty"]
        - 0.05 * scored["weak_anchor_penalty"]
        - 0.05 * scored["explanation_quality_penalty"]
    )
    scored["final_recommendation_score"] = make_display_score(scored["human_like_v3_adjusted_score"])

    high_risk = scored["false_positive_risk"] >= 0.55
    strong_exception = (scored["anchor_match_score"] >= 0.90) & (scored["anchor_specificity_score"] >= 0.70) & (scored["preference_margin_score"] >= 0.70)
    scored.loc[high_risk & ~strong_exception, "human_like_rank_score"] = scored.loc[high_risk & ~strong_exception, "human_like_rank_score"].clip(upper=0.84)
    low_margin = scored["preference_margin_score"] < 0.45
    scored.loc[low_margin, "human_like_rank_score"] = (scored.loc[low_margin, "human_like_rank_score"] * 0.90).clip(upper=0.78)
    scored.loc[huge_jump & ~strong_jump_justification, "human_like_rank_score"] = scored.loc[huge_jump & ~strong_jump_justification, "human_like_rank_score"].clip(upper=0.84)

    scored["allow_temporal_outlier"] = (
        (scored["anchor_match_score"] >= 0.80)
        & (scored["preference_margin_score"] >= 0.65)
        & (scored["false_positive_risk"] <= 0.45)
        & (rating >= 0.55)
        & (scored["anchor_specificity_score"] >= 0.45)
    )
    scored["recommendation_branch"] = scored.apply(infer_recommendation_branch, axis=1)
    scored["recommendation_bucket"] = scored.apply(_recommendation_bucket, axis=1)

    contribution_frame = pd.DataFrame({
        "anchor": scored["contrib_anchor"],
        "margin": scored["contrib_margin"],
        "semantic": scored["contrib_semantic"],
        "collaborative": scored["contrib_collab"],
        "quality": scored["contrib_quality"],
        "popularity": scored["contrib_popularity"],
        "temporal": scored["contrib_temporal"],
    }, index=scored.index)
    raw_dominant = contribution_frame.idxmax(axis=1)
    anchor_not_strong = (raw_dominant == "anchor") & (scored["anchor_confidence"] != "alto")
    second_best = contribution_frame.drop(columns=["anchor"]).idxmax(axis=1)
    raw_dominant.loc[anchor_not_strong] = second_best.loc[anchor_not_strong]
    scored["dominant_signal"] = raw_dominant
    scored.loc[scored["rerank_jump_penalty"] >= 0.30, "dominant_signal"] = "rerank_limited"
    scored.loc[(scored["penalty_risk"] >= contribution_frame.max(axis=1) * 0.80) | (scored["false_positive_risk"] >= scored["risk_medium_threshold"]), "dominant_signal"] = "risk_penalized"
    scored.loc[(scored["recommendation_bucket"] == "clasico_pendiente") & (scored["contrib_quality"] >= contribution_frame.drop(columns=["quality"]).max(axis=1)), "dominant_signal"] = "quality"
    scored.loc[(scored["recommendation_bucket"] == "clasico_pendiente") & (scored["contrib_temporal"] >= contribution_frame.drop(columns=["temporal"]).max(axis=1)), "dominant_signal"] = "temporal"
    scored.loc[(scored["recommendation_bucket"] == "riesgo_controlado") & (scored["dominant_signal"] != "rerank_limited"), "dominant_signal"] = "risk_penalized"

    def why_matches(row):
        parts = []
        anchors = str(row.get("anchor_movies_matched", "")).strip()
        terms = parse_term_list(row.get("personal_explanation_terms", ""))[:4]
        bucket = row.get("recommendation_bucket", "")
        branch = row.get("recommendation_branch", "general_quality_match")
        signal = row.get("dominant_signal", "")
        if anchors:
            parts.append(f"anclas: {anchors}")
        parts.append(f"rama: {branch}")
        if terms:
            parts.append("tags personales: " + ", ".join(terms))
        if bucket == "clasico_pendiente":
            parts.append("clasico pendiente compatible, no solo parecido directo")
        if signal == "temporal":
            parts.append("temporalidad cercana al perfil")
        if signal:
            parts.append(f"senal principal: {signal}")
        if row.get("final_recommendation_score", np.nan) == row.get("final_recommendation_score", np.nan):
            parts.append(f"score final: {float(row.get('final_recommendation_score')):.3f}")
        return " | ".join(parts)

    def risk_text(row):
        risk = float(row.get("false_positive_risk", 0.0) or 0.0)
        risk_medium = float(row.get("risk_medium_threshold", 0.40) or 0.40)
        risk_high = float(row.get("risk_high_threshold", max(risk_medium, 0.60)) or max(risk_medium, 0.60))
        reasons = []
        if float(row.get("negative_affinity_score", 0.0) or 0.0) >= 0.60:
            reasons.append("afinidad negativa relevante")
        if float(row.get("preference_margin_score", 0.0) or 0.0) < 0.45:
            reasons.append("margen positivo-negativo debil")
        if float(row.get("generic_semantic_penalty", 0.0) or 0.0) >= 0.60:
            reasons.append("terminos poco discriminativos")
        if float(row.get("popularity_without_anchor_penalty", 0.0) or 0.0) >= 0.30:
            reasons.append("dependencia de calidad/popularidad con ancla debil")
        if float(row.get("temporal_mismatch_penalty", 0.0) or 0.0) >= 0.55:
            reasons.append("mismatch temporal")
        if float(row.get("rerank_jump_penalty", 0.0) or 0.0) >= 0.35:
            reasons.append("salto grande desde el ranking base")
        label = "bajo" if risk < risk_medium else ("medio" if risk < risk_high else "alto")
        detail = "; ".join(reasons) if reasons else "sin alertas fuertes"
        return f"riesgo {label}: {detail}"

    scored["why_this_matches"] = scored.apply(why_matches, axis=1)
    scored["risk_explanation"] = scored.apply(risk_text, axis=1)
    scored["explanation_display"] = scored.apply(build_explanation_display_from_row, axis=1)
    scored.attrs["human_like_score_inputs"] = {
        "ranking_score": "final_recommendation_score",
        "auxiliary_rank_score": "human_like_rank_score",
        "normalization": "robust_minmax_over_candidate_pool_for_final_recommendation_score; human_like_score only visual/audit",
        "temporal": "reuse year_affinity_score/temporal_mismatch_penalty when available; reconstruct from Trakt profile when missing",
        "specificity": "tag frequencies, personal lift and negative lift from tags_clean/candidate terms",
    }
    return scored


def select_human_like_diverse_recommendations(
    df,
    top_n=20,
    max_per_branch=6,
    max_per_main_genre=8,
    max_risk_controlled=3,
    max_classic_pending=None,
    score_col="final_recommendation_score",
    candidate_pool_size=500,
):
    work = df.sort_values(score_col, ascending=False).head(candidate_pool_size).reset_index(drop=True).copy()
    if "main_genre" not in work.columns:
        work["main_genre"] = work["genres"].apply(main_genre_from_genres)
    if "recommendation_branch" not in work.columns:
        work["recommendation_branch"] = work.apply(infer_recommendation_branch, axis=1)
    if "previous_hybrid_rank" not in work.columns:
        work["previous_hybrid_rank"] = work["hybrid_score"].rank(method="min", ascending=False).astype(int) if "hybrid_score" in work.columns else np.nan
    if "previous_semantic_rank" not in work.columns:
        work["previous_semantic_rank"] = work["semantic_relevance_adjusted_score"].rank(method="min", ascending=False).astype(int) if "semantic_relevance_adjusted_score" in work.columns else np.nan
    work["pre_rerank_rank"] = np.arange(1, len(work) + 1)
    if "decade" not in work.columns:
        work["decade"] = ((pd.to_numeric(work.get("year"), errors="coerce") // 10) * 10).astype("Int64")
    excluded_by_risk = (work["false_positive_risk"] >= 0.70) & ~((work["anchor_match_score"] >= 0.90) & (work["anchor_specificity_score"] >= 0.70))

    selected_indices = []
    branch_counts = {}
    genre_counts = {}
    decade_counts = {}
    bucket_counts = {}
    reasons = {}
    profile_temporal_entropy = float(work.attrs.get("positive_temporal_entropy", np.nan)) if hasattr(work, "attrs") else np.nan
    if max_classic_pending is None:
        max_classic_pending = 4 + int(max(0.0, profile_temporal_entropy if pd.notna(profile_temporal_entropy) else 0.0) * 3)

    def temporal_portfolio_penalty(row, selected_count=None):
        selected_count = len(selected_indices) if selected_count is None else selected_count
        decade_value = row.get("decade", pd.NA)
        if pd.isna(decade_value) or selected_count < 1:
            return 0.0
        next_share = (decade_counts.get(int(decade_value), 0) + 1) / max(selected_count + 1, 1)
        expected = float(row.get("expected_decade_share", 0.0) or 0.0)
        margin = float(row.get("temporal_exploration_margin", 0.10) or 0.10)
        over = max(0.0, next_share - min(1.0, expected + margin))
        evidence_relief = 0.45 * float(row.get("preference_margin_score", 0.0) or 0.0) + 0.35 * float(row.get("base_model_evidence_score", 0.0) or 0.0) + 0.20 * float(row.get("anchor_coherence_score", 0.0) or 0.0)
        return float(over * (1.0 - min(evidence_relief, 0.85)))

    def selection_score(row):
        return float(row.get(score_col, 0.0) or 0.0) - 0.08 * temporal_portfolio_penalty(row)

    def can_add(row, relaxed=False):
        if excluded_by_risk.loc[row.name] and not relaxed:
            return False
        branch = row.get("recommendation_branch", "general_quality_match")
        genre = row.get("main_genre", "Unknown")
        bucket = row.get("recommendation_bucket", "")
        if branch_counts.get(branch, 0) >= max_per_branch + (1 if relaxed else 0):
            return False
        if genre_counts.get(genre, 0) >= max_per_main_genre + (1 if relaxed else 0):
            return False
        if bucket == "riesgo_controlado" and bucket_counts.get(bucket, 0) >= max_risk_controlled + (1 if relaxed else 0):
            return False
        if bucket == "clasico_pendiente" and bucket_counts.get(bucket, 0) >= max_classic_pending + (1 if relaxed else 0):
            return False
        return True

    def add(idx, row, reason):
        selected_indices.append(idx)
        reasons[idx] = reason
        branch = row.get("recommendation_branch", "general_quality_match")
        genre = row.get("main_genre", "Unknown")
        decade_value = row.get("decade", pd.NA)
        bucket = row.get("recommendation_bucket", "")
        branch_counts[branch] = branch_counts.get(branch, 0) + 1
        genre_counts[genre] = genre_counts.get(genre, 0) + 1
        if pd.notna(decade_value):
            decade_counts[int(decade_value)] = decade_counts.get(int(decade_value), 0) + 1
        bucket_counts[bucket] = bucket_counts.get(bucket, 0) + 1

    while len(selected_indices) < top_n:
        eligible = [(idx, row) for idx, row in work.iterrows() if idx not in selected_indices and can_add(row, relaxed=False)]
        if not eligible:
            break
        idx, row = max(eligible, key=lambda item: selection_score(item[1]))
        add(idx, row, "selected_final_score_dynamic_temporal")
    if len(selected_indices) < top_n:
        while len(selected_indices) < top_n:
            eligible = [(idx, row) for idx, row in work.iterrows() if idx not in selected_indices and can_add(row, relaxed=True)]
            if not eligible:
                break
            idx, row = max(eligible, key=lambda item: selection_score(item[1]))
            add(idx, row, "selected_relaxed_final_score")
    if len(selected_indices) < top_n:
        for idx, row in work.iterrows():
            if idx in selected_indices:
                continue
            add(idx, row, "fallback_fill_human_like")
            if len(selected_indices) >= top_n:
                break

    selected = work.loc[selected_indices].copy()
    selected_decade_counts = selected["decade"].value_counts(normalize=True, dropna=True).to_dict() if "decade" in selected.columns else {}
    selected["actual_top_decade_share"] = selected["decade"].apply(lambda value: float(selected_decade_counts.get(value, 0.0)) if pd.notna(value) else 0.0)
    selected["temporal_overrepresentation_penalty"] = (selected["actual_top_decade_share"] - selected["expected_decade_share"] - selected["temporal_exploration_margin"]).clip(lower=0.0)
    selected["temporal_portfolio_penalty"] = selected["temporal_overrepresentation_penalty"] * (1.0 - (0.45 * selected["preference_margin_score"] + 0.35 * selected["base_model_evidence_score"] + 0.20 * selected["anchor_coherence_score"]).clip(upper=0.85))
    selected = selected.sort_values("final_recommendation_score", ascending=False).reset_index(drop=True)
    selected["current_final_rank"] = np.arange(1, len(selected) + 1)
    if len(selected) > 1:
        selected["final_recommendation_score"] = 0.60 + 0.39 * (1.0 - ((selected["current_final_rank"] - 1) / (len(selected) - 1)))
    elif len(selected) == 1:
        selected["final_recommendation_score"] = 0.99
    selected["final_recommendation_score"] = selected["final_recommendation_score"].clip(0.0, 0.99)
    selected["recommendation_branch"] = selected.apply(infer_recommendation_branch, axis=1)
    selected["recommendation_bucket"] = selected.apply(_recommendation_bucket, axis=1)
    if len(selected):
        bucket_before_output_fix = selected["recommendation_bucket"].value_counts().to_dict()
        dominant_before_output_fix = selected["dominant_signal"].value_counts().to_dict() if "dominant_signal" in selected.columns else {}
        score_q70 = selected["final_recommendation_score"].quantile(0.70)
        score_q55 = selected["final_recommendation_score"].quantile(0.55)
        risk_q45 = selected["false_positive_risk"].quantile(0.45)
        risk_q75 = selected["false_positive_risk"].quantile(0.75)
        risk_q80 = selected["false_positive_risk"].quantile(0.80)
        margin_q60 = selected["preference_margin_score"].quantile(0.60)
        margin_q25 = selected["preference_margin_score"].quantile(0.25)
        margin_q50 = selected["preference_margin_score"].quantile(0.50)
        temporal_q70 = selected["temporal_mismatch_penalty"].quantile(0.70)
        temporal_q85 = selected["temporal_mismatch_penalty"].quantile(0.85)
        jump_q75 = selected["rerank_jump_penalty"].quantile(0.75)
        jump_q90 = selected["rerank_jump_penalty"].quantile(0.90)
        negative_q75 = selected["negative_affinity_score"].quantile(0.75)
        negative_q80 = selected["negative_affinity_score"].quantile(0.80)
        explanation_q35 = selected["explanation_quality_score"].quantile(0.35) if "explanation_quality_score" in selected.columns else 0.0
        explanation_q25 = selected["explanation_quality_score"].quantile(0.25) if "explanation_quality_score" in selected.columns else 0.0
        strong_anchor_mask = (selected["anchor_confidence"] == "alto") & (selected.get("num_coherent_anchors", pd.Series(0, index=selected.index)) >= 2) & (selected.get("weak_anchor_penalty", pd.Series(1.0, index=selected.index)) <= selected.get("weak_anchor_penalty", pd.Series(1.0, index=selected.index)).quantile(0.45))
        risk_threshold = selected.get("risk_medium_threshold", pd.Series(np.nan, index=selected.index))
        risk_threshold_valid = pd.to_numeric(risk_threshold, errors="coerce").notna() & (pd.to_numeric(risk_threshold, errors="coerce") > 0)
        above_dynamic_risk = risk_threshold_valid & (selected["false_positive_risk"] >= pd.to_numeric(risk_threshold, errors="coerce"))
        high_risk_with_support = (selected["false_positive_risk"] >= risk_q80) & ((selected["preference_margin_score"] <= margin_q25) | (selected["negative_affinity_score"] >= negative_q80) | (selected["anchor_confidence"] == "bajo") | (selected.get("explanation_quality_score", pd.Series(1.0, index=selected.index)) <= explanation_q25))
        rerank_risk = (selected["rerank_jump_penalty"] >= jump_q90) & (selected["base_model_evidence_score"] <= selected["base_model_evidence_score"].quantile(0.35))
        temporal_risk = (selected["temporal_mismatch_penalty"] >= temporal_q85) & (~selected.get("allow_temporal_outlier", pd.Series(False, index=selected.index)).astype(bool)) & (selected["preference_margin_score"] <= margin_q50)
        risk_mask = above_dynamic_risk | high_risk_with_support | rerank_risk | temporal_risk
        classic_mask = (selected["temporal_distance_from_profile"] >= selected["classic_distance_threshold"]) & ((selected["rating_score"] >= selected["classic_rating_threshold"]) | (selected["popularity_score"] >= selected["classic_popularity_threshold"]))
        safe_mask = (selected["final_recommendation_score"] >= score_q70) & (selected["false_positive_risk"] <= risk_q45) & (selected["preference_margin_score"] >= margin_q60) & (selected["base_model_evidence_score"] >= selected["base_model_evidence_score"].quantile(0.40)) & (selected["temporal_mismatch_penalty"] <= temporal_q70) & (selected["rerank_jump_penalty"] <= jump_q75) & (selected.get("explanation_quality_score", pd.Series(1.0, index=selected.index)) >= explanation_q35)
        selected.loc[:, "recommendation_bucket"] = "descubrimiento_compatible"
        selected.loc[safe_mask, "recommendation_bucket"] = "apuesta_segura"
        selected.loc[strong_anchor_mask & (selected["false_positive_risk"] <= risk_q75), "recommendation_bucket"] = "muy_parecida_a_favoritas"
        selected.loc[classic_mask & ~risk_mask, "recommendation_bucket"] = "clasico_pendiente"
        selected.loc[risk_mask, "recommendation_bucket"] = "riesgo_controlado"
        safe_indices = selected.index[selected["recommendation_bucket"] == "apuesta_segura"].tolist()
        max_safe = int(np.ceil(len(selected) * 0.60))
        if len(safe_indices) > max_safe:
            demote = selected.loc[safe_indices].sort_values("final_recommendation_score", ascending=False).index[max_safe:]
            selected.loc[demote, "recommendation_bucket"] = "descubrimiento_compatible"
        signal_frame = pd.DataFrame({
            "anchor": selected["anchor_match_score"].where(strong_anchor_mask, -np.inf),
            "margin": selected["preference_margin_score"],
            "semantic": selected["semantic_relevance_adjusted_score"],
            "collaborative": selected["item_item_collab_score"],
            "quality": 0.55 * selected["rating_score"] + 0.45 * selected["base_model_evidence_score"],
            "popularity": selected["popularity_score"],
            "temporal": selected["user_temporal_affinity_score"] - selected["temporal_mismatch_penalty"],
        }, index=selected.index)
        selected["dominant_signal"] = signal_frame.idxmax(axis=1)
        selected.loc[(selected["recommendation_bucket"] == "riesgo_controlado") & (above_dynamic_risk | (selected["false_positive_risk"] >= risk_q80) | (selected["negative_affinity_score"] >= negative_q80)), "dominant_signal"] = "risk_penalized"
        selected.loc[(selected["rerank_jump_penalty"] >= jump_q90) & (selected["recommendation_bucket"] == "riesgo_controlado"), "dominant_signal"] = "rerank_limited"
        selected.loc[(selected["recommendation_bucket"] == "clasico_pendiente") & (selected["rating_score"] >= selected["classic_rating_threshold"]), "dominant_signal"] = "quality"
        selected.loc[(selected["recommendation_bucket"] == "clasico_pendiente") & (selected["temporal_mismatch_penalty"] >= selected["temporal_mismatch_medium_threshold"]), "dominant_signal"] = "temporal"
        selected.attrs["output_fix_summary"] = {
            "bucket_before": bucket_before_output_fix,
            "bucket_after": selected["recommendation_bucket"].value_counts().to_dict(),
            "dominant_before": dominant_before_output_fix,
            "dominant_after": selected["dominant_signal"].value_counts().to_dict(),
        }
    selected["explanation_display"] = selected.apply(build_explanation_display_from_row, axis=1)
    selected["final_rank"] = np.arange(1, len(selected) + 1)
    selected["rank"] = selected["final_rank"]
    selected["selected_by_rerank"] = True
    selected_source_index = selected["selection_source_index"] if "selection_source_index" in selected.columns else pd.Series(selected.index, index=selected.index)
    selected["rerank_reason"] = selected_source_index.map(reasons).fillna("fallback_fill_human_like")
    selected["rerank_selection_score"] = selected[score_col]
    selected["was_in_pre_rerank_top20"] = selected.get("previous_hybrid_rank", pd.Series(np.inf, index=selected.index)) <= top_n
    selected["entered_by_rerank"] = ~selected["was_in_pre_rerank_top20"]
    selected["human_like_rank_delta"] = selected["previous_hybrid_rank"] - selected["final_rank"] if "previous_hybrid_rank" in selected.columns else np.nan
    output_fix_summary = selected.attrs.get("output_fix_summary", {})
    rerank_summary_payload = {
        "score_floor": float(work.head(top_n)[score_col].min()) if len(work) else np.nan,
        "score_floor_ratio": 1.0,
        "pre_rerank_top_n_min_score": float(work.head(top_n)[score_col].min()) if len(work) else np.nan,
        "pre_rerank_top_n_avg_score": float(work.head(top_n)[score_col].mean()) if len(work) else np.nan,
        "candidate_pool_q75_score": float(work[score_col].quantile(0.75)) if len(work) else np.nan,
        "excluded_by_high_risk": int(excluded_by_risk.sum()),
        "max_per_branch": max_per_branch,
        "max_per_main_genre": max_per_main_genre,
        "max_risk_controlled": max_risk_controlled,
        "max_classic_pending": max_classic_pending,
        "positive_decade_share": work.attrs.get("positive_decade_share", {}) if hasattr(work, "attrs") else {},
        "selected_decade_share": selected_decade_counts,
        "temporal_portfolio_penalty_mean": float(selected["temporal_portfolio_penalty"].mean()) if "temporal_portfolio_penalty" in selected.columns and len(selected) else np.nan,
        "output_fix_bucket_before": output_fix_summary.get("bucket_before", {}),
        "output_fix_bucket_after": output_fix_summary.get("bucket_after", {}),
        "output_fix_dominant_before": output_fix_summary.get("dominant_before", {}),
        "output_fix_dominant_after": output_fix_summary.get("dominant_after", {}),
    }
    selected.attrs["rerank_summary"] = rerank_summary_payload
    selected = selected.reset_index(drop=True)
    selected.attrs["rerank_summary"] = rerank_summary_payload
    selected.attrs["output_fix_summary"] = output_fix_summary
    return selected

if "main_genre" not in candidates_scored.columns:
    candidates_scored["main_genre"] = candidates_scored["genres"].apply(main_genre_from_genres)
candidates_scored["semantic_branch"] = candidates_scored.apply(assign_semantic_branch, axis=1)

user_branch_profile = build_user_branch_profile(
    liked_movies=liked_movies,
    tags_clean=tags_clean,
    discriminative_tag_profile=discriminative_tag_profile,
    assign_branch_func=assign_semantic_branch,
)
display(user_branch_profile.sort_values("branch_share", ascending=False))
if EXPORT_LEGACY_EXPORTS:
    user_branch_profile.to_csv(REPORTS_RESULTADOS / "user_branch_profile.csv", index=False)

candidates_scored = add_branch_affinity_to_candidates(
    candidates_scored,
    user_branch_profile,
    branch_col="semantic_branch",
)

pre_semantic_adjustment_top20 = candidates_scored.sort_values("hybrid_score", ascending=False).head(20).copy() if "hybrid_score" in candidates_scored.columns else pd.DataFrame()
pre_semantic_adjustment_titles = set(pre_semantic_adjustment_top20["title"].astype(str)) if not pre_semantic_adjustment_top20.empty else set()
candidates_scored = compute_semantic_relevance_scores(candidates_scored)
candidates_scored = compute_hybrid_score(candidates_scored, weights)
candidates_scored = add_human_like_recommendation_layer(
    candidates_scored,
    tags_clean=tags_clean if "tags_clean" in globals() else None,
    liked_movies=liked_movies if "liked_movies" in globals() else None,
    disliked_movies=disliked_movies if "disliked_movies" in globals() else None,
)
candidates_scored["previous_hybrid_rank"] = candidates_scored["hybrid_score"].rank(method="min", ascending=False).astype(int)
if "semantic_relevance_adjusted_score" in candidates_scored.columns:
    candidates_scored["previous_semantic_rank"] = candidates_scored["semantic_relevance_adjusted_score"].rank(method="min", ascending=False).astype(int)
candidates_scored["human_like_candidate_rank"] = candidates_scored["final_recommendation_score"].rank(method="min", ascending=False).astype(int)
post_semantic_adjustment_top20 = candidates_scored.sort_values("hybrid_score", ascending=False).head(20).copy()
post_semantic_adjustment_titles = set(post_semantic_adjustment_top20["title"].astype(str))
overlap_top20_before_after_semantic_adjustment = (
    len(pre_semantic_adjustment_titles & post_semantic_adjustment_titles) / 20
    if pre_semantic_adjustment_titles else np.nan
)
print("Top 20 por hybrid_score antes de recalcular con semantic_relevance_adjusted_score:")
if not pre_semantic_adjustment_top20.empty:
    display(pre_semantic_adjustment_top20[["title", "year", "genres", "hybrid_score"]])
print("Top 20 por hybrid_score despues de recalcular con semantic_relevance_adjusted_score:")
display(post_semantic_adjustment_top20[["title", "year", "genres", "hybrid_score", "semantic_relevance_adjusted_score", "dominant_signal"]])
print("Overlap top20 antes/despues de semantic_relevance_adjusted_score:", overlap_top20_before_after_semantic_adjustment)
if pd.notna(overlap_top20_before_after_semantic_adjustment) and overlap_top20_before_after_semantic_adjustment > 0.85:
    warnings.warn("El nuevo semantic_relevance_adjusted_score apenas modifica el ranking base.")

branch_assignment_cols = [
    "movieId", "title", "year", "genres", "semantic_branch", "branch_strength", "branch_share",
    "latent_core_preference_score", "semantic_net_score", "semantic_relevance_score",
    "semantic_relevance_adjusted_score", "hybrid_score", "core_semantic_explanation_terms",
    "semantic_explanation_terms", "nearest_core_anchor_movies"
]
branch_assignment_diagnostics = candidates_scored.sort_values("hybrid_score", ascending=False).head(150).copy()
branch_assignment_diagnostics["branch_assignment_warning"] = branch_assignment_diagnostics.apply(_branch_warning_for_row, axis=1)
branch_assignment_available_cols = [col for col in branch_assignment_cols + ["branch_assignment_warning"] if col in branch_assignment_diagnostics.columns]
display(branch_assignment_diagnostics[branch_assignment_available_cols])
if EXPORT_LEGACY_EXPORTS:
    branch_assignment_diagnostics[branch_assignment_available_cols].to_csv(REPORTS_RESULTADOS / "branch_assignment_diagnostics.csv", index=False)
branch_warning_counts = branch_assignment_diagnostics["branch_assignment_warning"].value_counts()
if any(idx for idx in branch_warning_counts.index if idx):
    warnings.warn(f"Avisos de asignacion de ramas: {branch_warning_counts.to_dict()}")

print("Distribucion de semantic_branch en candidatos:")
display(candidates_scored["semantic_branch"].value_counts(dropna=False))

diversity_metrics = analyze_user_diversity_profile(
    liked_movies=liked_movies,
    tags_clean=tags_clean,
    discriminative_tag_profile=discriminative_tag_profile,
    temporal_metrics=temporal_metrics,
    user_branch_profile=user_branch_profile,
)
diversity_metrics_df = pd.DataFrame(diversity_metrics.items(), columns=["metric", "value"])
display(diversity_metrics_df.sort_values("metric"))
if EXPORT_LEGACY_EXPORTS:
    diversity_metrics_df.to_csv(REPORTS_RESULTADOS / "diversity_profile_metrics.csv", index=False)

adaptive_rerank_config = derive_adaptive_rerank_config(diversity_metrics)
adaptive_rerank_config_df = pd.DataFrame(adaptive_rerank_config.items(), columns=["parameter", "value"])
display(adaptive_rerank_config_df)
if EXPORT_LEGACY_EXPORTS:
    adaptive_rerank_config_df.to_csv(REPORTS_RESULTADOS / "adaptive_rerank_config.csv", index=False)

top_old_by_hybrid = candidates_scored.sort_values("hybrid_score", ascending=False).head(20).copy()
top_old_by_semantic = candidates_scored.sort_values("semantic_relevance_adjusted_score", ascending=False).head(20).copy() if "semantic_relevance_adjusted_score" in candidates_scored.columns else pd.DataFrame()
top_new_by_human_like = candidates_scored.sort_values("final_recommendation_score", ascending=False).head(20).copy()
pre_rerank_top = top_old_by_hybrid.copy()
pre_rerank_titles = set(pre_rerank_top["title"].astype(str))
new_human_like_titles = set(top_new_by_human_like["title"].astype(str))
semantic_top_titles = set(top_old_by_semantic["title"].astype(str)) if not top_old_by_semantic.empty else set()
human_like_ranking_overlap_top20 = len(pre_rerank_titles & new_human_like_titles) / 20 if pre_rerank_titles else np.nan
semantic_human_like_overlap_top20 = len(semantic_top_titles & new_human_like_titles) / 20 if semantic_top_titles else np.nan
human_like_entering_titles = sorted(new_human_like_titles - pre_rerank_titles)
human_like_leaving_titles = sorted(pre_rerank_titles - new_human_like_titles)
print("Comparacion ranking antiguo por hybrid_score vs nuevo por final_recommendation_score:")
print(f"Overlap hybrid vs nuevo top20: {human_like_ranking_overlap_top20:.3f}")
print(f"Overlap semantic vs nuevo top20: {semantic_human_like_overlap_top20:.3f}" if pd.notna(semantic_human_like_overlap_top20) else "Overlap semantic vs nuevo top20: no disponible")
print("Entran:", human_like_entering_titles)
print("Salen:", human_like_leaving_titles)
print("Distribucion recommendation_branch en top nuevo:")
display(top_new_by_human_like["recommendation_branch"].value_counts(dropna=False))
print("Distribucion recommendation_bucket en top nuevo:")
display(top_new_by_human_like["recommendation_bucket"].value_counts(dropna=False))
print("Medias ranking final visible:")
display(top_new_by_human_like[["anchor_match_score", "anchor_coherence_score", "anchor_specificity_score", "preference_margin_score", "user_temporal_affinity_score", "temporal_mismatch_penalty", "false_positive_risk", "human_like_v3_adjusted_score", "final_recommendation_score"]].agg(["mean", "min", "max"]).T)
print("Top nuevo con mayor riesgo:")
display(top_new_by_human_like.sort_values("false_positive_risk", ascending=False)[["title", "false_positive_risk", "negative_affinity_score", "preference_margin_score", "recommendation_bucket"]].head(10))
print("Top nuevo con mejor margen positivo-negativo:")
display(top_new_by_human_like.sort_values("preference_margin_score", ascending=False)[["title", "preference_margin_score", "positive_affinity_score", "negative_affinity_score", "anchor_movies_matched"]].head(10))
print("Top nuevo con anclas mas fuertes:")
display(top_new_by_human_like.sort_values("anchor_match_score", ascending=False)[["title", "anchor_match_score", "anchor_confidence", "anchor_movies_matched"]].head(10))

weight_configs_for_comparison = dict(WEIGHT_CONFIGS)
weight_configs_for_comparison["adaptive"] = adaptive_weights
display(compare_weight_configs(candidates_scored_base, weight_configs_for_comparison, top_n=20))

recommendations = select_human_like_diverse_recommendations(
    candidates_scored,
    top_n=20,
    max_per_branch=6,
    max_per_main_genre=8,
    max_risk_controlled=3,
    max_classic_pending=None,
    score_col="final_recommendation_score",
)
rerank_summary = recommendations.attrs.get("rerank_summary", {})
recommendations = recommendations.sort_values("final_rank").reset_index(drop=True)
recommendations["frozen_rank"] = np.arange(1, len(recommendations) + 1)
frozen_top_control = recommendations[[col for col in ["title", "movieId", "frozen_rank", "final_recommendation_score", "human_like_v3_adjusted_score"] if col in recommendations.columns]].copy()
frozen_title_order = frozen_top_control["title"].astype(str).tolist() if "title" in frozen_top_control.columns else []


def _num_series_for_labels(df, column, default=0.0):
    if column in df.columns:
        return pd.to_numeric(df[column], errors="coerce").fillna(default).astype(float)
    return pd.Series(default, index=df.index, dtype=float)


def _bool_series_for_labels(df, column, default=False):
    if column in df.columns:
        return df[column].fillna(default).astype(bool)
    return pd.Series(default, index=df.index, dtype=bool)


def _has_real_variance(series, eps=1e-9):
    values = pd.to_numeric(series, errors="coerce").dropna()
    return bool(values.nunique() > 1 and (values.max() - values.min()) > eps)


def _q_for_labels(series, q, default=0.0):
    values = pd.to_numeric(series, errors="coerce").dropna()
    if values.empty:
        return float(default)
    return float(values.quantile(q))


def _clean_anchor_names_for_labels(value, max_items=2):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    raw = str(value).replace(";", "|").replace(",", "|")
    anchors = []
    for item in raw.split("|"):
        item = " ".join(str(item).split()).strip(" -")
        if item and item.lower() not in {"nan", "none"} and item not in anchors:
            anchors.append(item)
    return " y ".join(anchors[:max_items])


def apply_final_interpretive_labels(df):
    work = df.copy()
    score = _num_series_for_labels(work, "final_recommendation_score")
    risk = _num_series_for_labels(work, "false_positive_risk")
    margin = _num_series_for_labels(work, "preference_margin_score")
    negative = _num_series_for_labels(work, "negative_affinity_score")
    anchor = _num_series_for_labels(work, "anchor_match_score")
    weak_anchor = _num_series_for_labels(work, "weak_anchor_penalty")
    coherent_anchors = _num_series_for_labels(work, "num_coherent_anchors")
    base = _num_series_for_labels(work, "base_model_evidence_score")
    rating = _num_series_for_labels(work, "rating_score")
    popularity = _num_series_for_labels(work, "popularity_score")
    semantic = _num_series_for_labels(work, "semantic_relevance_adjusted_score")
    collab = _num_series_for_labels(work, "item_item_collab_score")
    temporal_affinity = _num_series_for_labels(work, "user_temporal_affinity_score")
    temporal_distance = _num_series_for_labels(work, "temporal_distance_from_profile")
    temporal_mismatch = _num_series_for_labels(work, "temporal_mismatch_penalty")
    rerank_jump = _num_series_for_labels(work, "rerank_jump_penalty")
    explanation_quality = _num_series_for_labels(work, "explanation_quality_score", default=1.0)
    anchor_confidence = work.get("anchor_confidence", pd.Series("", index=work.index)).fillna("").astype(str).str.lower()
    previous_dominant_signal = work.get("dominant_signal", pd.Series("margin", index=work.index)).fillna("margin").astype(str)

    score_high = _q_for_labels(score, 0.70)
    score_very_high = _q_for_labels(score, 0.85)
    risk_low = _q_for_labels(risk, 0.40)
    risk_top = _q_for_labels(risk, 0.85)
    if "risk_medium_threshold" in work.columns:
        medium_values = pd.to_numeric(work["risk_medium_threshold"], errors="coerce")
        valid_medium = medium_values[(medium_values > 0) & medium_values.notna()]
        risk_medium = float(valid_medium.median()) if not valid_medium.empty else _q_for_labels(risk, 0.80)
    else:
        risk_medium = _q_for_labels(risk, 0.80)
    margin_high = _q_for_labels(margin, 0.65)
    margin_low = _q_for_labels(margin, 0.35)
    margin_reasonable = _q_for_labels(margin, 0.45)
    negative_high = _q_for_labels(negative, 0.70)
    anchor_high = _q_for_labels(anchor, 0.70)
    anchor_low = _q_for_labels(anchor, 0.35)
    weak_anchor_low = _q_for_labels(weak_anchor, 0.40)
    quality = pd.concat([rating, base], axis=1).max(axis=1)
    quality_high = _q_for_labels(quality, 0.70)
    quality_reasonable = _q_for_labels(base, 0.45)
    temporal_distance_var = _has_real_variance(temporal_distance)
    temporal_mismatch_var = _has_real_variance(temporal_mismatch)
    temporal_affinity_var = _has_real_variance(temporal_affinity)
    rerank_var = _has_real_variance(rerank_jump)
    temporal_distance_high = _q_for_labels(temporal_distance, 0.75) if temporal_distance_var else np.inf
    temporal_mismatch_high = _q_for_labels(temporal_mismatch, 0.80) if temporal_mismatch_var else np.inf
    rerank_high = _q_for_labels(rerank_jump, 0.80) if rerank_var else np.inf
    explanation_low = _q_for_labels(explanation_quality, 0.35)

    low_margin = margin <= margin_low
    high_negative = negative >= negative_high
    high_temporal_mismatch = temporal_mismatch_var & (temporal_mismatch >= temporal_mismatch_high) & (temporal_mismatch > 0)
    high_rerank_jump = rerank_var & (rerank_jump >= rerank_high) & (rerank_jump > 0)
    low_anchor_confidence = anchor_confidence.eq("bajo")
    low_explanation = explanation_quality <= explanation_low
    weak_anchor_signal = (anchor <= anchor_low) | (weak_anchor > weak_anchor_low)
    real_uncertainty = low_margin | high_negative | high_temporal_mismatch | high_rerank_jump | low_anchor_confidence | low_explanation
    real_risk = (risk >= risk_medium) & real_uncertainty
    top_risk_with_weak_support = (risk >= max(risk_medium, risk_top)) & (low_margin | weak_anchor_signal | low_anchor_confidence)
    risk_mask = real_risk | top_risk_with_weak_support

    similar_mask = (
        anchor_confidence.eq("alto")
        & (anchor >= anchor_high)
        & (risk <= risk_medium)
        & (("weak_anchor_penalty" not in work.columns) | (weak_anchor <= weak_anchor_low))
        & (("num_coherent_anchors" not in work.columns) | (coherent_anchors >= 2))
        & (margin >= margin_reasonable)
    )
    classic_mask = (
        ~risk_mask
        & ~similar_mask
        & (quality >= quality_high)
        & (
            (temporal_distance_var & (temporal_distance >= temporal_distance_high))
            | (~anchor_confidence.eq("alto") & (quality >= quality_high))
        )
    )
    safe_mask = (
        ~risk_mask
        & ~similar_mask
        & ~classic_mask
        & (score >= score_high)
        & ((risk <= risk_low) | (risk <= risk_medium))
        & (margin >= margin_high)
        & (base >= quality_reasonable)
        & ~high_temporal_mismatch
        & ~high_rerank_jump
    )
    discovery_mask = ~risk_mask & ~similar_mask & ~classic_mask & ~safe_mask

    work["recommendation_bucket"] = "descubrimiento_compatible"
    work.loc[risk_mask, "recommendation_bucket"] = "riesgo_controlado"
    work.loc[similar_mask, "recommendation_bucket"] = "muy_parecida_a_favoritas"
    work.loc[classic_mask, "recommendation_bucket"] = "clasico_pendiente"
    work.loc[safe_mask, "recommendation_bucket"] = "apuesta_segura"
    work.loc[discovery_mask, "recommendation_bucket"] = "descubrimiento_compatible"

    if not (work["recommendation_bucket"] == "apuesta_segura").any():
        safe_candidates = work.index[(~risk_mask) & (score >= score_very_high) & (risk <= risk_low) & (margin >= margin_reasonable)]
        if len(safe_candidates):
            work.loc[safe_candidates[:1], "recommendation_bucket"] = "apuesta_segura"
    if not (work["recommendation_bucket"] == "muy_parecida_a_favoritas").any():
        anchor_candidates = work.index[(~risk_mask) & anchor_confidence.eq("alto") & (anchor >= anchor_high) & (margin >= margin_reasonable)]
        if len(anchor_candidates):
            work.loc[anchor_candidates[:1], "recommendation_bucket"] = "muy_parecida_a_favoritas"

    temporal_high = min(_q_for_labels(temporal_affinity, 0.66), 0.85)
    temporal_medium = min(_q_for_labels(temporal_affinity, 0.33), 0.65)
    work["temporal_affinity_label"] = np.select(
        [temporal_affinity >= temporal_high, temporal_affinity >= temporal_medium],
        ["Alta afinidad temporal", "Afinidad temporal media"],
        default="Baja afinidad temporal",
    )

    signal_frame = pd.DataFrame(index=work.index)
    signal_frame["anchor"] = anchor.where(anchor_confidence.eq("alto") & (anchor >= anchor_high), -np.inf)
    signal_frame["margin"] = margin
    signal_frame["semantic"] = semantic
    signal_frame["collaborative"] = collab
    signal_frame["quality"] = quality
    core_signal_max = signal_frame[["anchor", "margin", "semantic", "collaborative", "quality"]].max(axis=1)
    popularity_high = _q_for_labels(popularity, 0.75)
    signal_frame["popularity"] = popularity.where((popularity >= popularity_high) & (popularity >= core_signal_max + 0.07), -np.inf)
    signal_frame["risk_penalized"] = risk.where((work["recommendation_bucket"] == "riesgo_controlado") & risk_mask & (risk >= risk_medium), -np.inf)
    signal_frame["rerank_limited"] = rerank_jump.where(rerank_var & (rerank_jump >= rerank_high) & (rerank_jump > 0), -np.inf)
    work["dominant_signal"] = signal_frame.idxmax(axis=1).replace({"risk_penalized": "risk_penalized", "rerank_limited": "rerank_limited"})
    work.loc[(work["dominant_signal"] == "risk_penalized") & (work["recommendation_bucket"] != "riesgo_controlado"), "dominant_signal"] = "margin"
    if not rerank_var:
        work.loc[work["dominant_signal"] == "rerank_limited", "dominant_signal"] = "margin"
    work.loc[work["dominant_signal"].eq("temporal"), "dominant_signal"] = "margin"
    classic_mask_final = work["recommendation_bucket"].eq("clasico_pendiente")
    work.loc[classic_mask_final & (quality >= quality_high), "dominant_signal"] = "quality"
    work.loc[classic_mask_final & anchor_confidence.eq("alto") & (anchor >= quality + 0.05), "dominant_signal"] = "anchor"
    work.loc[classic_mask_final & (margin >= margin_high) & (margin >= quality + 0.05), "dominant_signal"] = "margin"
    work.loc[(work["recommendation_bucket"] == "muy_parecida_a_favoritas") & anchor_confidence.eq("alto"), "dominant_signal"] = "anchor"
    safe_mask_final = work["recommendation_bucket"].eq("apuesta_segura")
    work.loc[safe_mask_final & (quality >= margin), "dominant_signal"] = "quality"
    work.loc[safe_mask_final & (margin > quality), "dominant_signal"] = "margin"
    discovery_mask_final = work["recommendation_bucket"].eq("descubrimiento_compatible")
    discovery_frame = signal_frame[["margin", "semantic", "anchor", "collaborative"]].copy()
    work.loc[discovery_mask_final, "dominant_signal"] = discovery_frame.loc[discovery_mask_final].idxmax(axis=1).fillna("margin")
    temporal_source_mask = previous_dominant_signal.eq("temporal")
    work["dominant_signal"] = previous_dominant_signal.where(~temporal_source_mask, work["dominant_signal"])
    work.loc[work["dominant_signal"].isin(["temporal", "", "nan"]), "dominant_signal"] = "margin"

    def _final_explanation(row):
        bucket = str(row.get("recommendation_bucket", "descubrimiento_compatible"))
        signal = str(row.get("dominant_signal", "margin"))
        branch = str(row.get("recommendation_branch", row.get("semantic_branch", "perfil principal"))).replace("_", " ")
        anchors = _clean_anchor_names_for_labels(row.get("anchor_movies_matched", ""))
        if bucket == "apuesta_segura":
            text = "Recomendacion segura: combina buen encaje con tu perfil, bajo riesgo de falso positivo y buena evidencia del modelo base."
        elif bucket == "muy_parecida_a_favoritas":
            text = f"Recomendacion cercana a tus favoritas: comparte senales con {anchors} y mantiene buen margen frente al perfil negativo." if anchors else "Recomendacion cercana a tus favoritas: comparte senales de anclaje fuertes y mantiene buen margen frente al perfil negativo."
        elif bucket == "clasico_pendiente":
            text = "Titulo consolidado compatible: entra por calidad, afinidad razonable con tu perfil y control de riesgo."
        elif bucket == "riesgo_controlado":
            text = "Recomendacion con riesgo controlado: tiene senales positivas, pero tambien algun factor de incertidumbre detectado por el modelo."
        else:
            text = f"Descubrimiento compatible: no es la opcion mas obvia, pero encaja con la rama {branch} y mantiene buen margen positivo frente al perfil negativo."
        details = []
        if bucket not in {"descubrimiento_compatible"} and branch:
            details.append(f"Rama: {branch}.")
        if signal == "margin":
            details.append("Senal principal: margen positivo frente al perfil negativo.")
        elif signal == "anchor" and anchors:
            details.append(f"Senal principal: anclas coherentes ({anchors}).")
        elif signal == "quality":
            details.append("Senal principal: calidad y evidencia base consolidadas.")
        elif signal == "semantic":
            details.append("Senal principal: similitud semantica con tus preferencias.")
        elif signal == "collaborative":
            details.append("Senal principal: afinidad colaborativa item-item.")
        elif signal == "popularity":
            details.append("Senal principal: popularidad compatible con el perfil.")
        elif signal == "rerank_limited":
            details.append("Senal principal: salto de reranking limitado por prudencia.")
        elif signal == "risk_penalized":
            details.append("Senal principal: riesgo relativo revisado de forma explicita.")
        if row.get("temporal_affinity_label", "") == "Alta afinidad temporal":
            details.append("Ademas, mantiene buena afinidad temporal con tu perfil.")
        return " ".join([text] + details).strip()

    work["explanation_display"] = work.apply(_final_explanation, axis=1)
    work.attrs["final_label_thresholds"] = {
        "high_score_threshold": score_high,
        "very_high_score_threshold": score_very_high,
        "low_risk_threshold": risk_low,
        "medium_risk_threshold": risk_medium,
        "high_margin_threshold": margin_high,
        "high_anchor_threshold": anchor_high,
        "high_quality_threshold": quality_high,
        "temporal_mismatch_has_variance": temporal_mismatch_var,
        "rerank_jump_has_variance": rerank_var,
    }
    work.attrs["final_label_real_risk_mask"] = risk_mask.reindex(work.index, fill_value=False).astype(bool)
    return work


dominant_signal_before_interpretive_fix = recommendations["dominant_signal"].value_counts(dropna=False).to_dict() if "dominant_signal" in recommendations.columns else {}
print("Distribucion dominant_signal antes del ajuste interpretativo final:")
display(pd.Series(dominant_signal_before_interpretive_fix, dtype="object"))
recommendations = apply_final_interpretive_labels(recommendations)
recommendations = recommendations.sort_values("frozen_rank").reset_index(drop=True)
post_label_title_order = recommendations["title"].astype(str).tolist() if "title" in recommendations.columns else []
frozen_rank_order_preserved = frozen_title_order == post_label_title_order
if not frozen_rank_order_preserved:
    print("ERROR: el orden del top cambio durante el ajuste interpretativo; se restaura por frozen_rank.")
    recommendations = recommendations.sort_values("frozen_rank").reset_index(drop=True)
    post_label_title_order = recommendations["title"].astype(str).tolist() if "title" in recommendations.columns else []
    frozen_rank_order_preserved = frozen_title_order == post_label_title_order
recommendations.attrs["frozen_top_control"] = frozen_top_control
recommendations.attrs["frozen_rank_order_preserved"] = frozen_rank_order_preserved
final_human_like_recommendations = recommendations.copy()
print(f"Top preservado por frozen_rank: {frozen_rank_order_preserved}")
print("Distribucion final recommendation_bucket tras ajuste interpretativo:")
display(recommendations["recommendation_bucket"].value_counts(dropna=False))
FINAL_DOMINANT_SIGNAL_LABELS = {
    "anchor": "Anclas de favoritas",
    "margin": "Margen positivo",
    "semantic": "Similitud sem\u00e1ntica",
    "collaborative": "Se\u00f1al colaborativa",
    "quality": "Calidad / valoraci\u00f3n",
    "popularity": "Popularidad",
    "risk_penalized": "Riesgo penalizado",
    "rerank_limited": "Reordenaci\u00f3n limitada",
}
recommendations["dominant_signal_label"] = recommendations["dominant_signal"].map(FINAL_DOMINANT_SIGNAL_LABELS).fillna(recommendations["dominant_signal"])
print("Distribucion final dominant_signal tras ajuste interpretativo:")
display(recommendations["dominant_signal"].value_counts(dropna=False))
print("Distribucion final dominant_signal_label tras ajuste interpretativo:")
display(recommendations["dominant_signal_label"].value_counts(dropna=False))
print("Distribucion temporal_affinity_label tras ajuste interpretativo:")
display(recommendations["temporal_affinity_label"].value_counts(dropna=False))
print("Top 20 con senal dominante y afinidad temporal secundaria:")
display(recommendations[[col for col in ["rank", "title", "dominant_signal", "dominant_signal_label", "temporal_affinity_label"] if col in recommendations.columns]].head(20))
if "temporal" not in set(recommendations["dominant_signal"].astype(str)):
    print("Confirmado: dominant_signal temporal no aparece en el top final.")
print("Ranking final ordenado por final_recommendation_score")
print("Top 20 final por final_recommendation_score con diversidad dinamica:")
display(recommendations[[
    "title", "year", "genres", "recommendation_branch", "recommendation_bucket",
    "anchor_match_score", "anchor_specificity_score", "preference_margin_score",
    "false_positive_risk", "final_recommendation_score", "human_like_v3_adjusted_score", "human_like_rank_score", "hybrid_score",
    "final_rank", "rerank_reason", "dominant_signal",
]])

post_rerank_titles = set(recommendations["title"].astype(str))
rerank_entered_titles = sorted(post_rerank_titles - pre_rerank_titles)
rerank_removed_titles = sorted(pre_rerank_titles - post_rerank_titles)
post_decades = ((pd.to_numeric(recommendations["year"], errors="coerce") // 10) * 10).astype("Int64")
unseen_mask = recommendations["branch_strength"] == "unseen"
weak_mask = recommendations["branch_strength"] == "weak"
animation_family_mask = recommendations["semantic_branch"] == "animation_family"
animation_dark_surreal_mask = recommendations["semantic_branch"] == "animation_dark_surreal"
psychological_thriller_mask = recommendations["semantic_branch"] == "psychological_thriller"
psychological_drama_mask = recommendations["semantic_branch"] == "psychological_drama"
exploration_mask = unseen_mask | (recommendations["branch_share"] < 0.03)
unseen_branch_counts = recommendations.loc[unseen_mask, "semantic_branch"].value_counts()
weak_branch_counts = recommendations.loc[weak_mask, "semantic_branch"].value_counts()
unseen_titles = " | ".join(recommendations.loc[unseen_mask, "title"].astype(str))
weak_titles = " | ".join(recommendations.loc[weak_mask, "title"].astype(str))
rerank_diagnostic_rows = [
    {"metric": "diversity_level", "value": diversity_metrics["diversity_level"]},
    {"metric": "overall_user_diversity_score", "value": diversity_metrics["overall_user_diversity_score"]},
    {"metric": "profile_concentration_score", "value": diversity_metrics["profile_concentration_score"]},
    {"metric": "branch_diversity_score", "value": diversity_metrics["branch_diversity_score"]},
    {"metric": "top_branch", "value": diversity_metrics["top_branch"]},
    {"metric": "top_branch_share", "value": diversity_metrics["top_branch_share"]},
    {"metric": "score_floor", "value": rerank_summary.get("score_floor")},
    {"metric": "score_floor_ratio", "value": adaptive_rerank_config["score_floor_ratio"]},
    {"metric": "exploration_budget", "value": rerank_summary.get("exploration_budget")},
    {"metric": "avg_hybrid_score_pre_rerank_top20", "value": pre_rerank_top["hybrid_score"].mean()},
    {"metric": "avg_hybrid_score_post_rerank_top20", "value": recommendations["hybrid_score"].mean()},
    {"metric": "avg_human_like_rank_score_final_top20", "value": recommendations["human_like_rank_score"].mean()},
    {"metric": "mean_anchor_match_score_final_top20", "value": recommendations["anchor_match_score"].mean()},
    {"metric": "mean_anchor_specificity_score_final_top20", "value": recommendations["anchor_specificity_score"].mean()},
    {"metric": "mean_preference_margin_score_final_top20", "value": recommendations["preference_margin_score"].mean()},
    {"metric": "mean_false_positive_risk_final_top20", "value": recommendations["false_positive_risk"].mean()},
    {"metric": "human_like_overlap_top20_vs_hybrid", "value": human_like_ranking_overlap_top20},
    {"metric": "human_like_overlap_top20_vs_semantic", "value": semantic_human_like_overlap_top20},
    {"metric": "min_hybrid_score_pre_rerank_top20", "value": pre_rerank_top["hybrid_score"].min()},
    {"metric": "min_hybrid_score_post_rerank_top20", "value": recommendations["hybrid_score"].min()},
    {"metric": "n_entered_by_rerank", "value": int(recommendations["entered_by_rerank"].sum())},
    {"metric": "n_exploration_branches_top20", "value": int(exploration_mask.sum())},
    {"metric": "n_fallback_fill", "value": int(recommendations["rerank_reason"].astype(str).str.startswith("fallback_fill").sum())},

    {"metric": "n_unseen_branches_top20", "value": int(unseen_branch_counts.size)},
    {"metric": "n_unseen_movies_top20", "value": int(unseen_mask.sum())},
    {"metric": "unseen_branch_distribution_top20", "value": unseen_branch_counts.to_dict()},
    {"metric": "n_weak_movies_top20", "value": int(weak_mask.sum())},
    {"metric": "weak_branch_distribution_top20", "value": weak_branch_counts.to_dict()},
    {"metric": "n_selected_exploration_branch", "value": int((recommendations["rerank_reason"] == "selected_exploration_branch").sum())},
    {"metric": "n_fallback_fill_unseen_branch", "value": int((recommendations["rerank_reason"] == "fallback_fill_unseen_branch").sum())},
    {"metric": "max_unseen_branch_count_top20", "value": int(unseen_branch_counts.max()) if not unseen_branch_counts.empty else 0},
    {"metric": "max_weak_branch_count_top20", "value": int(weak_branch_counts.max()) if not weak_branch_counts.empty else 0},
    {"metric": "titles_unseen_branch_top20", "value": unseen_titles},
    {"metric": "titles_weak_branch_top20", "value": weak_titles},
    {"metric": "titles_entered_by_rerank", "value": "; ".join(rerank_entered_titles)},
    {"metric": "titles_removed_by_rerank", "value": "; ".join(rerank_removed_titles)},
    {"metric": "semantic_branch_distribution_top20", "value": recommendations["semantic_branch"].value_counts().to_dict()},
    {"metric": "recommendation_branch_distribution_top20", "value": recommendations["recommendation_branch"].value_counts().to_dict()},
    {"metric": "recommendation_bucket_distribution_top20", "value": recommendations["recommendation_bucket"].value_counts().to_dict()},
    {"metric": "branch_strength_distribution_top20", "value": recommendations["branch_strength"].value_counts().to_dict()},
    {"metric": "main_genre_distribution_top20", "value": recommendations["main_genre"].value_counts().to_dict()},
    {"metric": "dominant_signal_distribution_top20", "value": recommendations["dominant_signal"].value_counts().to_dict()},
    {"metric": "decade_distribution_top20", "value": post_decades.value_counts().sort_index().to_dict()},
]
rerank_diagnostics_df = pd.DataFrame(rerank_diagnostic_rows)
display(rerank_diagnostics_df)
if EXPORT_LEGACY_EXPORTS:
    rerank_diagnostics_df.to_csv(REPORTS_RESULTADOS / "diagnostico_reranking_diversificado.csv", index=False)

latent_semantic_diagnostics.update({
    "mean_latent_core_similarity_top20": recommendations["latent_core_similarity_score"].mean() if "latent_core_similarity_score" in recommendations.columns else 0.0,
    "mean_semantic_relevance_top20": recommendations["semantic_relevance_score"].mean() if "semantic_relevance_score" in recommendations.columns else 0.0,
    "mean_negative_latent_similarity_top20": recommendations["negative_latent_similarity_score"].mean() if "negative_latent_similarity_score" in recommendations.columns else 0.0,
})
latent_semantic_diagnostics_df = pd.DataFrame(latent_semantic_diagnostics.items(), columns=["metric", "value"])
display(latent_semantic_diagnostics_df)
if EXPORT_LEGACY_EXPORTS:
    latent_semantic_diagnostics_df.to_csv(REPORTS_RESULTADOS / "latent_semantic_diagnostics.csv", index=False)

anchor_branch_distribution = core_positive_anchors["semantic_branch"].value_counts().to_dict() if "core_positive_anchors" in globals() and core_positive_anchors is not None and not core_positive_anchors.empty and "semantic_branch" in core_positive_anchors.columns else {}
top20_branch_distribution = recommendations["semantic_branch"].value_counts().to_dict() if "semantic_branch" in recommendations.columns else {}
top20_titles_by_latent_core_preference = ""
if "latent_core_preference_score" in candidates_scored.columns:
    top20_titles_by_latent_core_preference = "; ".join(candidates_scored.sort_values("latent_core_preference_score", ascending=False).head(20)["title"].astype(str))
liked_available = 0
negative_available = 0
if "movie_id_to_latent_idx" in globals():
    latent_movie_ids = set(movie_id_to_latent_idx.keys())
    liked_available = len(set(liked_movies["movieId"].dropna().astype(int)) & latent_movie_ids) if "movieId" in liked_movies.columns else 0
    negative_seed = disliked_movies[pd.to_numeric(disliked_movies.get("user_rating_5"), errors="coerce") <= 2.5] if disliked_movies is not None and not disliked_movies.empty else pd.DataFrame()
    negative_available = len(set(negative_seed["movieId"].dropna().astype(int)) & latent_movie_ids) if "movieId" in negative_seed.columns else 0

latent_core_preference_diagnostics = {
    "n_liked_movies_available": int(liked_available),
    "n_positive_anchor_evidence": int(len(positive_anchor_evidence)) if "positive_anchor_evidence" in globals() and positive_anchor_evidence is not None else 0,
    "n_core_anchors_selected": int(len(core_positive_anchors)) if "core_positive_anchors" in globals() and core_positive_anchors is not None else 0,
    "n_negative_movies_available": int(negative_available),
    "mean_anchor_weight": float(core_positive_anchors["anchor_weight"].mean()) if "core_positive_anchors" in globals() and core_positive_anchors is not None and not core_positive_anchors.empty and "anchor_weight" in core_positive_anchors.columns else 0.0,
    "mean_centrality_score_anchors": float(core_positive_anchors["centrality_score"].mean()) if "core_positive_anchors" in globals() and core_positive_anchors is not None and not core_positive_anchors.empty and "centrality_score" in core_positive_anchors.columns else 0.0,
    "mean_latent_core_preference_top20": float(recommendations["latent_core_preference_score"].mean()) if "latent_core_preference_score" in recommendations.columns else 0.0,
    "mean_latent_core_similarity_old_top20": float(recommendations["latent_core_similarity_score"].mean()) if "latent_core_similarity_score" in recommendations.columns else 0.0,
    "mean_semantic_net_score_top20": float(recommendations["semantic_net_score"].mean()) if "semantic_net_score" in recommendations.columns else 0.0,
    "mean_semantic_relevance_score_top20": float(recommendations["semantic_relevance_score"].mean()) if "semantic_relevance_score" in recommendations.columns else 0.0,
    "mean_negative_latent_preference_top20": float(recommendations["negative_latent_preference_score"].mean()) if "negative_latent_preference_score" in recommendations.columns else 0.0,
    "top_anchor_branch_distribution": anchor_branch_distribution,
    "top20_branch_distribution": top20_branch_distribution,
    "top20_titles_by_latent_core_preference": top20_titles_by_latent_core_preference,
    "overlap_top20_before_after_latent": latent_ranking_overlap_top20 if "latent_ranking_overlap_top20" in globals() else np.nan,
    "n_titles_changed_after_latent": n_titles_changed_after_latent if "n_titles_changed_after_latent" in globals() else np.nan,
}
latent_core_preference_diagnostics_df = pd.DataFrame(latent_core_preference_diagnostics.items(), columns=["metric", "value"])
display(latent_core_preference_diagnostics_df)
if EXPORT_LEGACY_EXPORTS:
    latent_core_preference_diagnostics_df.to_csv(REPORTS_RESULTADOS / "latent_core_preference_diagnostics.csv", index=False)
if pd.notna(latent_core_preference_diagnostics.get("overlap_top20_before_after_latent")) and latent_core_preference_diagnostics["overlap_top20_before_after_latent"] > 0.85:
    warnings.warn("La senal latent_core_preference_score apenas esta modificando el ranking base.")
if "pre_latent_top20_titles" in globals() and post_rerank_titles == pre_latent_top20_titles:
    warnings.warn("El top final es identico al ranking anterior al cambio latente.")
medium_branch_counts = recommendations.loc[recommendations["branch_strength"] == "medium", "semantic_branch"].value_counts() if "branch_strength" in recommendations.columns else pd.Series(dtype=int)
if not medium_branch_counts.empty and int(medium_branch_counts.max()) > 3:
    warnings.warn("Hay una rama medium con mas de 3 peliculas en el top 20.")
if int(animation_family_mask.sum()) > 3:
    warnings.warn("animation_family supera 3 peliculas en el top 20.")

overlap_top20_pre_rerank_vs_final = len(pre_rerank_titles & post_rerank_titles) / 20 if pre_rerank_titles else np.nan
final_model_stability_diagnostics = {
    "n_candidates": int(len(candidates_scored)),
    "n_recommendations": int(len(recommendations)),
    "overlap_top20_semantic_net_vs_latent_preference": overlap_top20_semantic_net_vs_latent_preference if "overlap_top20_semantic_net_vs_latent_preference" in globals() else np.nan,
    "overlap_top20_before_after_semantic_adjustment": overlap_top20_before_after_semantic_adjustment if "overlap_top20_before_after_semantic_adjustment" in globals() else np.nan,
    "overlap_top20_pre_rerank_vs_final": overlap_top20_pre_rerank_vs_final,
    "mean_hybrid_score_pre_rerank_top20": float(pre_rerank_top["hybrid_score"].mean()),
    "mean_hybrid_score_final_top20": float(recommendations["hybrid_score"].mean()),
    "mean_human_like_rank_score_final_top20": float(recommendations["human_like_rank_score"].mean()),
    "mean_anchor_match_score_final_top20": float(recommendations["anchor_match_score"].mean()),
    "mean_anchor_specificity_score_final_top20": float(recommendations["anchor_specificity_score"].mean()),
    "mean_preference_margin_score_final_top20": float(recommendations["preference_margin_score"].mean()),
    "mean_false_positive_risk_final_top20": float(recommendations["false_positive_risk"].mean()),
    "human_like_rank_score_range_top20": float(recommendations["human_like_rank_score"].max() - recommendations["human_like_rank_score"].min()),
    "human_like_overlap_top20_vs_hybrid": human_like_ranking_overlap_top20,
    "human_like_overlap_top20_vs_semantic": semantic_human_like_overlap_top20,
    "min_hybrid_score_pre_rerank_top20": float(pre_rerank_top["hybrid_score"].min()),
    "min_hybrid_score_final_top20": float(recommendations["hybrid_score"].min()),
    "mean_latent_core_preference_final_top20": float(recommendations["latent_core_preference_score"].mean()) if "latent_core_preference_score" in recommendations.columns else 0.0,
    "mean_semantic_relevance_adjusted_final_top20": float(recommendations["semantic_relevance_adjusted_score"].mean()) if "semantic_relevance_adjusted_score" in recommendations.columns else 0.0,
    "n_animation_family_top20": int(animation_family_mask.sum()),
    "n_animation_dark_surreal_top20": int(animation_dark_surreal_mask.sum()),
    "n_psychological_thriller_top20": int(psychological_thriller_mask.sum()),
    "n_psychological_drama_top20": int(psychological_drama_mask.sum()),
    "n_unseen_branch_top20": int(unseen_mask.sum()),
    "n_fallback_fill": int(recommendations["rerank_reason"].astype(str).str.startswith("fallback_fill").sum()),
    "branch_distribution_top20": recommendations["semantic_branch"].value_counts().to_dict(),
    "recommendation_branch_distribution_top20": recommendations["recommendation_branch"].value_counts().to_dict(),
    "recommendation_bucket_distribution_top20": recommendations["recommendation_bucket"].value_counts().to_dict(),
    "branch_strength_distribution_top20": recommendations["branch_strength"].value_counts().to_dict(),
    "dominant_signal_distribution_top20": recommendations["dominant_signal"].value_counts().to_dict(),
    "titles_entered_by_rerank": "; ".join(rerank_entered_titles),
    "titles_removed_by_rerank": "; ".join(rerank_removed_titles),
}
final_model_stability_diagnostics_df = pd.DataFrame(final_model_stability_diagnostics.items(), columns=["metric", "value"])
display(final_model_stability_diagnostics_df)
if EXPORT_LEGACY_EXPORTS:
    final_model_stability_diagnostics_df.to_csv(REPORTS_RESULTADOS / "final_model_stability_diagnostics.csv", index=False)

print("Media latent_core_similarity_score top 20:", latent_semantic_diagnostics["mean_latent_core_similarity_top20"])
print("Media semantic_net_score top 20:", recommendations["semantic_net_score"].mean() if "semantic_net_score" in recommendations.columns else np.nan)
print("Media semantic_relevance_score top 20:", latent_semantic_diagnostics["mean_semantic_relevance_top20"])

print("Perfil de ramas del usuario:")
display(user_branch_profile.sort_values("branch_share", ascending=False))
print(f"Nivel de diversidad detectado: {diversity_metrics['diversity_level']}")
print(f"Branch diversity score: {diversity_metrics['branch_diversity_score']:.3f}")
print(f"Top branch: {diversity_metrics['top_branch']} ({diversity_metrics['top_branch_share']:.3f})")
print("Configuracion de re-ranking aplicada:")
print(adaptive_rerank_config)
print(f"Peliculas que entraron por rerank: {len(rerank_entered_titles)}")
print(f"Peliculas que salieron por rerank: {len(rerank_removed_titles)}")
print(f"Media hybrid_score antes/despues: {pre_rerank_top['hybrid_score'].mean():.3f} / {recommendations['hybrid_score'].mean():.3f}")
print(f"Exploraciones usadas: {int(exploration_mask.sum())}")
print("Distribucion de semantic_branch antes del rerank:")
display(pre_rerank_top["semantic_branch"].value_counts(dropna=False))
print("Distribucion de semantic_branch despues del rerank:")
display(recommendations["semantic_branch"].value_counts(dropna=False))
print("Distribucion de recommendation_branch despues del rerank:")
display(recommendations["recommendation_branch"].value_counts(dropna=False))
print("Distribucion de recommendation_bucket despues del rerank:")
display(recommendations["recommendation_bucket"].value_counts(dropna=False))
print("Distribucion de branch_strength despues del rerank:")
display(recommendations["branch_strength"].value_counts(dropna=False))
print("Distribucion de main_genre despues del rerank:")
display(recommendations["main_genre"].value_counts(dropna=False))
print("Distribucion de dominant_signal despues del rerank:")
display(recommendations["dominant_signal"].value_counts(dropna=False))
print("Sanity branch_strength top 20:")
display(recommendations["branch_strength"].value_counts(dropna=False))
print("Sanity semantic_branch top 20:")
display(recommendations["semantic_branch"].value_counts(dropna=False))
print(f"Peliculas unseen en top 20: {int(unseen_mask.sum())}")
max_unseen_branch_count = int(unseen_branch_counts.max()) if not unseen_branch_counts.empty else 0
print(f"Max peliculas por rama unseen: {max_unseen_branch_count}")
if max_unseen_branch_count > 1:
    warnings.warn("Hay una rama unseen con mas de una pelicula en el top 20.")
unseen_low_score = recommendations[unseen_mask & (recommendations["final_recommendation_score"] < rerank_summary.get("pre_rerank_top_n_min_score", -np.inf)) & (~recommendations["was_in_pre_rerank_top20"])]
if not unseen_low_score.empty:
    warnings.warn("Hay peliculas unseen fuera del top 20 final original con final_recommendation_score inferior al minimo original.")

final_display_cols = [
    "title", "year", "main_genre", "semantic_branch", "branch_strength",
    "branch_share", "hybrid_score", "base_model_evidence_score", "human_like_raw_score", "human_like_v3_raw_score", "human_like_v3_adjusted_score", "final_recommendation_score", "human_like_rank_score", "human_like_rank_percentile", "human_like_score",
    "positive_affinity_score", "negative_affinity_score", "preference_margin_score",
    "anchor_match_score", "anchor_confidence", "anchor_coherence_score", "weak_anchor_penalty", "num_coherent_anchors", "anchor_specificity_score",
    "generic_semantic_penalty", "explanation_quality_score", "explanation_quality_penalty", "user_temporal_affinity_score", "expected_decade_share", "actual_top_decade_share", "temporal_overrepresentation_penalty", "temporal_portfolio_penalty", "temporal_mismatch_penalty", "rerank_jump_penalty", "false_positive_risk", "risk_medium_threshold",
    "recommendation_branch", "recommendation_bucket", "semantic_relevance_score", "semantic_relevance_adjusted_score",
    "latent_core_preference_score", "latent_core_similarity_score", "semantic_net_score",
    "negative_latent_preference_score", "negative_latent_similarity_score",
    "nearest_core_anchor_movies", "nearest_liked_movies_latent", "pre_rerank_rank", "final_rank",
    "rerank_reason", "dominant_signal",
]
available_final_display_cols = [col for col in final_display_cols if col in recommendations.columns]
display(recommendations[available_final_display_cols])


## 16. Explicaciones

In [ ]:
def _clean_explanation_terms(value):
    if value is None:
        return ""
    if isinstance(value, (list, tuple, set)):
        value = ", ".join(str(item) for item in value if item)
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value).strip()


def build_explanation(row):
    if row.get("explanation_display", ""):
        return row.get("explanation_display", "")
    if row.get("why_this_matches", ""):
        risk = row.get("risk_explanation", "")
        return row.get("why_this_matches", "") + (". " + risk if risk else ".")
    dominant_signal = row.get("dominant_signal", "")
    core_semantic_terms = _clean_explanation_terms(row.get("core_semantic_explanation_terms", ""))
    semantic_terms = core_semantic_terms or _clean_explanation_terms(row.get("semantic_explanation_terms", ""))
    core_negative_terms = _clean_explanation_terms(row.get("core_negative_semantic_terms", ""))
    negative_semantic_terms = core_negative_terms or _clean_explanation_terms(row.get("negative_semantic_terms", ""))
    similar_liked = _clean_explanation_terms(row.get("similar_liked_movies", ""))
    nearest_liked_latent = _clean_explanation_terms(row.get("nearest_liked_movies_latent", ""))
    nearest_core_anchors = _clean_explanation_terms(row.get("nearest_core_anchor_movies", ""))
    semantic_branch = _clean_explanation_terms(row.get("semantic_branch", ""))
    branch_explanation = {
        "psychological_thriller": "encaja con un perfil de peliculas psicologicas o de tension",
        "animation_family": "encaja con una rama de animacion o fantasia presente en tu perfil",
        "documentary_reflective": "encaja con una rama mas documental o reflexiva",
        "crime_thriller": "encaja con peliculas de crimen, tension o investigacion",
        "surreal_fantasy": "encaja con rasgos surrealistas, fantasticos o poco convencionales",
    }

    if dominant_signal == "semantic" and semantic_terms:
        explanation = f"Recomendada porque comparte rasgos semánticos especialmente representativos de tus películas mejor valoradas: {semantic_terms}."
    elif dominant_signal == "genre":
        explanation = "Recomendada porque encaja con los géneros que mejor aparecen en tu perfil."
    elif dominant_signal == "collab" and similar_liked:
        explanation = "Recomendada porque presenta similitud colaborativa con películas que valoraste positivamente: " + similar_liked + "."
    elif dominant_signal == "rating":
        explanation = "Recomendada porque destaca por su valoración media en MovieLens."
    elif dominant_signal == "popularity":
        explanation = "Recomendada porque combina afinidad con una señal alta de popularidad en MovieLens."
    elif semantic_terms:
        explanation = f"Recomendada porque comparte algunos rasgos semánticos con tus películas mejor valoradas: {semantic_terms}."
    else:
        explanation = "Recomendada porque mantiene un equilibrio razonable entre afinidad, calidad y popularidad."

    if semantic_branch in branch_explanation:
        explanation += " También " + branch_explanation[semantic_branch] + "."

    negative_semantic_signal = max(row.get("semantic_core_negative_score", 0), row.get("negative_semantic_score", 0))
    if row.get("rating_score", 0) >= 0.65:
        explanation += " Además, tiene buena valoración media."
    if negative_semantic_signal > 0.35 and negative_semantic_terms:
        explanation += f" Se controla la similitud con rasgos presentes en películas peor valoradas: {negative_semantic_terms}."
    if row.get("item_item_negative_collab_score", 0) > 0.35:
        explanation += " También se aplica una penalización colaborativa por similitud con películas peor valoradas."
    return explanation


recommendations["explanation"] = recommendations.apply(build_explanation, axis=1)

## 17. Resultados

In [ ]:
RESULT_COLUMNS = [
    "title",
    "year",
    "genres",
    "rating_mean",
    "rating_count",
    "genre_profile_score",
    "semantic_raw_score",
    "negative_semantic_raw_score",
    "semantic_profile_score",
    "negative_genre_score",
    "negative_semantic_score",
    "semantic_explanation_terms",
    "negative_semantic_terms",
    "content_profile_score",
    "negative_similarity_score",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "rating_score",
    "popularity_score",
    "contrib_genre",
    "contrib_semantic",
    "contrib_collab",
    "contrib_rating",
    "contrib_popularity",
    "penalty_negative_genre",
    "penalty_negative_semantic",
    "penalty_negative_collab",
    "dominant_signal",
    "hybrid_score",
    "generic_semantic_penalty",
    "anchor_specificity_score",
    "anchor_confidence",
    "preference_margin_score",
    "negative_affinity_score",
    "positive_affinity_score",
    "base_model_evidence_score",
    "human_like_v3_raw_score",
    "human_like_v3_adjusted_score",
    "final_recommendation_score",
    "human_like_rank_score",
    "human_like_rank_percentile",
    "expected_decade_share",
    "actual_top_decade_share",
    "temporal_overrepresentation_penalty",
    "temporal_portfolio_penalty",
    "user_temporal_affinity_score",
    "temporal_mismatch_penalty",
    "temporal_preference_margin_score",
    "temporal_distance_from_profile",
    "temporal_confidence",
    "rerank_jump_penalty",
    "previous_hybrid_percentile",
    "previous_semantic_percentile",
    "human_like_raw_score",
    "human_like_score",
    "anchor_match_score",
    "anchor_coherence_score",
    "weak_anchor_penalty",
    "num_coherent_anchors",
    "explanation_quality_score",
    "explanation_quality_penalty",
    "false_positive_risk",
    "risk_medium_threshold",
    "recommendation_branch",
    "recommendation_bucket",
    "anchor_movies_matched",
    "why_this_matches",
    "explanation_display",
    "risk_explanation",
    "explanation",
]

available_result_columns = [col for col in RESULT_COLUMNS if col in recommendations.columns]
display(recommendations[available_result_columns].head(20))

## 18. Sanity checks finales

In [ ]:
n_recommendations = len(recommendations)
diagnostic_top_cols = [
    "rank", "title", "year", "decade", "final_recommendation_score", "human_like_rank_score",
    "human_like_v3_raw_score", "human_like_v3_adjusted_score", "recommendation_bucket",
    "dominant_signal", "anchor_confidence", "false_positive_risk", "previous_hybrid_rank",
    "anchor_movies_matched",
]
print("Diagnostico breve del top 20 actual:")
display(recommendations[[col for col in diagnostic_top_cols if col in recommendations.columns]].head(20))

already_watched = recommendations["movieId"].isin(watched_movie_ids).sum()
already_rated = recommendations["movieId"].isin(rated_movie_ids).sum()
mean_rating_top = recommendations["rating_mean"].mean()
mean_count_top = recommendations["rating_count"].mean()
distinct_main_genres = recommendations["main_genre"].nunique()
def _positive_profile_year_stats(profile_movies):
    if profile_movies is None or profile_movies.empty or "year" not in profile_movies.columns:
        return None, None
    rating_col = next((col for col in ["user_rating_5", "rating_5", "rating", "user_rating"] if col in profile_movies.columns), None)
    years = pd.to_numeric(profile_movies["year"], errors="coerce")
    if rating_col:
        ratings = pd.to_numeric(profile_movies[rating_col], errors="coerce")
        mask = years.notna() & ratings.notna() & (ratings >= 4.0)
        weights = (ratings.loc[mask] - 3.0).clip(lower=0.25)
    else:
        mask = years.notna()
        weights = pd.Series(1.0, index=years.loc[mask].index)
    if not mask.any() or weights.sum() <= 0:
        return None, None
    selected_years = years.loc[mask].astype(float)
    mean = float(np.average(selected_years, weights=weights))
    variance = float(np.average((selected_years - mean) ** 2, weights=weights)) if len(selected_years) > 1 else 0.0
    return mean, float(np.sqrt(max(variance, 0.0)))

positive_year_mean, positive_year_dispersion = _positive_profile_year_stats(liked_movies if "liked_movies" in globals() else None)
top_year_mean = float(pd.to_numeric(recommendations["year"], errors="coerce").mean()) if "year" in recommendations.columns else np.nan
positive_decade_share = rerank_summary.get("positive_decade_share", candidates_scored.attrs.get("positive_decade_share", {}) if "candidates_scored" in globals() else {})
top_decade_share = recommendations["decade"].value_counts(normalize=True, dropna=True).sort_index().to_dict() if "decade" in recommendations.columns else {}
all_temporal_keys = sorted(set(positive_decade_share) | set(top_decade_share))
temporal_profile_divergence = 0.5 * sum(abs(float(top_decade_share.get(key, 0.0)) - float(positive_decade_share.get(key, 0.0))) for key in all_temporal_keys) if all_temporal_keys else np.nan
distant_consolidated_mask = (
    (recommendations.get("temporal_distance_from_profile", pd.Series(0.0, index=recommendations.index)) >= recommendations.get("classic_distance_threshold", pd.Series(np.inf, index=recommendations.index)))
    & ((recommendations.get("rating_score", pd.Series(0.0, index=recommendations.index)) >= recommendations.get("classic_rating_threshold", pd.Series(np.inf, index=recommendations.index))) | (recommendations.get("popularity_score", pd.Series(0.0, index=recommendations.index)) >= recommendations.get("classic_popularity_threshold", pd.Series(np.inf, index=recommendations.index))))
)

print(f"Numero de recomendaciones: {n_recommendations}")
print(f"Ya vistas: {already_watched}")
print(f"Ya valoradas: {already_rated}")
print(f"rating_mean medio top 20: {mean_rating_top:.3f}")
print(f"rating_count medio top 20: {mean_count_top:.1f}")
print(f"Generos principales distintos: {distinct_main_genres}")
print(f"Media de año perfil positivo ponderado: {positive_year_mean:.1f}" if pd.notna(positive_year_mean) else "Media de año perfil positivo ponderado: no disponible")
print(f"Media de año top final: {top_year_mean:.1f}" if pd.notna(top_year_mean) else "Media de año top final: no disponible")
print(f"Dispersion temporal perfil positivo: {positive_year_dispersion:.1f}" if pd.notna(positive_year_dispersion) else "Dispersion temporal perfil positivo: no disponible")
print(f"Divergencia temporal top vs perfil positivo: {temporal_profile_divergence:.3f}" if pd.notna(temporal_profile_divergence) else "Divergencia temporal top vs perfil positivo: no disponible")

for column in [
    "hybrid_score",
    "base_model_evidence_score",
    "human_like_v3_raw_score",
    "human_like_v3_adjusted_score",
    "final_recommendation_score",
    "human_like_rank_score",
    "human_like_rank_percentile",
    "human_like_score",
    "positive_affinity_score",
    "negative_affinity_score",
    "preference_margin_score",
    "anchor_match_score",
    "anchor_coherence_score",
    "anchor_specificity_score",
    "explanation_quality_score",
    "user_temporal_affinity_score",
    "temporal_portfolio_penalty",
    "temporal_mismatch_penalty",
    "rerank_jump_penalty",
    "false_positive_risk",
    "semantic_relevance_adjusted_score",
    "item_item_collab_score",
]:
    if column in recommendations.columns:
        print(f"Media {column} top 20: {recommendations[column].mean():.3f}")

print("Distribucion recommendation_bucket:")
display(recommendations["recommendation_bucket"].value_counts(dropna=False))
print("Distribucion recommendation_branch:")
display(recommendations["recommendation_branch"].value_counts(dropna=False))
print("Distribucion dominant_signal:")
display(recommendations["dominant_signal"].value_counts(dropna=False))
print("Distribucion anchor_confidence:")
display(recommendations["anchor_confidence"].value_counts(dropna=False))
print("Distribucion por decada:")
display(recommendations["decade"].value_counts(dropna=False).sort_index() if "decade" in recommendations.columns else pd.Series(dtype=int))
print("Distribucion temporal esperada del perfil positivo:")
display(pd.Series(positive_decade_share, dtype=float).sort_index() if positive_decade_share else pd.Series(dtype=float))
print("Distribucion temporal del top final:")
display(pd.Series(top_decade_share, dtype=float).sort_index() if top_decade_share else pd.Series(dtype=float))

quality_warnings = []

def _max_share(column):
    counts = recommendations[column].value_counts(dropna=False) if column in recommendations.columns else pd.Series(dtype=int)
    return float(counts.max() / len(recommendations)) if len(recommendations) and not counts.empty else 0.0

def _good_evidence_mask(df):
    risk_threshold = df.get("risk_medium_threshold", pd.Series(0.40, index=df.index))
    return (
        (df.get("anchor_confidence", pd.Series("", index=df.index)) == "alto")
        | (
            (df.get("preference_margin_score", pd.Series(0.0, index=df.index)) >= 0.70)
            & (df.get("false_positive_risk", pd.Series(1.0, index=df.index)) < risk_threshold)
            & (df.get("user_temporal_affinity_score", pd.Series(0.0, index=df.index)) >= 0.45)
            & (df.get("base_model_evidence_score", pd.Series(0.0, index=df.index)) >= 0.45)
        )
    )

if not bool(recommendations.attrs.get("frozen_rank_order_preserved", True)):
    quality_warnings.append("El orden final cambio respecto a frozen_rank.")
if EXPORT_LEGACY_EXPORTS:
    quality_warnings.append("EXPORT_FULL_RESULTS=True puede generar mas de un archivo; debe permanecer False.")
expected_generated_files = [str(path) for path in POWERBI_EXPECTED_FILES] if "POWERBI_EXPECTED_FILES" in globals() and EXPORT_FULL_RESULTS else []
if len(expected_generated_files) > 4:
    quality_warnings.append("Se generarian mas de 4 archivos CSV para Power BI.")
if "final_recommendation_score" in recommendations.columns and (recommendations["final_recommendation_score"].max() - recommendations["final_recommendation_score"].min()) < 0.12:
    quality_warnings.append("final_recommendation_score del top 20 tiene rango menor a 0.12.")
if "final_recommendation_score" in recommendations.columns and int((recommendations["final_recommendation_score"] == 1.0).sum()) > 2:
    quality_warnings.append("Hay mas de 2 valores exactamente 1.0 en final_recommendation_score.")
if EXPORT_LEGACY_EXPORTS:
    quality_warnings.append("EXPORT_FULL_RESULTS esta activo; en fase de afinado debe permanecer False.")
if positive_year_mean is None or positive_year_dispersion is None:
    quality_warnings.append("Metricas temporales positivas no disponibles; no se imprimen como NaN.")
if _max_share("dominant_signal") > 0.75:
    quality_warnings.append("dominant_signal concentra mas del 75% en una sola categoria.")
if "dominant_signal" in recommendations.columns and (recommendations["dominant_signal"] == "risk_penalized").mean() == 1.0:
    quality_warnings.append("dominant_signal tiene 100% risk_penalized; revisar capa de salida.")
if "dominant_signal" in recommendations.columns and (recommendations["dominant_signal"] == "rerank_limited").mean() == 1.0:
    quality_warnings.append("dominant_signal tiene 100% rerank_limited; revisar capa de salida.")
if {"dominant_signal", "rerank_jump_penalty"}.issubset(recommendations.columns):
    rerank_values = pd.to_numeric(recommendations["rerank_jump_penalty"], errors="coerce").fillna(0.0)
    if not _has_real_variance(rerank_values) and (recommendations["dominant_signal"] == "rerank_limited").any():
        quality_warnings.append("rerank_limited aparece aunque rerank_jump_penalty no tiene varianza real.")
if {"dominant_signal", "recommendation_bucket"}.issubset(recommendations.columns) and ((recommendations["dominant_signal"] == "risk_penalized") & (recommendations["recommendation_bucket"] != "riesgo_controlado")).any():
    quality_warnings.append("risk_penalized aparece en peliculas cuyo bucket no es riesgo_controlado.")
if "anchor_confidence" in recommendations.columns and (recommendations["anchor_confidence"] == "alto").mean() > 0.70:
    quality_warnings.append("anchor_confidence alto supera el 70% del top.")
if _max_share("recommendation_bucket") > 0.70:
    quality_warnings.append("recommendation_bucket concentra mas del 70% en una sola categoria.")
if "recommendation_bucket" in recommendations.columns and (recommendations["recommendation_bucket"] == "riesgo_controlado").mean() == 1.0:
    quality_warnings.append("recommendation_bucket tiene 100% riesgo_controlado; revisar capa de salida.")
if "recommendation_bucket" in recommendations.columns and (recommendations["recommendation_bucket"] == "clasico_pendiente").mean() == 1.0:
    quality_warnings.append("recommendation_bucket tiene 100% clasico_pendiente; revisar capa de salida.")
if "recommendation_bucket" in recommendations.columns and (recommendations["recommendation_bucket"] == "apuesta_segura").mean() > 0.60:
    quality_warnings.append("Mas del 60% del top esta etiquetado como apuesta_segura.")
if "explanation_display" in recommendations.columns and recommendations["explanation_display"].fillna("").astype(str).str.strip().eq("").any():
    quality_warnings.append("Hay explanation_display vacio.")
if "explanation_display" in recommendations.columns and len(recommendations):
    explanation_share = recommendations["explanation_display"].fillna("").astype(str).str.strip().value_counts(normalize=True, dropna=False)
    if not explanation_share.empty and float(explanation_share.iloc[0]) > 0.60:
        quality_warnings.append("explanation_display repite el mismo texto exacto en mas del 60% del top.")
if "recommendation_branch" in recommendations.columns and recommendations["recommendation_branch"].value_counts().max() > 6:
    quality_warnings.append("Hay mas de 6 peliculas de la misma rama.")
if "main_genre" in recommendations.columns and recommendations["main_genre"].value_counts().max() > 8:
    quality_warnings.append("Hay mas de 8 peliculas del mismo genero principal.")
if "decade" in recommendations.columns:
    decade_counts = recommendations["decade"].value_counts(dropna=False)
    if len(decade_counts) >= 4 and decade_counts.nunique() == 1:
        quality_warnings.append("Distribucion por decada sospechosamente uniforme; revisar si hay cuota temporal accidental.")
if pd.notna(temporal_profile_divergence) and temporal_profile_divergence > 0.45:
    quality_warnings.append("La distribucion temporal del top se aleja demasiado del perfil positivo.")
if pd.notna(positive_year_mean) and pd.notna(top_year_mean) and abs(top_year_mean - positive_year_mean) > 18:
    quality_warnings.append("La media de año del top final se aleja demasiado del perfil positivo.")
if "false_positive_risk" in recommendations.columns and recommendations["false_positive_risk"].mean() > 0.50:
    quality_warnings.append("false_positive_risk medio del top 20 supera 0.50.")
if "false_positive_risk" in recommendations.columns and (recommendations["false_positive_risk"] > 0.70).any():
    quality_warnings.append("Hay peliculas con false_positive_risk > 0.70 en el top.")
if "false_positive_risk" in recommendations.columns and "recommendation_bucket" in recommendations.columns and "risk_medium_threshold" in recommendations.columns:
    if (recommendations["false_positive_risk"] >= recommendations["risk_medium_threshold"]).any() and not (recommendations["recommendation_bucket"] == "riesgo_controlado").any():
        quality_warnings.append("Hay peliculas por encima del umbral dinamico de riesgo medio pero ninguna queda etiquetada como riesgo_controlado.")
    weak_risk_bucket = (recommendations["recommendation_bucket"] == "riesgo_controlado") & (recommendations["false_positive_risk"] < recommendations["risk_medium_threshold"]) & (recommendations.get("rerank_jump_penalty", pd.Series(0.0, index=recommendations.index)) < recommendations.get("rerank_jump_penalty", pd.Series(0.0, index=recommendations.index)).quantile(0.90)) & (recommendations.get("temporal_mismatch_penalty", pd.Series(0.0, index=recommendations.index)) < recommendations.get("temporal_mismatch_penalty", pd.Series(0.0, index=recommendations.index)).quantile(0.85))
    if weak_risk_bucket.any():
        quality_warnings.append("Hay riesgo_controlado con false_positive_risk bajo el umbral dinamico y sin otro riesgo fuerte detectable.")
if {"recommendation_bucket", "false_positive_risk"}.issubset(recommendations.columns):
    sanity_risk = pd.to_numeric(recommendations["false_positive_risk"], errors="coerce").fillna(0.0)
    sanity_margin = recommendations.get("preference_margin_score", pd.Series(0.0, index=recommendations.index))
    sanity_margin = pd.to_numeric(sanity_margin, errors="coerce").fillna(0.0)
    sanity_negative = recommendations.get("negative_affinity_score", pd.Series(0.0, index=recommendations.index))
    sanity_negative = pd.to_numeric(sanity_negative, errors="coerce").fillna(0.0)
    sanity_anchor_confidence = recommendations.get("anchor_confidence", pd.Series("", index=recommendations.index)).fillna("").astype(str).str.lower()
    if "risk_medium_threshold" in recommendations.columns:
        sanity_medium = pd.to_numeric(recommendations["risk_medium_threshold"], errors="coerce").fillna(sanity_risk.quantile(0.80))
    else:
        sanity_medium = pd.Series(float(sanity_risk.quantile(0.80)), index=recommendations.index)
    sanity_rerank = pd.to_numeric(recommendations.get("rerank_jump_penalty", pd.Series(0.0, index=recommendations.index)), errors="coerce").fillna(0.0)
    sanity_temporal = pd.to_numeric(recommendations.get("temporal_mismatch_penalty", pd.Series(0.0, index=recommendations.index)), errors="coerce").fillna(0.0)
    sanity_rerank_high = _has_real_variance(sanity_rerank) & (sanity_rerank >= sanity_rerank.quantile(0.80)) & (sanity_rerank > 0)
    sanity_temporal_high = _has_real_variance(sanity_temporal) & (sanity_temporal >= sanity_temporal.quantile(0.80)) & (sanity_temporal > 0)
    sanity_real_risk = (sanity_risk >= sanity_medium) & ((sanity_margin <= sanity_margin.quantile(0.35)) | (sanity_negative >= sanity_negative.quantile(0.70)) | sanity_anchor_confidence.eq("bajo") | sanity_rerank_high | sanity_temporal_high)
    if ((recommendations["recommendation_bucket"] == "riesgo_controlado") & ~sanity_real_risk).any():
        quality_warnings.append("riesgo_controlado aparece sin riesgo real segun los umbrales finales.")
if "recommendation_bucket" in recommendations.columns and distant_consolidated_mask.sum() >= 2 and not (recommendations["recommendation_bucket"] == "clasico_pendiente").any():
    quality_warnings.append("Hay peliculas temporalmente alejadas y consolidadas, pero ninguna queda como clasico_pendiente.")
if "anchor_coherence_score" in recommendations.columns and "anchor_movies_matched" in recommendations.columns:
    weak_anchor_used = recommendations[recommendations["anchor_movies_matched"].fillna("").astype(str).str.len().gt(0) & (recommendations["anchor_coherence_score"] < 0.35)]
    if not weak_anchor_used.empty:
        quality_warnings.append("Hay anclas usadas con anchor_coherence_score bajo.")
if "anchor_specificity_score" in recommendations.columns and (recommendations["anchor_specificity_score"] < 0.40).mean() > 0.50:
    quality_warnings.append("Mas del 50% del top tiene anchor_specificity_score bajo.")
if "preference_margin_score" in recommendations.columns and (recommendations["preference_margin_score"] < 0.45).mean() > 0.30:
    quality_warnings.append("Mas del 30% del top tiene preference_margin_score bajo.")
if "previous_hybrid_rank" in recommendations.columns and ((recommendations["previous_hybrid_rank"] > 300) & ~_good_evidence_mask(recommendations)).mean() > 0.30:
    quality_warnings.append("Mas del 30% del top tiene previous_hybrid_rank > 300 sin justificacion fuerte.")
if "base_model_evidence_score" in recommendations.columns and (recommendations["base_model_evidence_score"] < 0.40).mean() > 0.40:
    quality_warnings.append("Mas del 40% del top entra con base_model_evidence_score bajo.")

for message in quality_warnings:
    warnings.warn(message)

if not quality_warnings:
    print("Validadores de calidad: sin alertas fuertes.")

summary_execution = {
    "dataframe_final": "final_human_like_recommendations",
    "top_order_preserved_by": "frozen_rank",
    "frozen_rank_order_preserved": bool(recommendations.attrs.get("frozen_rank_order_preserved", False)),
    "score_ranking_final": "final_recommendation_score",
    "n_candidates_before_top": int(len(candidates_scored)) if "candidates_scored" in globals() else np.nan,
    "final_recommendation_score_range_top20": float(recommendations["final_recommendation_score"].max() - recommendations["final_recommendation_score"].min()) if "final_recommendation_score" in recommendations.columns else np.nan,
    "final_recommendation_score_exact_ones_top20": int((recommendations["final_recommendation_score"] == 1.0).sum()) if "final_recommendation_score" in recommendations.columns else np.nan,
    "human_like_v3_adjusted_score_range_top20": float(recommendations["human_like_v3_adjusted_score"].max() - recommendations["human_like_v3_adjusted_score"].min()) if "human_like_v3_adjusted_score" in recommendations.columns else np.nan,
    "human_like_rank_score_range_top20_aux": float(recommendations["human_like_rank_score"].max() - recommendations["human_like_rank_score"].min()) if "human_like_rank_score" in recommendations.columns else np.nan,
    "human_like_rank_percentile_range_top20": float(recommendations["human_like_rank_percentile"].max() - recommendations["human_like_rank_percentile"].min()) if "human_like_rank_percentile" in recommendations.columns else np.nan,
    "positive_profile_weighted_year_mean": positive_year_mean,
    "positive_profile_year_dispersion": positive_year_dispersion,
    "top_final_year_mean": top_year_mean,
    "positive_decade_share": positive_decade_share,
    "top_decade_share": top_decade_share,
    "temporal_profile_divergence": temporal_profile_divergence,
    "decades": recommendations["decade"].value_counts(dropna=False).sort_index().to_dict() if "decade" in recommendations.columns else {},
    "buckets": recommendations["recommendation_bucket"].value_counts().to_dict() if "recommendation_bucket" in recommendations.columns else {},
    "buckets_before_output_fix": rerank_summary.get("output_fix_bucket_before", {}),
    "buckets_after_output_fix": recommendations["recommendation_bucket"].value_counts().to_dict() if "recommendation_bucket" in recommendations.columns else {},
    "buckets_after_final_label_fix": recommendations["recommendation_bucket"].value_counts().to_dict() if "recommendation_bucket" in recommendations.columns else {},
    "branches": recommendations["recommendation_branch"].value_counts().to_dict() if "recommendation_branch" in recommendations.columns else {},
    "dominant_signal": recommendations["dominant_signal"].value_counts().to_dict() if "dominant_signal" in recommendations.columns else {},
    "dominant_signal_before_output_fix": rerank_summary.get("output_fix_dominant_before", {}),
    "dominant_signal_after_output_fix": recommendations["dominant_signal"].value_counts().to_dict() if "dominant_signal" in recommendations.columns else {},
    "dominant_signal_after_final_label_fix": recommendations["dominant_signal"].value_counts().to_dict() if "dominant_signal" in recommendations.columns else {},
    "anchor_confidence": recommendations["anchor_confidence"].value_counts().to_dict() if "anchor_confidence" in recommendations.columns else {},
    "mean_false_positive_risk_top20": float(recommendations["false_positive_risk"].mean()) if "false_positive_risk" in recommendations.columns else np.nan,
    "max_false_positive_risk_top20": float(recommendations["false_positive_risk"].max()) if "false_positive_risk" in recommendations.columns else np.nan,
    "risk_medium_threshold": float(recommendations["risk_medium_threshold"].median()) if "risk_medium_threshold" in recommendations.columns else np.nan,
    "n_riesgo_controlado": int((recommendations["recommendation_bucket"] == "riesgo_controlado").sum()) if "recommendation_bucket" in recommendations.columns else np.nan,
    "n_risk_penalized": int((recommendations["dominant_signal"] == "risk_penalized").sum()) if "dominant_signal" in recommendations.columns else np.nan,
    "n_rerank_limited": int((recommendations["dominant_signal"] == "rerank_limited").sum()) if "dominant_signal" in recommendations.columns else np.nan,
    "riesgo_controlado_titles": " | ".join(recommendations.loc[recommendations["recommendation_bucket"] == "riesgo_controlado", "title"].astype(str)) if "recommendation_bucket" in recommendations.columns else "",
    "clasico_pendiente_titles": " | ".join(recommendations.loc[recommendations["recommendation_bucket"] == "clasico_pendiente", "title"].astype(str)) if "recommendation_bucket" in recommendations.columns else "",
    "mean_user_temporal_affinity_top20": float(recommendations["user_temporal_affinity_score"].mean()) if "user_temporal_affinity_score" in recommendations.columns else np.nan,
    "overlap_hybrid_vs_final": human_like_ranking_overlap_top20 if "human_like_ranking_overlap_top20" in globals() else np.nan,
    "overlap_semantic_vs_final": semantic_human_like_overlap_top20 if "semantic_human_like_overlap_top20" in globals() else np.nan,
    "powerbi_dataset_dir": str(POWERBI_DATASETS),
    "generated_files_default": expected_generated_files,
    "debug_snapshot_disabled": not EXPORT_DEBUG_SNAPSHOT,
    "legacy_exports_disabled": not EXPORT_LEGACY_EXPORTS,
}
display(pd.DataFrame(summary_execution.items(), columns=["metric", "value"]))

print("Resumen final del ajuste interpretativo:")
print(f"Top preservado: {bool(recommendations.attrs.get('frozen_rank_order_preserved', False))}")
print(f"Distribucion final recommendation_bucket: {recommendations['recommendation_bucket'].value_counts(dropna=False).to_dict() if 'recommendation_bucket' in recommendations.columns else {}}")
print(f"Distribucion final dominant_signal: {recommendations['dominant_signal'].value_counts(dropna=False).to_dict() if 'dominant_signal' in recommendations.columns else {}}")
if "final_recommendation_score" in recommendations.columns:
    print(f"Rango final_recommendation_score: {recommendations['final_recommendation_score'].min():.6f} - {recommendations['final_recommendation_score'].max():.6f}")
print(f"Conteo riesgo_controlado: {int((recommendations['recommendation_bucket'] == 'riesgo_controlado').sum()) if 'recommendation_bucket' in recommendations.columns else 0}")
print(f"Conteo risk_penalized: {int((recommendations['dominant_signal'] == 'risk_penalized').sum()) if 'dominant_signal' in recommendations.columns else 0}")
print(f"Conteo rerank_limited: {int((recommendations['dominant_signal'] == 'rerank_limited').sum()) if 'dominant_signal' in recommendations.columns else 0}")
print(f"Directorio Power BI: {POWERBI_DATASETS}")
print(f"Archivos Power BI esperados: {expected_generated_files}")
print(f"EXPORT_DEBUG_SNAPSHOT={EXPORT_DEBUG_SNAPSHOT}")
print(f"EXPORT_FULL_RESULTS={EXPORT_FULL_RESULTS}")
final_summary_cols = ["rank", "title", "final_recommendation_score", "recommendation_bucket", "dominant_signal", "explanation_display"]
if "rank" not in recommendations.columns and "frozen_rank" in recommendations.columns:
    recommendations["rank"] = recommendations["frozen_rank"]
display(recommendations[[col for col in final_summary_cols if col in recommendations.columns]].head(20))


## 19. Exportación

In [ ]:
BUCKET_LABELS = {
    "apuesta_segura": "Apuesta segura",
    "muy_parecida_a_favoritas": "Muy parecida a favoritas",
    "descubrimiento_compatible": "Descubrimiento compatible",
    "clasico_pendiente": "T\u00edtulo consolidado",
    "riesgo_controlado": "Riesgo controlado",
}

DOMINANT_SIGNAL_LABELS = {
    "anchor": "Anclas de favoritas",
    "margin": "Margen positivo",
    "semantic": "Similitud sem\u00e1ntica",
    "collaborative": "Se\u00f1al colaborativa",
    "quality": "Calidad / valoraci\u00f3n",
    "popularity": "Popularidad",
    "risk_penalized": "Riesgo penalizado",
    "rerank_limited": "Reordenaci\u00f3n limitada",
}

SCORE_EXPORT_COLUMNS = [
    "final_recommendation_score",
    "false_positive_risk",
    "preference_margin_score",
    "user_temporal_affinity_score",
    "rating_score",
    "popularity_score",
    "hybrid_score",
    "human_like_v3_adjusted_score",
]

POWERBI_CSV_SEP = ";"
POWERBI_CSV_DECIMAL = ","
POWERBI_CSV_ENCODING = "utf-8-sig"

POWERBI_INTEGER_COLUMNS = ["rank", "movieId", "year", "decade", "count"]
POWERBI_DECIMAL_COLUMNS = [
    "final_recommendation_score",
    "false_positive_risk",
    "preference_margin_score",
    "user_temporal_affinity_score",
    "rating_score",
    "popularity_score",
    "hybrid_score",
    "human_like_v3_adjusted_score",
    "percentage",
    "trakt_rating",
    "user_rating_5",
]


def _clean_export_text(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(value, (list, tuple, set)):
        value = " | ".join(str(item) for item in value if str(item).strip())
    text = str(value).replace("\r", " ").replace("\n", " ").strip()
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, (list, tuple, set)):
                text = " | ".join(str(item) for item in parsed if str(item).strip())
        except (SyntaxError, ValueError):
            pass
    return " ".join(text.split())


def _clean_pipe_text(value):
    text = _clean_export_text(value).replace(";", "|")
    parts = [part.strip(" -") for part in text.split("|")]
    return " | ".join(part for part in parts if part and part.lower() not in {"nan", "none"})


def _coalesce_columns(df, candidates, default=""):
    for column in candidates:
        if column in df.columns:
            return df[column]
    return pd.Series(default, index=df.index)


def _force_powerbi_csv_types(df):
    work = df.copy()
    for column in POWERBI_INTEGER_COLUMNS:
        if column in work.columns:
            work[column] = pd.to_numeric(work[column], errors="coerce").astype("Int64")
    for column in POWERBI_DECIMAL_COLUMNS:
        if column in work.columns:
            work[column] = pd.to_numeric(work[column], errors="coerce")
    return work


def _format_metric_value_for_powerbi(value):
    if isinstance(value, (int, np.integer)) and not isinstance(value, bool):
        return str(int(value))
    if isinstance(value, (float, np.floating)) and pd.notna(value):
        text = f"{float(value):.4f}".rstrip("0").rstrip(".")
        return text.replace(".", ",")
    return value


def _add_temporal_affinity_label(df):
    work = df.copy()
    affinity = pd.to_numeric(work.get("user_temporal_affinity_score", pd.Series(np.nan, index=work.index)), errors="coerce")
    valid_affinity = affinity.dropna()
    if valid_affinity.empty:
        work["temporal_affinity_label"] = "Baja afinidad temporal"
        return work
    high_threshold = min(float(valid_affinity.quantile(0.66)), 0.85)
    medium_threshold = min(float(valid_affinity.quantile(0.33)), 0.65)
    work["temporal_affinity_label"] = np.select(
        [affinity >= high_threshold, affinity >= medium_threshold],
        ["Alta afinidad temporal", "Afinidad temporal media"],
        default="Baja afinidad temporal",
    )
    return work


def _with_basic_movie_fields(df):
    work = df.copy()
    work["title"] = _coalesce_columns(work, ["title", "title_ml", "title_movie", "title_trakt"], "")
    work["year"] = pd.to_numeric(_coalesce_columns(work, ["year", "year_ml", "year_movie"], np.nan), errors="coerce")
    work["decade"] = ((work["year"] // 10) * 10).astype("Int64")
    work["genres"] = _coalesce_columns(work, ["genres", "genres_ml", "genres_movie", "genres_trakt"], "")
    if "main_genre" not in work.columns:
        work["main_genre"] = work["genres"].apply(lambda value: split_genres(value)[0] if split_genres(value) else "Unknown")
    return work


def _prepare_recommendations_for_powerbi(source):
    recs = source.copy()
    if "frozen_rank" in recs.columns:
        sort_col = "frozen_rank"
    elif "rank" in recs.columns:
        sort_col = "rank"
    elif "final_rank" in recs.columns:
        sort_col = "final_rank"
    else:
        sort_col = None
        recs["rank"] = np.arange(1, len(recs) + 1)
    if sort_col:
        recs = recs.sort_values(sort_col).reset_index(drop=True)
        rank_values = pd.to_numeric(recs[sort_col], errors="coerce")
        fallback_rank = pd.Series(np.arange(1, len(recs) + 1), index=recs.index)
        recs["rank"] = rank_values.where(rank_values.notna(), fallback_rank).astype(int)
    recs = _add_temporal_affinity_label(recs)
    recs["recommendation_bucket_label"] = recs.get("recommendation_bucket", pd.Series("", index=recs.index)).map(BUCKET_LABELS).fillna(recs.get("recommendation_bucket", ""))
    recs["dominant_signal_label"] = recs.get("dominant_signal", pd.Series("", index=recs.index)).map(DOMINANT_SIGNAL_LABELS).fillna(recs.get("dominant_signal", ""))
    if "anchor_movies_matched" in recs.columns:
        recs["anchor_movies_matched"] = recs["anchor_movies_matched"].map(_clean_pipe_text)
    if "genres" in recs.columns:
        recs["genres"] = recs["genres"].map(_clean_pipe_text)
    if "explanation_display" in recs.columns:
        recs["explanation_display"] = recs["explanation_display"].map(_clean_export_text)
    for column in SCORE_EXPORT_COLUMNS:
        if column in recs.columns:
            recs[column] = pd.to_numeric(recs[column], errors="coerce").round(4)
    export_columns = [
        "rank", "movieId", "title", "year", "decade", "genres", "main_genre",
        "final_recommendation_score", "recommendation_bucket", "recommendation_bucket_label",
        "recommendation_branch", "dominant_signal", "dominant_signal_label", "explanation_display",
        "anchor_movies_matched", "false_positive_risk", "preference_margin_score",
        "user_temporal_affinity_score", "temporal_affinity_label", "rating_score", "popularity_score", "hybrid_score",
        "human_like_v3_adjusted_score",
    ]
    available_columns = [column for column in export_columns if column in recs.columns]
    return clean_dataframe_text_for_csv(recs[available_columns].copy())


def _prepare_profile_for_powerbi():
    profile_parts = []
    if "trakt_ratings_profile" in globals() and not trakt_ratings_profile.empty:
        rated = _with_basic_movie_fields(trakt_ratings_profile.copy())
    elif "trakt_ratings" in globals() and not trakt_ratings.empty:
        rated = _with_basic_movie_fields(trakt_ratings.copy())
    else:
        rated = pd.DataFrame()
    if not rated.empty and "user_rating_5" in rated.columns:
        rated["user_rating_5"] = pd.to_numeric(rated["user_rating_5"], errors="coerce")
        rated["trakt_rating"] = pd.to_numeric(_coalesce_columns(rated, ["user_rating_trakt", "user_rating_normalized", "user_rating_5"], np.nan), errors="coerce")
        rated_profile = rated[(rated["user_rating_5"] >= 4.0) | (rated["user_rating_5"] <= 2.5)].copy()
        rated_profile["profile_type"] = np.where(rated_profile["user_rating_5"] >= 4.0, "positivo", "negativo")
        rated_profile["rated"] = True
        rated_profile["watched"] = rated_profile.get("movieId", pd.Series(index=rated_profile.index)).isin(watched_movie_ids if "watched_movie_ids" in globals() else set())
        profile_parts.append(rated_profile)
    if "trakt_watched" in globals() and not trakt_watched.empty:
        watched = _with_basic_movie_fields(trakt_watched.copy())
        rated_ids = set(rated["movieId"].dropna().astype(int)) if not rated.empty and "movieId" in rated.columns else set()
        watched_unrated = watched[~watched["movieId"].isin(rated_ids)].copy() if "movieId" in watched.columns else pd.DataFrame()
        if not watched_unrated.empty:
            watched_unrated["trakt_rating"] = np.nan
            watched_unrated["user_rating_5"] = np.nan
            watched_unrated["profile_type"] = "visto_sin_rating"
            watched_unrated["rated"] = False
            watched_unrated["watched"] = True
            profile_parts.append(watched_unrated)
    if profile_parts:
        profile = pd.concat(profile_parts, ignore_index=True, sort=False)
    else:
        profile = pd.DataFrame(columns=["movieId", "title", "year", "decade", "genres", "main_genre", "trakt_rating", "user_rating_5", "watched", "rated", "profile_type", "source"])
    profile["source"] = "Trakt"
    profile_columns = ["movieId", "title", "year", "decade", "genres", "main_genre", "trakt_rating", "user_rating_5", "watched", "rated", "profile_type", "source"]
    for column in ["genres", "title", "main_genre", "profile_type", "source"]:
        if column in profile.columns:
            profile[column] = profile[column].map(_clean_pipe_text if column == "genres" else _clean_export_text)
    for unknown_column in ["genres", "main_genre"]:
        if unknown_column in profile.columns:
            empty_mask = profile[unknown_column].fillna("").astype(str).str.strip().eq("")
            profile.loc[empty_mask, unknown_column] = "Unknown"
    for column in ["trakt_rating", "user_rating_5"]:
        if column in profile.columns:
            profile[column] = pd.to_numeric(profile[column], errors="coerce").round(4)
    return clean_dataframe_text_for_csv(profile[[column for column in profile_columns if column in profile.columns]].copy())


def _prepare_metrics_for_powerbi(recs):
    metrics = {
        "total_recommendations": int(len(recs)),
        "total_candidates": int(len(candidates_scored)) if "candidates_scored" in globals() else "no_disponible",
        "trakt_ratings_mapped": int(len(trakt_ratings)) if "trakt_ratings" in globals() else "no_disponible",
        "trakt_watched_mapped": int(len(trakt_watched)) if "trakt_watched" in globals() else "no_disponible",
        "score_min": float(pd.to_numeric(recs.get("final_recommendation_score"), errors="coerce").min()) if "final_recommendation_score" in recs.columns else "no_disponible",
        "score_max": float(pd.to_numeric(recs.get("final_recommendation_score"), errors="coerce").max()) if "final_recommendation_score" in recs.columns else "no_disponible",
        "score_mean": float(pd.to_numeric(recs.get("final_recommendation_score"), errors="coerce").mean()) if "final_recommendation_score" in recs.columns else "no_disponible",
        "false_positive_risk_mean": float(pd.to_numeric(recs.get("false_positive_risk"), errors="coerce").mean()) if "false_positive_risk" in recs.columns else "no_disponible",
        "preference_margin_mean": float(pd.to_numeric(recs.get("preference_margin_score"), errors="coerce").mean()) if "preference_margin_score" in recs.columns else "no_disponible",
        "buckets_count": int(recs["recommendation_bucket"].nunique()) if "recommendation_bucket" in recs.columns else "no_disponible",
        "branches_count": int(recs["recommendation_branch"].nunique()) if "recommendation_branch" in recs.columns else "no_disponible",
        "dominant_signals_count": int(recs["dominant_signal"].nunique()) if "dominant_signal" in recs.columns else "no_disponible",
        "export_date": pd.Timestamp.today().strftime("%Y-%m-%d"),
        "ranking_score_used": "final_recommendation_score",
        "final_notebook": "notebooks/06_recomendador_hibrido_final.ipynb",
    }
    metric_rows = []
    for metric, value in metrics.items():
        if isinstance(value, float) and pd.notna(value):
            value = round(value, 4)
        value = _format_metric_value_for_powerbi(value)
        metric_rows.append({"metric": metric, "value": value})
    return pd.DataFrame(metric_rows)


def _prepare_distribution_for_powerbi(recs):
    rows = []
    dimensions = ["decade", "main_genre", "recommendation_bucket_label", "recommendation_branch", "dominant_signal_label"]
    total = max(len(recs), 1)
    for dimension in dimensions:
        if dimension not in recs.columns:
            continue
        counts = recs[dimension].dropna().astype(str).str.strip()
        counts = counts[counts != ""].value_counts()
        for category, count in counts.items():
            rows.append({
                "dimension": dimension,
                "category": category,
                "count": int(count),
                "percentage": round((int(count) / total) * 100, 2),
            })
    distribution = pd.DataFrame(rows, columns=["dimension", "category", "count", "percentage"])
    if not distribution.empty:
        distribution = distribution.sort_values(["dimension", "count", "category"], ascending=[True, False, True]).reset_index(drop=True)
    return distribution


if EXPORT_FULL_RESULTS:
    ensure_directories([POWERBI_DATASETS])
    source_recommendations = final_human_like_recommendations.copy() if "final_human_like_recommendations" in globals() else recommendations.copy()
    if "frozen_rank" in source_recommendations.columns:
        source_order_check = source_recommendations.sort_values("frozen_rank").copy()
    elif "rank" in source_recommendations.columns:
        source_order_check = source_recommendations.sort_values("rank").copy()
    elif "final_rank" in source_recommendations.columns:
        source_order_check = source_recommendations.sort_values("final_rank").copy()
    else:
        source_order_check = source_recommendations.copy()
    source_order = source_order_check["title"].astype(str).tolist() if "title" in source_order_check.columns else []
    recomendaciones_finales = _prepare_recommendations_for_powerbi(source_recommendations)
    exported_order = recomendaciones_finales["title"].astype(str).tolist() if "title" in recomendaciones_finales.columns else []
    perfil_usuario_trakt = _prepare_profile_for_powerbi()
    metricas_recomendador = _prepare_metrics_for_powerbi(recomendaciones_finales)
    distribucion_recomendaciones = _prepare_distribution_for_powerbi(recomendaciones_finales)

    powerbi_warnings = []
    if source_order and exported_order and source_order != exported_order:
        powerbi_warnings.append("El orden del top final cambio antes de exportar Power BI.")
    if "final_recommendation_score" in recomendaciones_finales.columns:
        score_values = pd.to_numeric(recomendaciones_finales["final_recommendation_score"], errors="coerce")
        if (score_values.max() - score_values.min()) < 0.12:
            powerbi_warnings.append("final_recommendation_score tiene rango menor a 0.12.")
    for required_text_col in ["explanation_display", "recommendation_bucket", "dominant_signal"]:
        if required_text_col in recomendaciones_finales.columns and recomendaciones_finales[required_text_col].fillna("").astype(str).str.strip().eq("").any():
            powerbi_warnings.append(f"{required_text_col} esta vacio en alguna recomendacion.")
    if "explanation_display" in recomendaciones_finales.columns and recomendaciones_finales["explanation_display"].astype(str).str.contains(r"[\r\n]", regex=True).any():
        powerbi_warnings.append("explanation_display contiene saltos de linea.")
    sensitive_terms = ["token", "access_token", "refresh_token", "trakt_token"]
    sensitive_text_count = 0
    for frame_name, frame in {
        "recomendaciones_finales": recomendaciones_finales,
        "perfil_usuario_trakt": perfil_usuario_trakt,
        "metricas_recomendador": metricas_recomendador,
        "distribucion_recomendaciones": distribucion_recomendaciones,
    }.items():
        sensitive_columns = [column for column in frame.columns if any(term in str(column).lower() for term in sensitive_terms)]
        if sensitive_columns:
            powerbi_warnings.append(f"{frame_name} contiene columnas sensibles: {sensitive_columns}")
        text_values = frame.select_dtypes(include=["object", "string"]).astype(str)
        if not text_values.empty:
            sensitive_text_count += int(text_values.apply(lambda col: col.str.lower().str.contains("|".join(sensitive_terms), regex=True, na=False)).sum().sum())
    if sensitive_text_count:
        powerbi_warnings.append(f"Se detectaron {sensitive_text_count} textos sensibles en las exportaciones Power BI.")

    for old_csv in POWERBI_DATASETS.glob("*.csv"):
        old_csv.unlink()

    output_frames = {
        POWERBI_EXPECTED_FILES[0]: recomendaciones_finales,
        POWERBI_EXPECTED_FILES[1]: perfil_usuario_trakt,
        POWERBI_EXPECTED_FILES[2]: metricas_recomendador,
        POWERBI_EXPECTED_FILES[3]: distribucion_recomendaciones,
    }
    generated_paths = []
    output_frames = {path: _force_powerbi_csv_types(frame) for path, frame in output_frames.items()}
    for output_path, frame in output_frames.items():
        frame.to_csv(output_path, sep=POWERBI_CSV_SEP, decimal=POWERBI_CSV_DECIMAL, encoding=POWERBI_CSV_ENCODING, index=False)
        generated_paths.append(output_path)

    csv_count = len(list(POWERBI_DATASETS.glob("*.csv")))
    if csv_count > 4:
        powerbi_warnings.append("Se generan mas de 4 CSV en powerbi/datasets/.")

    recomendaciones_finales = output_frames[POWERBI_EXPECTED_FILES[0]]
    perfil_usuario_trakt = output_frames[POWERBI_EXPECTED_FILES[1]]
    metricas_recomendador = output_frames[POWERBI_EXPECTED_FILES[2]]
    distribucion_recomendaciones = output_frames[POWERBI_EXPECTED_FILES[3]]

    print("Top 20 exportado a Power BI:")
    display(recomendaciones_finales[[col for col in ["rank", "title", "final_recommendation_score", "recommendation_bucket", "dominant_signal", "dominant_signal_label", "temporal_affinity_label"] if col in recomendaciones_finales.columns]].head(20))
    if "final_recommendation_score" in recomendaciones_finales.columns:
        score_values = pd.to_numeric(recomendaciones_finales["final_recommendation_score"], errors="coerce")
        print(f"Rango final_recommendation_score: {score_values.min():.4f} - {score_values.max():.4f}")
    for column in ["recommendation_bucket", "dominant_signal", "dominant_signal_label", "temporal_affinity_label", "recommendation_branch", "decade"]:
        if column in recomendaciones_finales.columns:
            print(f"Distribucion {column}:")
            display(recomendaciones_finales[column].value_counts(dropna=False))
    bucket_label_question_count = int(recomendaciones_finales.get("recommendation_bucket_label", pd.Series(dtype=str)).fillna("").astype(str).map(lambda text: text.count("?")).sum())
    signal_label_question_count = int(recomendaciones_finales.get("dominant_signal_label", pd.Series(dtype=str)).fillna("").astype(str).map(lambda text: text.count("?")).sum())
    explanation_newline_count = int(recomendaciones_finales.get("explanation_display", pd.Series(dtype=str)).fillna("").astype(str).str.contains(r"[\r\n]", regex=True).sum())
    print("Valores unicos recommendation_bucket_label:")
    display(recomendaciones_finales["recommendation_bucket_label"].dropna().drop_duplicates().sort_values().reset_index(drop=True) if "recommendation_bucket_label" in recomendaciones_finales.columns else pd.Series(dtype=str))
    print("Valores unicos dominant_signal_label:")
    display(recomendaciones_finales["dominant_signal_label"].dropna().drop_duplicates().sort_values().reset_index(drop=True) if "dominant_signal_label" in recomendaciones_finales.columns else pd.Series(dtype=str))
    print(f"Saltos de linea en explanation_display: {explanation_newline_count}")
    print(f"Caracteres '?' en recommendation_bucket_label: {bucket_label_question_count}")
    print(f"Caracteres '?' en dominant_signal_label: {signal_label_question_count}")
    print(f"Conteo de tokens/textos sensibles: {sensitive_text_count}")
    if bucket_label_question_count or signal_label_question_count:
        powerbi_warnings.append("Hay caracteres '?' en labels de Power BI; revisar encoding de literales.")
    if explanation_newline_count:
        powerbi_warnings.append("Hay saltos de linea en explanation_display tras limpieza.")
    print(f"EXPORT_DEBUG_SNAPSHOT = {EXPORT_DEBUG_SNAPSHOT}")
    print(f"EXPORT_FULL_RESULTS = {EXPORT_FULL_RESULTS}")
    print("Muestra recomendaciones_finales para Power BI:")
    display(recomendaciones_finales[[col for col in ["rank", "title", "year", "final_recommendation_score"] if col in recomendaciones_finales.columns]].head())
    print("Muestra metricas_recomendador para Power BI:")
    display(metricas_recomendador.head())
    print("Archivos Power BI generados:")
    for output_path in generated_paths:
        print(f"- {output_path} shape={output_frames[output_path].shape} sep='{POWERBI_CSV_SEP}' decimal='{POWERBI_CSV_DECIMAL}' encoding='{POWERBI_CSV_ENCODING}'")
    for warning_message in powerbi_warnings:
        warnings.warn(warning_message)
else:
    print("EXPORT_FULL_RESULTS=False: exportaciones finales Power BI desactivadas.")


In [ ]:
DEBUG_EXPORT_COLUMNS = [
    "rank",
    "frozen_rank",
    "movieId",
    "title",
    "year",
    "decade",
    "genres",
    "main_genre",
    "hybrid_score",
    "base_model_evidence_score",
    "human_like_raw_score",
    "human_like_v3_raw_score",
    "human_like_v3_adjusted_score",
    "final_recommendation_score",
    "human_like_rank_score",
    "human_like_rank_percentile",
    "positive_affinity_score",
    "negative_affinity_score",
    "preference_margin_score",
    "semantic_relevance_adjusted_score",
    "anchor_match_score",
    "anchor_confidence",
    "anchor_coherence_score",
    "weak_anchor_penalty",
    "num_coherent_anchors",
    "anchor_specificity_score",
    "generic_semantic_penalty",
    "explanation_quality_score",
    "explanation_quality_penalty",
    "user_temporal_affinity_score",
    "year_affinity_score",
    "expected_decade_share",
    "actual_top_decade_share",
    "temporal_overrepresentation_penalty",
    "temporal_portfolio_penalty",
    "temporal_mismatch_penalty",
    "temporal_preference_margin_score",
    "temporal_distance_from_profile",
    "temporal_confidence",
    "is_temporal_outlier",
    "allow_temporal_outlier",
    "rerank_jump_penalty",
    "false_positive_risk",
    "risk_medium_threshold",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "rating_score",
    "popularity_score",
    "negative_semantic_relevance_score",
    "negative_genre_score",
    "recommendation_branch",
    "recommendation_bucket",
    "dominant_signal",
    "branch_strength",
    "anchor_movies_matched",
    "nearest_core_anchor_movies",
    "core_semantic_explanation_terms",
    "semantic_explanation_terms",
    "why_this_matches",
    "explanation_display",
    "risk_explanation",
    "explanation",
    "previous_hybrid_rank",
    "previous_semantic_rank",
    "human_like_rank_delta",
]

if EXPORT_DEBUG_SNAPSHOT:
    final_human_like_recommendations = recommendations.copy()
    debug_dataframe_name = "final_human_like_recommendations"
    debug_source = final_human_like_recommendations.copy()
    if "rank" not in debug_source.columns:
        rank_source = "final_rank" if "final_rank" in debug_source.columns else None
        debug_source["rank"] = debug_source[rank_source] if rank_source else np.arange(1, len(debug_source) + 1)

    debug_export_path = REPORTS_DEBUG / "recomendador_debug_snapshot.csv"
    debug_snapshot = export_debug_snapshot(
        debug_source,
        debug_export_path,
        DEBUG_EXPORT_COLUMNS,
        sort_col="rank",
        max_rows=100,
    )
    score_range = (
        (float(debug_snapshot["final_recommendation_score"].min()), float(debug_snapshot["final_recommendation_score"].max()))
        if "final_recommendation_score" in debug_snapshot.columns and len(debug_snapshot)
        else (np.nan, np.nan)
    )
    bucket_distribution = debug_snapshot["recommendation_bucket"].value_counts(dropna=False).to_dict() if "recommendation_bucket" in debug_snapshot.columns else {}
    dominant_signal_distribution = debug_snapshot["dominant_signal"].value_counts(dropna=False).to_dict() if "dominant_signal" in debug_snapshot.columns else {}
    risk_penalized_count = int((debug_snapshot["dominant_signal"] == "risk_penalized").sum()) if "dominant_signal" in debug_snapshot.columns else 0
    rerank_limited_count = int((debug_snapshot["dominant_signal"] == "rerank_limited").sum()) if "dominant_signal" in debug_snapshot.columns else 0
    frozen_order_ok = bool(recommendations.attrs.get("frozen_rank_order_preserved", False)) if "recommendations" in globals() else False
    print(f"Debug snapshot exportado desde dataframe final corregido: {debug_dataframe_name}")
    print(f"Shape dataframe exportado: {debug_source.shape}")
    print(f"Rango final_recommendation_score exportado: {score_range[0]:.6f} - {score_range[1]:.6f}")
    print(f"Distribucion recommendation_bucket exportada: {bucket_distribution}")
    print(f"Distribucion dominant_signal exportada: {dominant_signal_distribution}")
    print(f"Conteo risk_penalized exportado: {risk_penalized_count}")
    print(f"Conteo rerank_limited exportado: {rerank_limited_count}")
    print(f"Orden congelado por frozen_rank preservado: {frozen_order_ok}")
    print(f"Snapshot de debug generado en: {debug_export_path}")
    print(f"Filas exportadas: {len(debug_snapshot)}")
    print(f"Columnas exportadas: {len(debug_snapshot.columns)}")
    print("Primeras 20 recomendaciones:")
    display(debug_snapshot.head(20))
else:
    print("EXPORT_DEBUG_SNAPSHOT=False: snapshot compacto no generado.")


## Cierre metodologico del recomendador final

El `hybrid_score` se mantiene como senal base de auditoria porque resume el recomendador hibrido original: contenido, colaborativo item-item, calidad, popularidad, temporalidad y penalizaciones. No se elimina porque permite comprobar cuanto se aleja el ranking final del modelo base.

El ranking final usa `final_recommendation_score`, calculado desde `human_like_v3_adjusted_score`. `human_like_v3_raw_score` queda como score interno principal antes de penalizaciones; `human_like_rank_score` se conserva como auxiliar y `human_like_score` queda solo como referencia visual/auditoria cuando este saturado. `final_recommendation_score` es un score visible/exportable, no un modelo nuevo.

La capa final combina anclas coherentes, margen positivo-negativo, relevancia semantica ajustada, colaborativo item-item, calidad, popularidad con peso bajo, especificidad de tags, temporalidad personalizada y evidencia del modelo base. Tambien penaliza riesgo de falso positivo, mismatch temporal, anclas debiles, baja calidad de explicacion y saltos excesivos de reranking.

La temporalidad se recupera de forma personalizada: si existen `year_affinity_score` y `temporal_mismatch_penalty`, se reutilizan; si faltan, se reconstruyen desde las peliculas positivas y negativas del usuario con un kernel suave por año. No usa limites temporales fijos ni cuotas exactas por decada: calcula `expected_decade_share` y penaliza suavemente la sobre-representacion cuando el top se aleja del perfil positivo.

Las anclas solo se usan si son coherentes por genero, temporalidad suave y evidencia positiva; es preferible mostrar una o dos anclas buenas antes que tres forzadas. `explanation_display` es la explicacion limpia para usuario final y evita mostrar tags ruidosos aunque `why_this_matches` se mantenga para auditoria.

`clasico_pendiente` significa pelicula temporalmente mas alejada del perfil positivo o consolidada por calidad/popularidad, no un corte fijo por año. `riesgo_controlado` reconoce recomendaciones defendibles pero menos seguras por riesgo, margen debil, mismatch temporal, salto de ranking o anclas debiles.

La diversidad limita rama, genero, exceso temporal dinamico y buckets de riesgo/clasicos, manteniendo el orden base por `final_recommendation_score` y usando diversidad como desempate cuando los scores estan cerca. Los buckets son etiquetas interpretativas y no reentrenan ni rehacen el ranking.

Para la entrega final, `EXPORT_FULL_RESULTS = True` genera los cuatro datasets de `powerbi/datasets/`. `EXPORT_DEBUG_SNAPSHOT = False` mantiene desactivado el snapshot compacto de debug y `EXPORT_LEGACY_EXPORTS = False` evita exportaciones intermedias antiguas.

Limitaciones: el resultado sigue dependiendo de pocas interacciones personales, calidad/cobertura de tags, sesgo de popularidad de MovieLens, temporalidad imperfecta y calidad del mapeo Trakt-MovieLens. El score es defendible y auditable, pero el gusto humano conserva subjetividad.
